# Aetherion GamaX1 — Full Colab Pipeline v3

This notebook is intentionally **validation-first**.

It does not jump straight into expensive training. The order is:

**environment → repository/package integrity → Drive/data integrity → corpus schema → tokenizer → layers → bulk BPE cache → corpus statistics → capacity planning → real-data smoke training → checkpoint/resume → full training → generation → controlled dense comparison**

Current four sources:

1. `books_cleaned_v1` — prose
2. `Math_Reasoning/train/books` — user/assistant
3. `Conversations-200k_clean` — user/assistant
4. `QnA` — JSONL user/assistant Q&A

The notebook also handles the current repository situation where source files may be flat (`train.py`, `model.py`, ...) while the code itself uses package-relative imports (`from .model import ...`).


**v3 changes (bug fixes only, no new cells/behavior added):**
- Cell 24 (`Q&A` source-format check) now passes as-is -- `bulk_corpus.py` already defines a 4th `Q&A` source with proper `.jsonl` support.
- Cell 50 (`compare_dense`) previously passed `--data_dir`/`--bulk_cache_dir`/`--out_dir`/`--max_steps`/`--no_amp`, none of which exist in `compare_dense.py`'s actual CLI -- it would have failed with "unrecognized arguments" the moment it was uncommented. Fixed to use `compare_dense.py`'s real flags (`--data` pointing at one real book file, `--steps`) instead.
- Not a notebook change, but required before Cell 28 (build/load bulk cache) can run without crashing: `bulk_corpus.py` had a duplicate `_file_stat()` definition whose second copy silently shadowed the first and dropped the `"fingerprint"` key that 3 other places in the file still read -- this crashed with `KeyError: 'fingerprint'` on the very first file encoded, and separately, the encode-checkpoint block referenced `interval_elapsed`/`files_per_sec`/`eta_sec` before they were computed, crashing with `UnboundLocalError` on the first checkpoint of any batch. Both are fixed in the `bulk_corpus.py` delivered alongside this notebook -- copy it into your repo before running Cell 28.

## 0. Configuration


In [ ]:
from pathlib import Path
import os

# ---------- Project ----------
REPO_DIR = Path("/content/gamax1_project")

# Leave empty if the repo already exists in /content or will be uploaded below.
# Put your real GitHub URL here only if you want this notebook to clone it.
REPO_URL = "https://github.com/mrroy-dev/GamaX1_Aetherion.git"

# ---------- Drive ----------
DRIVE_ROOT = Path("/content/drive/MyDrive/Aetherion_GamaX1")
DATA_ROOT = DRIVE_ROOT / "data"

BOOKS_DIR = DATA_ROOT / "books_cleaned_v1"
MATH_DIR = DATA_ROOT / "Math_Reasoning" / "train" / "books"
CONVERSATIONS_DIR = DATA_ROOT / "Conversations-200k_clean"
QNA_DIR = DATA_ROOT / "QnA"

CACHE_DIR = DRIVE_ROOT / "cache" / "gamax1_bulk"
RUN_ROOT = DRIVE_ROOT / "runs" / "gamax1"

# ---------- Corpus / tokenizer ----------
BPE_VOCAB_SIZE = 16_000
BPE_SAMPLE_CHARS = 3_000_000
REBUILD_BULK_CACHE = False

# ---------- Real-data smoke test ----------
SMOKE_STEPS = 5
SMOKE_D_MODEL = 64
SMOKE_HEADS = 4
SMOKE_LAYERS = 1
SMOKE_FEATURES = 128
SMOKE_BLOCK = 64
SMOKE_BATCH = 2

# ---------- Full training ----------
D_MODEL = 256
N_HEADS = 4
N_LAYERS = 4
N_FEATURES = 1024
BLOCK_SIZE = 256
BATCH_SIZE = 8
LR = 3e-4
WEIGHT_DECAY = 0.01
DROPOUT = 0.1
MAX_STEPS = 2_000
WARMUP_STEPS = 200
EVAL_INTERVAL = 200
EVAL_BATCHES = 10
CHECKPOINT_INTERVAL = 200
VALIDATION_SPLIT = "tail"
VALIDATION_SPLIT_SEED = 42
DEVICE = "cuda"
NO_AMP = False
AUTO_SIZE_MODEL = False

# Safety: full training is opt-in inside the notebook.
RUN_FULL_TRAINING = False

print("Configuration loaded.")


Configuration loaded.


## 1. Mount Drive


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Verify the four real data sources


In [ ]:
from collections import Counter

sources = {
    "books_cleaned_v1": BOOKS_DIR,
    "Math_Reasoning_train": MATH_DIR,
    "Conversations-200k_clean": CONVERSATIONS_DIR,
    "Q&A": QNA_DIR,
}

total_files = 0
for name, root in sources.items():
    print(f"\n### {name}")
    print("path   :", root)
    print("exists :", root.exists())
    if not root.exists():
        continue
    files = [p for p in root.rglob("*") if p.is_file()]
    counts = Counter(p.suffix.lower() for p in files)
    print("files  :", len(files))
    print("formats:", dict(counts))
    print("sample :", [p.name for p in files[:5]])
    total_files += len(files)

print("\nTOTAL FILES:", total_files)

assert BOOKS_DIR.exists(), BOOKS_DIR
assert MATH_DIR.exists(), MATH_DIR
assert CONVERSATIONS_DIR.exists(), CONVERSATIONS_DIR
assert QNA_DIR.exists(), QNA_DIR



### books_cleaned_v1
path   : /content/drive/MyDrive/Aetherion_GamaX1/data/books_cleaned_v1
exists : True
files  : 4593
formats: {'.txt': 4593}
sample : ['4587.txt', '4588.txt', '4589.txt', '459.txt', '4590.txt']

### Math_Reasoning_train
path   : /content/drive/MyDrive/Aetherion_GamaX1/data/Math_Reasoning/train/books
exists : True
files  : 0
formats: {}
sample : []

### Conversations-200k_clean
path   : /content/drive/MyDrive/Aetherion_GamaX1/data/Conversations-200k_clean
exists : True
files  : 416
formats: {'.txt': 416}
sample : ['ultrachat_00000.txt', 'ultrachat_00001.txt', 'ultrachat_00002.txt', 'ultrachat_00003.txt', 'ultrachat_00004.txt']

### Q&A
path   : /content/drive/MyDrive/Aetherion_GamaX1/data/QnA
exists : True
files  : 3
formats: {'.jsonl': 3}
sample : ['science_qa_cleaned.jsonl', 'indian_world_geography_english_qa_cleaned.jsonl', 'indian_polity_qa_cleaned.jsonl']

TOTAL FILES: 5012


## 3. Check data quality before tokenization


## 4. Find or obtain the repository


In [ ]:
import subprocess, shutil
from pathlib import Path

# If /content/gamax1_project already exists, use it.
# Otherwise clone if REPO_URL is set.
# Otherwise upload a ZIP containing the project.

if not REPO_DIR.exists():
    if REPO_URL.strip():
        print("Cloning repository...")
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        from google.colab import files
        print("Repository not found.")
        print("Upload the GamaX1 project ZIP now.")
        uploaded = files.upload()
        zip_files = [Path("/content") / name for name in uploaded if name.lower().endswith(".zip")]
        if not zip_files:
            raise RuntimeError("No .zip project file was uploaded.")
        import zipfile
        with zipfile.ZipFile(zip_files[0]) as z:
            z.extractall("/content")

        # Try common extracted names.
        candidates = [
            Path("/content/gamax1_project"),
            Path("/content/GamaX1_Aetherion"),
            Path("/content/gamax1"),
        ]
        found = next((p for p in candidates if p.exists()), None)
        if found is None:
            py_hits = list(Path("/content").glob("**/train.py"))
            if not py_hits:
                raise RuntimeError("Could not locate train.py after ZIP extraction.")
            found = py_hits[0].parent
        REPO_DIR = found

print("REPO_DIR:", REPO_DIR.resolve())
print("Repository contents:")
for p in sorted(REPO_DIR.iterdir()):
    print(" ", p.name, "/" if p.is_dir() else "")


Cloning repository...
REPO_DIR: /content/gamax1_project
Repository contents:
  .git /
  .gitignore 
  BUGFIXES.md 
  CHANGES_v2.md 
  GamaX1_Aetherion_COMPLETE_Master_Colab.ipynb 
  GamaX1_Aetherion_Final_Colab_v7.ipynb 
  GamaX1_Aetherion_Full_Colab_v3.ipynb 
  README.md 
  compare_dense.py 
  gamax1 /
  prepare_large_corpus.py 
  requirements.txt 
  setup.py 
  test_runs /


## 5. Normalize flat repo into an importable `gamax1` package


In [ ]:
import sys, shutil
from pathlib import Path

# The current source uses relative imports such as `from .model import ...`.
# Therefore `python train.py` / `import train` is not the correct execution mode.
# If the repository is already packaged, keep it. If it is flat, create a
# package mirror inside the Colab runtime.

PACKAGE_DIR = REPO_DIR / "gamax1"

required = [
    "tokenizer.py", "layers.py", "model.py", "train.py",
    "bulk_corpus.py", "instruction_data.py", "generate.py", "compare_dense.py"
]

if PACKAGE_DIR.exists() and (PACKAGE_DIR / "__init__.py").exists():
    print("Existing gamax1 package detected.")
else:
    PACKAGE_DIR.mkdir(exist_ok=True)
    for filename in required:
        src = REPO_DIR / filename
        if not src.exists():
            raise FileNotFoundError(f"Missing required source: {src}")
        shutil.copy2(src, PACKAGE_DIR / filename)
    (PACKAGE_DIR / "__init__.py").write_text(
        '"""Aetherion GamaX1 runtime package."""\n',
        encoding="utf-8",
    )
    print("Flat repository normalized into:", PACKAGE_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Package files:")
for p in sorted(PACKAGE_DIR.iterdir()):
    print(" ", p.name)


## 6. Check Python / GPU without modifying Colab's PyTorch


In [ ]:
import sys, torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM: {props.total_memory / 1024**3:.2f} GB")
else:
    print("WARNING: CUDA is unavailable; training will be very slow on CPU.")

# Do not `pip install -U torch` in this notebook.
# Colab's CUDA-compatible PyTorch build should be left intact.


## 7. Import the packaged code


In [ ]:
import gamax1
from gamax1 import tokenizer, layers, model, train, bulk_corpus, instruction_data

print("gamax1 package import: PASS")
print("tokenizer:", tokenizer.__file__)
print("train    :", train.__file__)
print("bulk     :", bulk_corpus.__file__)


## 8. API/integrity checks


In [ ]:
import inspect
from gamax1.tokenizer import BPETokenizer, CharTokenizer
from gamax1.layers import (
    build_hex_neighbor_table,
    DynamicSparsityController,
    ProbationaryMemoryTracker,
    SparseSuperpositionLinear,
)
from gamax1.instruction_data import encode_example

required_symbols = [
    BPETokenizer, CharTokenizer, build_hex_neighbor_table,
    DynamicSparsityController, ProbationaryMemoryTracker,
    SparseSuperpositionLinear, encode_example
]
assert all(x is not None for x in required_symbols)

# Detect the specific historical CharTokenizer contamination bug.
char_source = inspect.getsource(CharTokenizer.encode)
bad_names = ["eos_id", "user_id", "assistant_id", "_merge_trie", "byte_to_id"]
bad_hits = [x for x in bad_names if x in char_source]

if bad_hits:
    print("WARNING: CharTokenizer.encode still contains BPE-only references:", bad_hits)
    print("BPE training can still be tested separately, but the tokenizer file should be fixed before using char mode.")
else:
    print("CharTokenizer implementation looks independent of BPE internals.")

print("Required APIs: PASS")


## 9. BPE tokenizer smoke test


In [ ]:
from gamax1.tokenizer import BPETokenizer

sample = "Aetherion is a small language-model research project. Numbers: 12345."
bpe = BPETokenizer(sample, vocab_size=512)

ids = bpe.encode(sample)
decoded = bpe.decode(ids)

print("vocab size:", bpe.vocab_size)
print("tokens:", len(ids))
print("decoded:", decoded)
assert decoded == sample
print("BPE round-trip: PASS")


## 10. Layers smoke test


In [ ]:
import torch
from gamax1.layers import (
    build_hex_neighbor_table,
    DynamicSparsityController,
    ProbationaryMemoryTracker,
    SparseSuperpositionLinear,
)

torch.manual_seed(42)

n_features = 32
d_model = 32
x = torch.randn(2, 8, d_model)

neighbors = build_hex_neighbor_table(n_features)
assert neighbors.shape == (n_features, 6)
assert neighbors.dtype == torch.long
assert int(neighbors.min()) >= 0 and int(neighbors.max()) < n_features

ctrl = DynamicSparsityController(16, 4, 32, trend_window=10, patience=20)
for i in range(40):
    ctrl.step(1.0 - 0.001 * i)
assert 4 <= ctrl.k <= 32

ptm = ProbationaryMemoryTracker(n_features)
mask = torch.zeros(16, n_features, dtype=torch.bool)
mask[:, :2] = True
for _ in range(4):
    ptm.update(mask)

layer = SparseSuperpositionLinear(d_model, n_features)
y = layer(x, k=12)
y.square().mean().backward()

assert y.shape == x.shape
assert layer.last_active_mask.shape == (16, n_features)

print("neighbor table:", neighbors.shape)
print("controller k:", ctrl.k)
print("active mask:", layer.last_active_mask.shape)
print("LAYERS SMOKE TEST: PASS")


## 11. Inspect source-format assumptions


In [ ]:
from gamax1.bulk_corpus import DEFAULT_SOURCE_DIRS, DEFAULT_SOURCE_FORMATS

print("Default source directories:")
for k, v in DEFAULT_SOURCE_DIRS.items():
    print(f"  {k}: {v}")

print("\nDefault source formats:")
for k, v in DEFAULT_SOURCE_FORMATS.items():
    print(f"  {k}: {v}")

assert set(DEFAULT_SOURCE_FORMATS) >= set(sources)
assert DEFAULT_SOURCE_FORMATS["books_cleaned_v1"] == "prose"
assert DEFAULT_SOURCE_FORMATS["Math_Reasoning_train"] == "user_assistant"
assert DEFAULT_SOURCE_FORMATS["Conversations-200k_clean"] == "user_assistant"
assert DEFAULT_SOURCE_FORMATS["Q&A"] == "user_assistant"

print("\nSource-format mapping: PASS")


## 12. Preview representative files


In [ ]:
from pathlib import Path
import json

# Actual corpus sources + their directories.
sources = {
    "books_cleaned_v1": Path(
        "/content/drive/MyDrive/Aetherion_GamaX1/data/books_cleaned_v1"
    ),
    "Math_Reasoning_train": Path(
        "/content/drive/MyDrive/Aetherion_GamaX1/data/Math_Reasoning/train/books"
    ),
    "Conversations-200k_clean": Path(
        "/content/drive/MyDrive/Aetherion_GamaX1/data/Conversations-200k_clean"
    ),
    "Q&A": Path(
        "/content/drive/MyDrive/Aetherion_GamaX1/data/QnA"
    ),
}


def preview(path, n=700):
    print("\nFILE:", path)

    if path.suffix.lower() == ".jsonl":
        with path.open("r", encoding="utf-8") as f:
            for _ in range(2):
                line = f.readline()
                if not line:
                    break
                print(line[:n])
    else:
        print(
            path.read_text(
                encoding="utf-8",
                errors="replace"
            )[:n]
        )


for name, root in sources.items():
    print(f"\n{'=' * 70}")
    print(f"SOURCE: {name}")
    print(f"ROOT:   {root}")

    if not root.exists():
        print("⚠️ DIRECTORY NOT FOUND")
        continue

    files = [
        p for p in root.rglob("*")
        if p.is_file()
    ]

    print(f"Files found: {len(files)}")

    if files:
        preview(files[0])
    else:
        print("⚠️ No files found")


print("\nRepresentative preview complete.")

## 13. Build/load persistent bulk BPE cache


In [ ]:
from gamax1.bulk_corpus import build_or_load_bulk_tokens

CACHE_DIR.mkdir(parents=True, exist_ok=True)

tok, store, metadata = build_or_load_bulk_tokens(
    str(DATA_ROOT),
    str(CACHE_DIR),
    bpe_vocab_size=BPE_VOCAB_SIZE,
    bpe_sample_chars=BPE_SAMPLE_CHARS,
    tokenizer=None,
    rebuild=REBUILD_BULK_CACHE,
)

print("Bulk cache ready.")
print("Tokenizer vocab:", tok.vocab_size)
print("Token store:", type(store).__name__)
print("Metadata keys:", sorted(metadata.keys()))
print("File count:", metadata.get("file_count"))
print("Token count:", metadata.get("token_count"))


## 14. Validate cache identity and token ranges


In [ ]:
import json
from pathlib import Path

meta_path = CACHE_DIR / "metadata.json"
token_path = CACHE_DIR / "tokens.int32.bin"

assert meta_path.exists(), meta_path
assert token_path.exists(), token_path

cache_meta = json.loads(meta_path.read_text(encoding="utf-8"))

print("cache metadata:")
for k in ["corpus_format_version", "file_count", "token_count", "vocab_size", "bpe_vocab_size"]:
    if k in cache_meta:
        print(f"  {k}: {cache_meta[k]}")

print("token file MB:", token_path.stat().st_size / 1024**2)

# BulkStore exposes an int32 memory map as `.tensor`.
data = store.tensor
print("tensor dtype:", data.dtype)
print("tensor length:", len(data))

assert str(data.dtype).endswith("int32")
assert len(data) > 0
assert int(data.min()) >= 0
assert int(data.max()) < tok.vocab_size

print("CACHE INTEGRITY: PASS")


## 15. Per-source corpus statistics


In [ ]:
# The metadata contains the source/file index used by the incremental cache.
# Print whatever source-level information this version exposes.

for key in ["source_stats", "sources", "source_token_counts", "file_count", "token_count"]:
    if key in metadata:
        print(f"\n{key}:")
        print(metadata[key])

print("\nIf source-level counts are available above, use them to check that")
print("books, math, conversations, and Q&A all contributed before training.")


## 16. Special-token and instruction-format validation


In [ ]:
specials = ["eos_id", "user_id", "assistant_id", "pad_id", "think_id", "think_end_id"]
for name in specials:
    print(name, getattr(tok, name, None))

assert tok.eos_id is not None
assert tok.user_id is not None
assert tok.assistant_id is not None

sample = "<|user|>What is Aetherion?<|assistant|>Aetherion is a research project.<|eos|>"
ids = tok.encode(sample)
decoded = tok.decode(ids)
print("\nRound-trip:")
print(decoded)

ids2, mask = encode_example(
    tok,
    "What is Aetherion?",
    "Aetherion is a research project.",
    max_len=128,
)

assert ids2[0] == tok.user_id
assert tok.assistant_id in ids2
assert ids2[-1] == tok.eos_id
assert len(ids2) == len(mask)

print("\nInstruction IDs:", len(ids2))
print("Assistant position:", ids2.index(tok.assistant_id))
print("EOS position:", len(ids2)-1)
print("INSTRUCTION FORMAT: PASS")


## 17. Capacity planning BEFORE training


In [ ]:
from gamax1.train import _parameter_count_for_config, tokens_per_parameter, auto_size_model

token_count = int(len(store.tensor))

param_count = _parameter_count_for_config(
    tok.vocab_size,
    BLOCK_SIZE,
    D_MODEL,
    N_HEADS,
    N_LAYERS,
    N_FEATURES,
)

ratio = tokens_per_parameter(token_count, param_count)

print(f"Corpus tokens     : {token_count:,}")
print(f"Vocab             : {tok.vocab_size:,}")
print(f"Model parameters  : {param_count:,}")
print(f"Tokens/parameter  : {ratio:.2f}")

if ratio < 10:
    print("WARNING: corpus/model ratio is below the training heuristic floor.")
elif ratio < 40:
    print("NOTICE: corpus is above the minimum heuristic but below the 40 tok/param target.")
else:
    print("Capacity check: comfortably above the configured 40 tok/param target.")

if AUTO_SIZE_MODEL:
    plan = auto_size_model(
        token_count,
        tok.vocab_size,
        BLOCK_SIZE,
        N_HEADS,
        target_tokens_per_param=40.0,
        min_tokens_per_param=10.0,
    )
    print("\nAUTO-SIZE PLAN:")
    print(plan)


## 18. Real-data smoke training — mandatory gate


In [ ]:
import subprocess, sys
from pathlib import Path

SMOKE_OUT = RUN_ROOT / "smoke_real_data"
SMOKE_OUT.mkdir(parents=True, exist_ok=True)

smoke_cmd = [
    sys.executable, "-m", "gamax1.train",
    "--data_dir", str(DATA_ROOT),
    "--bulk_cache_dir", str(CACHE_DIR),
    "--out_dir", str(SMOKE_OUT),
    "--d_model", str(SMOKE_D_MODEL),
    "--n_heads", str(SMOKE_HEADS),
    "--n_layers", str(SMOKE_LAYERS),
    "--n_features", str(SMOKE_FEATURES),
    "--block_size", str(SMOKE_BLOCK),
    "--batch_size", str(SMOKE_BATCH),
    "--lr", "3e-4",
    "--max_steps", str(SMOKE_STEPS),
    "--warmup_steps", "1",
    "--eval_interval", str(SMOKE_STEPS),
    "--eval_batches", "2",
    "--checkpoint_interval", str(SMOKE_STEPS),
    "--validation_split", VALIDATION_SPLIT,
    "--validation_split_seed", str(VALIDATION_SPLIT_SEED),
    "--tokenizer", "bpe",
    "--bpe_vocab_size", str(BPE_VOCAB_SIZE),
    "--bpe_sample_chars", str(BPE_SAMPLE_CHARS),
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
]
if NO_AMP:
    smoke_cmd.append("--no_amp")

print(" ".join(smoke_cmd))
subprocess.run(smoke_cmd, cwd=str(REPO_DIR), check=True)

assert (SMOKE_OUT / "gamax1.pt").exists()
assert (SMOKE_OUT / "gamax1_latest.pt").exists()

print("REAL-DATA SMOKE TRAINING: PASS")


## 19. Verify smoke checkpoint can resume


In [ ]:
latest = SMOKE_OUT / "gamax1_latest.pt"
resume_out = RUN_ROOT / "resume_test"
resume_out.mkdir(parents=True, exist_ok=True)

resume_cmd = [
    sys.executable, "-m", "gamax1.train",
    "--data_dir", str(DATA_ROOT),
    "--bulk_cache_dir", str(CACHE_DIR),
    "--out_dir", str(resume_out),
    "--resume_from", str(latest),
    "--max_steps", str(SMOKE_STEPS + 2),
    "--eval_interval", "2",
    "--eval_batches", "1",
    "--checkpoint_interval", "2",
    "--tokenizer", "bpe",
    "--bpe_vocab_size", str(BPE_VOCAB_SIZE),
    "--bpe_sample_chars", str(BPE_SAMPLE_CHARS),
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
]
if NO_AMP:
    resume_cmd.append("--no_amp")

print(" ".join(resume_cmd))
subprocess.run(resume_cmd, cwd=str(REPO_DIR), check=True)

print("RESUME TEST: PASS")


## 20. Full training — opt-in


In [ ]:
# ===== FULL TRAINING (real Aetherion config: 768 / 12 layers, ~97.7M params) =====
import os, subprocess, sys

RUN_FULL_TRAINING = True   # is cell ke liye override

if not RUN_FULL_TRAINING:
    print("RUN_FULL_TRAINING=False — full training skipped.")
else:
    F_D_MODEL       = 768
    F_N_HEADS       = 12
    F_N_LAYERS      = 12
    F_N_FEATURES    = 3072
    F_BLOCK_SIZE    = 512
    F_BATCH_SIZE    = 16      # T4 par OOM aaye to 8 kar do
    F_MAX_STEPS     = 100_000
    F_WARMUP        = 500
    F_EVAL_INTERVAL = 200
    F_CKPT_INTERVAL = 500

    FULL_OUT = RUN_ROOT / "full"
    FULL_OUT.mkdir(parents=True, exist_ok=True)
    EXP_DIR = RUN_ROOT / "experiments"
    EXP_DIR.mkdir(parents=True, exist_ok=True)

    cmd = [
        sys.executable, "-u", "-m", "gamax1.train",     # -u = unbuffered
        "--data_dir", str(DATA_ROOT),
        "--bulk_cache_dir", str(CACHE_DIR),
        "--out_dir", str(FULL_OUT),
        "--d_model", str(F_D_MODEL),
        "--n_heads", str(F_N_HEADS),
        "--n_layers", str(F_N_LAYERS),
        "--n_features", str(F_N_FEATURES),
        "--block_size", str(F_BLOCK_SIZE),
        "--batch_size", str(F_BATCH_SIZE),
        "--lr", str(LR),
        "--weight_decay", str(WEIGHT_DECAY),
        "--dropout", str(DROPOUT),
        "--max_steps", str(F_MAX_STEPS),
        "--warmup_steps", str(F_WARMUP),
        "--eval_interval", str(F_EVAL_INTERVAL),
        "--eval_batches", str(EVAL_BATCHES),
        "--checkpoint_interval", str(F_CKPT_INTERVAL),
        "--validation_split", VALIDATION_SPLIT,
        "--validation_split_seed", str(VALIDATION_SPLIT_SEED),
        "--tokenizer", "bpe",
        "--bpe_vocab_size", str(BPE_VOCAB_SIZE),
        "--bpe_sample_chars", str(BPE_SAMPLE_CHARS),
        "--experiment_dir", str(EXP_DIR),
        "--run_name", "full_v7_master",
        "--device", DEVICE if DEVICE == "cuda" and torch.cuda.is_available() else "cpu",
    ]

    if NO_AMP:
        cmd.append("--no_amp")

    print(" ".join(cmd))
    print("\n--- training output (live) ---\n", flush=True)

    env = dict(os.environ, PYTHONUNBUFFERED="1")
    proc = subprocess.Popen(
        cmd, cwd=str(REPO_DIR), env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    try:
        for line in proc.stdout:
            print(line, end="", flush=True)
        proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("\nTraining interrupted by user.")
        raise

    if proc.returncode != 0:
        raise RuntimeError(f"gamax1.train exited with code {proc.returncode}")

## 21. Resume full training


In [ ]:
FULL_OUT = RUN_ROOT / "full"
latest_full = FULL_OUT / "gamax1_latest.pt"

if not latest_full.exists():
    print("No full-run checkpoint exists yet.")
    print("Run the full-training cell first.")
else:
    cmd = [
        sys.executable, "-m", "gamax1.train",
        "--data_dir", str(DATA_ROOT),
        "--bulk_cache_dir", str(CACHE_DIR),
        "--out_dir", str(FULL_OUT),
        "--resume_from", str(latest_full),
        "--max_steps", str(MAX_STEPS),
        "--eval_interval", str(EVAL_INTERVAL),
        "--eval_batches", str(EVAL_BATCHES),
        "--checkpoint_interval", str(CHECKPOINT_INTERVAL),
        "--tokenizer", "bpe",
        "--bpe_vocab_size", str(BPE_VOCAB_SIZE),
        "--bpe_sample_chars", str(BPE_SAMPLE_CHARS),
        "--device", DEVICE if DEVICE == "cuda" and torch.cuda.is_available() else "cpu",
    ]
    if NO_AMP:
        cmd.append("--no_amp")
    subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


In [ ]:
import sys, subprocess, os, torch

FULL_OUT = RUN_ROOT / "full"
latest_full = FULL_OUT / "gamax1_latest.pt"

cmd = [
    sys.executable, "-u", "-m", "gamax1.train",
    "--data_dir", str(DATA_ROOT),
    "--bulk_cache_dir", str(CACHE_DIR),
    "--rebuild_bulk_cache",
    "--out_dir", str(FULL_OUT),
    "--resume_from", str(latest_full),
    "--d_model", "768", "--n_heads", "12", "--n_layers", "12", "--n_features", "3072",
    "--block_size", "512",
    "--batch_size", "16",            # zaroori: default 32 se OOM ho sakta hai
    "--lr", "0.0003", "--weight_decay", "0.01", "--dropout", "0.1",
    "--max_steps", "100000",
    "--warmup_steps", "500",
    "--eval_interval", "200", "--eval_batches", "10",
    "--checkpoint_interval", "500",
    "--validation_split", "tail", "--validation_split_seed", "42",
    "--tokenizer", "bpe", "--bpe_vocab_size", "16000", "--bpe_sample_chars", "3000000",
    "--experiment_dir", str(RUN_ROOT / "experiments"),
    "--run_name", "full_v7_master",
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
]
print(" ".join(cmd))
print("\n--- training output (live) ---\n", flush=True)

env = dict(os.environ, PYTHONUNBUFFERED="1")
proc = subprocess.Popen(cmd, cwd=str(REPO_DIR), env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"exit code {proc.returncode}")

In [ ]:
# ============================================================
# GAMAX1 / AETHERION — FINAL SAFE RESUME
# 30,000 -> 100,000 STEPS
# EXISTING VERIFIED CACHE — NO REBUILD
# ============================================================

from pathlib import Path
import subprocess
import sys
import os

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

REPO_DIR = Path("/content/gamax1_project")

DATA_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/data"
)

CACHE_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/gamax1_bulk"
)

OUT_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full"
)

CHECKPOINT = OUT_DIR / "gamax1_latest.pt"

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/experiments"
)

# ------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------

MAX_STEPS = 100_000

# ------------------------------------------------------------
# SAFETY CHECKS
# ------------------------------------------------------------

print("=" * 90)
print("GAMAX1 / AETHERION — FINAL SAFE RESUME")
print("=" * 90)

print(f"Repository : {REPO_DIR}")
print(f"Data       : {DATA_DIR}")
print(f"Cache      : {CACHE_DIR}")
print(f"Output     : {OUT_DIR}")
print(f"Checkpoint : {CHECKPOINT}")
print()

# Repo
if not REPO_DIR.exists():
    raise FileNotFoundError(
        f"Repository not found: {REPO_DIR}"
    )

# Cache
TOKENS = CACHE_DIR / "tokens.int32.bin"
TOKENIZER = CACHE_DIR / "tokenizer.json"
FILE_INDEX = CACHE_DIR / "file_index.json"
METADATA = CACHE_DIR / "metadata.json"

for p in [TOKENS, TOKENIZER, FILE_INDEX, METADATA]:
    if not p.exists():
        raise FileNotFoundError(
            f"Required cache file missing:\n{p}"
        )

# Checkpoint
if not CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Checkpoint not found:\n{CHECKPOINT}"
    )

# ------------------------------------------------------------
# VERIFY TOKEN COUNT WITHOUT REBUILDING
# ------------------------------------------------------------

token_count = TOKENS.stat().st_size // 4

print("[CACHE VERIFICATION]")
print(f"tokens.int32.bin : {TOKENS.stat().st_size / (1024**3):.2f} GB")
print(f"Token count      : {token_count:,}")

EXPECTED_TOKENS = 1_170_998_016

if token_count != EXPECTED_TOKENS:
    raise RuntimeError(
        f"SAFETY STOP:\n"
        f"Expected {EXPECTED_TOKENS:,} tokens\n"
        f"Found    {token_count:,} tokens"
    )

print("✓ Verified 1.17B-token cache")

# ------------------------------------------------------------
# CHECKPOINT STEP
# ------------------------------------------------------------

import torch

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

current_step = int(checkpoint.get("step", 0))

print()
print("[CHECKPOINT]")
print(f"Current step : {current_step:,}")
print(f"Target step  : {MAX_STEPS:,}")
print(f"Remaining    : {MAX_STEPS - current_step:,}")

if current_step >= MAX_STEPS:
    print()
    print("✓ Training is already at/above target step.")
    raise SystemExit(0)

# ------------------------------------------------------------
# CUDA
# ------------------------------------------------------------

print()
print("[DEVICE]")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Training was NOT started."
    )

print("CUDA available : True")
print("GPU            :", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# FINAL COMMAND
# IMPORTANT:
# NO --rebuild_bulk_cache
# ------------------------------------------------------------

cmd = [
    sys.executable,
    "-u",
    "-m",
    "gamax1.train",

    "--data_dir",
    str(DATA_DIR),

    "--bulk_cache_dir",
    str(CACHE_DIR),

    "--out_dir",
    str(OUT_DIR),

    "--resume_from",
    str(CHECKPOINT),

    "--d_model",
    "768",

    "--n_heads",
    "12",

    "--n_layers",
    "12",

    "--n_features",
    "3072",

    "--block_size",
    "512",

    "--batch_size",
    "16",

    "--lr",
    "0.0003",

    "--weight_decay",
    "0.01",

    "--dropout",
    "0.1",

    "--max_steps",
    str(MAX_STEPS),

    "--warmup_steps",
    "500",

    "--eval_interval",
    "200",

    "--eval_batches",
    "10",

    "--checkpoint_interval",
    "500",

    "--validation_split",
    "tail",

    "--validation_split_seed",
    "42",

    "--tokenizer",
    "bpe",

    "--bpe_vocab_size",
    "16000",

    "--bpe_sample_chars",
    "3000000",

    "--experiment_dir",
    str(EXPERIMENT_DIR),

    "--run_name",
    "full_v7_master",

    "--device",
    "cuda",
]

# ------------------------------------------------------------
# DISPLAY COMMAND
# ------------------------------------------------------------

print()
print("=" * 90)
print("🚀 STARTING GAMAX1 RESUME")
print("=" * 90)

print(f"Resume : {current_step:,}")
print(f"Target : {MAX_STEPS:,}")
print(f"Remain : {MAX_STEPS - current_step:,}")
print("Cache  : VERIFIED 1.17B TOKENS")
print("Rebuild: DISABLED")
print("Device :", "cuda")
print("=" * 90)

print()
print("COMMAND:")
print(" ".join(cmd))
print()

# ------------------------------------------------------------
# HARD SAFETY ASSERTION
# ------------------------------------------------------------

if "--rebuild_bulk_cache" in cmd:
    raise RuntimeError(
        "SAFETY STOP: --rebuild_bulk_cache detected!"
    )

print("✓ --rebuild_bulk_cache is NOT present")
print("✓ Existing bulk cache will be reused")
print("✓ 34,436 files will NOT be re-tokenized")
print()
print("=" * 90)
print("TRAINING STARTING...")
print("=" * 90)
print()

# ------------------------------------------------------------
# RUN
# ------------------------------------------------------------

proc = subprocess.run(
    cmd,
    cwd=str(REPO_DIR),
    env=os.environ.copy(),
)

if proc.returncode != 0:
    raise RuntimeError(
        f"\n❌ gamax1.train exited with code {proc.returncode}"
    )

print()
print("=" * 90)
print("🎉 GAMAX1 TRAINING FINISHED")
print("=" * 90)

In [ ]:
# ============================================================
# GAMAX1 — TOKENIZER COMPATIBILITY DIAGNOSTIC
# READ ONLY — NO REBUILD / NO DELETE
# ============================================================

import json
import hashlib
from pathlib import Path
import torch

CACHE_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/bulk_bpe"
)

CHECKPOINT = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1_latest.pt"
)

TOKENIZER_JSON = CACHE_DIR / "tokenizer.json"
TOKENS_BIN = CACHE_DIR / "tokens.int32.bin"

print("=" * 88)
print("GAMAX1 TOKENIZER COMPATIBILITY CHECK")
print("=" * 88)

print(f"Cache      : {CACHE_DIR}")
print(f"Checkpoint : {CHECKPOINT}")
print()

# ------------------------------------------------------------
# FILE CHECK
# ------------------------------------------------------------

for p in [TOKENIZER_JSON, TOKENS_BIN, CHECKPOINT]:
    print(f"{p.name:20} : {'✓' if p.exists() else '❌'}")

if not TOKENIZER_JSON.exists():
    raise FileNotFoundError(TOKENIZER_JSON)

if not CHECKPOINT.exists():
    raise FileNotFoundError(CHECKPOINT)

print()

# ------------------------------------------------------------
# CACHE TOKENIZER
# ------------------------------------------------------------

with open(TOKENIZER_JSON, "r", encoding="utf-8") as f:
    cache_tok = json.load(f)

print("[CACHE TOKENIZER]")

cache_model = cache_tok.get("model", {})
cache_vocab = cache_model.get("vocab", {})
cache_merges = cache_model.get("merges", [])

print(f"Vocab size : {len(cache_vocab):,}")
print(f"Merge count: {len(cache_merges):,}")

cache_bytes = TOKENIZER_JSON.read_bytes()
cache_hash = hashlib.sha256(cache_bytes).hexdigest()

print(f"SHA256     : {cache_hash}")
print()

# ------------------------------------------------------------
# CHECKPOINT
# ------------------------------------------------------------

print("[CHECKPOINT TOKENIZER]")

ckpt = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

state = ckpt.get("tokenizer_state")

if state is None:
    print("❌ checkpoint has no tokenizer_state")
else:
    print("✓ tokenizer_state found")

    ckpt_vocab = state.get("vocab")
    ckpt_merges = state.get("merges")

    if ckpt_vocab is not None:
        print(f"Vocab size : {len(ckpt_vocab):,}")
    else:
        print("Vocab size : unavailable")

    if ckpt_merges is not None:
        print(f"Merge count: {len(ckpt_merges):,}")
    else:
        print("Merge count: unavailable")

print()

# ------------------------------------------------------------
# MODEL EMBEDDING SIZE
# ------------------------------------------------------------

print("[MODEL CHECK]")

state_dict = ckpt.get("model", ckpt)

embedding_keys = [
    k for k in state_dict.keys()
    if "token" in k.lower() and "embed" in k.lower()
]

for k in embedding_keys[:10]:
    value = state_dict[k]
    if hasattr(value, "shape"):
        print(f"{k}: {tuple(value.shape)}")

print()

# ------------------------------------------------------------
# EXACT COMPARISON
# ------------------------------------------------------------

print("=" * 88)
print("RESULT")
print("=" * 88)

if state is None:
    print("⚠️ Cannot perform exact tokenizer comparison.")
    print("Checkpoint tokenizer_state is missing.")

else:
    ckpt_vocab = state.get("vocab")
    ckpt_merges = state.get("merges")

    vocab_same = (
        ckpt_vocab is not None
        and ckpt_vocab == cache_vocab
    )

    merges_same = (
        ckpt_merges is not None
        and ckpt_merges == cache_merges
    )

    print(f"Vocabulary identical : {vocab_same}")
    print(f"Merges identical     : {merges_same}")

    if vocab_same and merges_same:
        print()
        print("✅ TOKENIZER IS IDENTICAL")
        print("The cache tokenizer matches the checkpoint tokenizer.")

    else:
        print()
        print("❌ TOKENIZER MISMATCH")
        print()
        print("DO NOT resume training with this tokens.int32.bin.")
        print("DO NOT delete the existing cache.")
        print("DO NOT force the training to continue.")
        print()
        print("The next step is to locate the cache that was created")
        print("with the 30,000-step checkpoint tokenizer.")

In [ ]:
from pathlib import Path
import json

CACHE_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/gamax1_bulk"
)

print("=" * 80)
print("GAMAX1 CACHE INSPECTION — READ ONLY")
print("=" * 80)

print(f"Cache: {CACHE_DIR}")
print()

for name in [
    "tokens.int32.bin",
    "tokenizer.json",
    "file_index.json",
    "encode_progress.json",
]:
    p = CACHE_DIR / name
    print(f"{name:25} : {'✓ EXISTS' if p.exists() else '— missing'}")

print()

# Inspect metadata/index only — no rebuild
for name in [
    "file_index.json",
    "encode_progress.json",
]:
    p = CACHE_DIR / name

    if not p.exists():
        continue

    print("-" * 80)
    print(name)

    try:
        with open(p, "r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, dict):
            print("keys:", list(obj.keys())[:30])

            if name == "file_index.json":
                print("entries:", len(obj))

            elif name == "encode_progress.json":
                print("content:", obj)

    except Exception as e:
        print("Could not read:", e)

print()
print("=" * 80)
print("NO FILES WERE MODIFIED")
print("=" * 80)

## 22. Generation test


In [ ]:
import subprocess, sys, torch

CKPT = RUN_ROOT / "full" / "gamax1.pt"

if not CKPT.exists():
    print("Full checkpoint not found:", CKPT)
else:
    cmd = [
        sys.executable, "-u", "-m", "gamax1.generate",
        "--ckpt", str(CKPT),
        "--prompt", "The United States Declaration of Independence ",
        "--repetition_penalty","0.3",
        "--max_new_tokens", "80",
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
    ]
    r = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    print(r.stdout)
    if r.stderr:
        print("--- stderr ---")
        print(r.stderr)
    print("exit code:", r.returncode)

## 23. Chat generation test


In [ ]:
import subprocess, sys, torch

CKPT = RUN_ROOT / "full" / "gamax1.pt"

if CKPT.exists():
    questions = "What is the capital of India?\nWho wrote Hamlet?\n"
    cmd = [
        sys.executable, "-u", "-m", "gamax1.generate",
        "--ckpt", str(CKPT),
        "--chat",
        "--max_new_tokens", "80",
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
    ]
    try:
        r = subprocess.run(cmd, cwd=str(REPO_DIR), input=questions,
                           capture_output=True, text=True, timeout=300)
        print(r.stdout)
        if r.stderr:
            print("--- stderr ---")
            print(r.stderr[-3000:])
        print("exit code:", r.returncode)
    except subprocess.TimeoutExpired as e:
        print("Timeout: chat mode shayad input ka wait kar raha hai.")
        print((e.stdout or "")[-2000:] if isinstance(e.stdout, str) else "")
else:
    print("Skip: no full checkpoint yet.")

In [ ]:
import subprocess, sys, torch

CKPT = RUN_ROOT / "full" / "gamax1.pt"
dev = "cuda" if torch.cuda.is_available() else "cpu"

def gen(prompt, chat=False, temp=0.8, top_k=40, rep=1.2, n=80):
    cmd = [sys.executable, "-u", "-m", "gamax1.generate",
           "--ckpt", str(CKPT), "--prompt", prompt,
           "--max_new_tokens", str(n),
           "--temperature", str(temp), "--top_k", str(top_k),
           "--repetition_penalty", str(rep), "--device", dev]
    if chat:
        cmd += ["--chat", "--stop_at_eos"]
    r = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    print(r.stdout)
    if r.returncode != 0:
        print("--- stderr ---\n", r.stderr[-2000:])
    print("=" * 60)

# 1) Chat mode, ab sawaal sahi jayega
gen("What is the capital of India?", chat=True)
gen("Who wrote Hamlet?", chat=True)

# 2) Base mode, sampling ke saath: kya repetition loop hat gaya?
gen("Once upon a time")
gen("The capital of India is")

## 24. Controlled sparse-vs-dense comparison


In [ ]:
COMPARE_OUT = RUN_ROOT / "compare_dense"
COMPARE_OUT.mkdir(parents=True, exist_ok=True)  # currently unused: compare_dense.py only prints, saves nothing

# compare_dense.py is a standalone, single-flat-file mechanism test (sparse vs
# dense FFN) -- it does NOT use the bulk_corpus.py multi-source/.jsonl pipeline
# the rest of this notebook uses. Its real CLI only supports:
#   --data --tokenizer --bpe_vocab_size --steps --d_model --n_heads --n_layers
#   --n_features --block_size --batch_size --lr --dropout --eval_batches --device
# (no --data_dir / --bulk_cache_dir / --out_dir / --max_steps / --no_amp).
# It wants ONE plain-text file, so we point it at a real book file instead of
# the multi-source corpus.
compare_sample_files = [p for p in BOOKS_DIR.rglob("*.txt") if p.is_file()]
if not compare_sample_files:
    raise FileNotFoundError(
        f"No .txt files found under {BOOKS_DIR} for compare_dense's --data. "
        "This experiment needs one representative plain-text file."
    )
compare_data_file = compare_sample_files[0]

cmd = [
    sys.executable, "-m", "gamax1.compare_dense",
    "--data", str(compare_data_file),
    "--d_model", str(D_MODEL),
    "--n_heads", str(N_HEADS),
    "--n_layers", str(N_LAYERS),
    "--n_features", str(N_FEATURES),
    "--block_size", str(BLOCK_SIZE),
    "--batch_size", str(BATCH_SIZE),
    "--steps", "200",
    "--eval_batches", str(EVAL_BATCHES),
    "--tokenizer", "bpe",
    "--bpe_vocab_size", str(BPE_VOCAB_SIZE),
    "--device", DEVICE if DEVICE == "cuda" and torch.cuda.is_available() else "cpu",
]

print("This is an optional experiment, not the first training gate.")
print("Comparison data file:", compare_data_file)
print(" ".join(cmd))
# Uncomment to execute:
# subprocess.run(cmd, cwd=str(REPO_DIR), check=True)


## 25. Final artifacts and environment report


## Decision gates

Before serious training, all of these should be green:

- [:: ] All four Drive sources exist.
- [ ] File formats match expectations, including **Q&A `.jsonl`**.
- [ ] No unexpected unreadable corpus files.
- [ ] Repository/package imports succeed.
- [ ] BPE round-trip succeeds.
- [ ] Layers smoke test succeeds.
- [ ] Bulk cache has a valid identity and token range.
- [ ] All four source categories contribute data.
- [ ] Tokens/parameter capacity is understood.
- [ ] Real-data smoke training succeeds.
- [ ] Checkpoint resume succeeds.
- [ ] Only then start the expensive full run.

**Important:** generation quality from a 5-step smoke test is not a quality benchmark. It only proves that the end-to-end technical path works.


**TEST GEA**

In [ ]:
import subprocess, sys, torch

# Specific step checkpoint use karo — safe, fully-written file
# Apna nearest available step number check karke CKPT_STEP badal lena
CKPT_STEP = 44000
CKPT = RUN_ROOT / "full" / f"gamax1_step_{CKPT_STEP}.pt"

if not CKPT.exists():
    print(f"⚠️ {CKPT} nahi mila — available checkpoints dekh lo:")
    print(sorted((RUN_ROOT / "full").glob("gamax1_step_*.pt")))
else:
    dev = "cuda" if torch.cuda.is_available() else "cpu"

    def gen(prompt, chat=False, temp=0.8, top_k=40, rep=1.2, n=80):
        cmd = [sys.executable, "-u", "-m", "gamax1.generate",
               "--ckpt", str(CKPT), "--prompt", prompt,
               "--max_new_tokens", str(n),
               "--temperature", str(temp), "--top_k", str(top_k),
               "--repetition_penalty", str(rep), "--device", dev]
        if chat:
            cmd += ["--chat", "--stop_at_eos"]
        r = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
        print(f"\n>>> PROMPT: {prompt!r} (chat={chat})")
        print(r.stdout)
        if r.returncode != 0:
            print("--- stderr ---\n", r.stderr[-2000:])
        print("=" * 60)

    # 1) Facts — pehle (30k pe) galat aaye the, dekhte hain 44k pe kuch sudhar hai kya
    gen("What is the capital of India is", temp=0.0, rep=1.0, chat=False)  # greedy, deterministic
    gen("What is the capital of India?", chat=True)
    gen("Who wrote Hamlet?", chat=True)

    # 2) Arithmetic — digit-tokenizer bug abhi bhi hai, par dekhte hain consistency
    gen("2 + 2 =", temp=0.0, rep=1.0, chat=False)

    # 3) Base-mode fluency/repetition sanity check
    gen("Once upon a time")
    gen("The capital of India is")

In [ ]:
import subprocess, sys, torch

CKPT_STEP = 44000   # apna current latest completed step yahan daalo
CKPT = RUN_ROOT / "full" / f"gamax1_step_{CKPT_STEP}.pt"
if not CKPT.exists():
    CKPT = RUN_ROOT / "full" / "gamax1_latest.pt"
    print(f"⚠️ step-specific checkpoint nahi mila, latest use kar rahe hain: {CKPT}")

dev = "cuda" if torch.cuda.is_available() else "cpu"

def gen(prompt, chat=False, temp=0.3, top_k=10, rep=1.0, n=60, label=""):
    cmd = [sys.executable, "-u", "-m", "gamax1.generate",
           "--ckpt", str(CKPT), "--prompt", prompt,
           "--max_new_tokens", str(n),
           "--temperature", str(temp), "--top_k", str(top_k),
           "--repetition_penalty", str(rep), "--device", dev]
    if chat:
        cmd += ["--chat", "--stop_at_eos"]
    r = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    print(f"\n{'='*70}\n[{label}] PROMPT: {prompt!r}\n{'='*70}")
    print(r.stdout)
    if r.returncode != 0:
        print("--- stderr ---\n", r.stderr[-1500:])


# TEST 1: Declaration of Independence — exact mid-sentence cut (training text continues:
# "...created equal, that they are endowed by their Creator with certain unalienable Rights...")
gen("We hold these truths to be self-evident, that all men are",
    temp=0.3, top_k=10, rep=1.0, n=60, label="DOI - exact continuation test")

gen("He has refused his Assent to Laws, the most wholesome and",
    temp=0.3, top_k=10, rep=1.0, n=40, label="DOI - second passage")

# TEST 2: Chat-format recall from ultrachat training data (exact Q+A pair exists in corpus)
gen("Which famous landmarks should I visit in London, beyond the usual ones?",
    chat=True, temp=0.3, top_k=10, rep=1.0, n=80, label="ultrachat - London Q (exact match in training data)")

gen("Can you guide me through the process of enabling the secondary image hover feature on my Collection pages and Featured Collections sections?",
    chat=True, temp=0.3, top_k=10, rep=1.0, n=100, label="ultrachat - Shopify Q (exact match in training data)")

# TEST 3: Code-golf/<think> reasoning data — does think-tag structure show up?
gen("Consider the sequence 1/2, 1/3 + 1/2, 1/4 + 1/3 + 1/2, and so on.",
    chat=True, temp=0.3, top_k=10, rep=1.0, n=80, label="code-golf reasoning data - think-tag structure test")

In [ ]:
# ============================================================
# AETHERION — MAIN DRIVE READ-ONLY AUDIT
# NO DELETE / NO MOVE / NO OVERWRITE / NO TRAINING
# ============================================================

from pathlib import Path
import os

BASE = Path("/content/drive/MyDrive/Aetherion_GamaX1")

DATA = BASE / "data"
CACHE = BASE / "cache/gamax1_bulk"
RUNS = BASE / "runs/gamax1"
FULL = RUNS / "full"
EXPERIMENTS = RUNS / "experiments"

print("=" * 90)
print("AETHERION — MAIN DRIVE READ-ONLY AUDIT")
print("=" * 90)
print(f"Base        : {BASE}")
print(f"Data        : {DATA}")
print(f"Cache       : {CACHE}")
print(f"Full runs   : {FULL}")
print(f"Experiments : {EXPERIMENTS}")
print()

# ------------------------------------------------------------
# EXISTENCE CHECK
# ------------------------------------------------------------

print("[DIRECTORIES]")

for name, path in [
    ("BASE", BASE),
    ("DATA", DATA),
    ("CACHE", CACHE),
    ("FULL", FULL),
    ("EXPERIMENTS", EXPERIMENTS),
]:
    print(f"{name:12} : {'✓ EXISTS' if path.exists() else '✗ MISSING'}")
    print(f"             {path}")

# ------------------------------------------------------------
# CACHE FILES
# ------------------------------------------------------------

print()
print("=" * 90)
print("[CACHE FILES]")
print("=" * 90)

cache_files = [
    CACHE / "tokens.int32.bin",
    CACHE / "tokenizer.json",
    CACHE / "file_index.json",
    CACHE / "metadata.json",
]

for p in cache_files:
    if p.exists():
        size = p.stat().st_size
        print(f"✓ {p.name:22} {size:,} bytes ({size / (1024**3):.3f} GB)")
    else:
        print(f"✗ {p.name:22} MISSING")

# ------------------------------------------------------------
# TOKEN COUNT
# ------------------------------------------------------------

tokens = CACHE / "tokens.int32.bin"

print()
print("=" * 90)
print("[TOKEN CACHE]")
print("=" * 90)

if tokens.exists():
    size = tokens.stat().st_size
    token_count = size // 4

    print(f"File size   : {size:,} bytes")
    print(f"Token count : {token_count:,}")
else:
    print("tokens.int32.bin NOT FOUND")

# ------------------------------------------------------------
# CHECKPOINTS
# ------------------------------------------------------------

print()
print("=" * 90)
print("[CHECKPOINTS]")
print("=" * 90)

if FULL.exists():

    checkpoints = sorted(
        FULL.glob("gamax1_step_*.pt"),
        key=lambda p: int(
            p.stem.split("_")[-1]
        )
    )

    latest = FULL / "gamax1_latest.pt"

    print(f"Step checkpoints : {len(checkpoints)}")
    print()

    for i, p in enumerate(checkpoints, 1):
        size_gb = p.stat().st_size / (1024**3)
        print(f"{i:3}. {p.name:30} {size_gb:.3f} GB")

    print()
    print("-" * 90)

    if latest.exists():
        print(
            f"Latest checkpoint : ✓ {latest.name} "
            f"({latest.stat().st_size / (1024**3):.3f} GB)"
        )
    else:
        print("Latest checkpoint : ✗ MISSING")

else:
    print("FULL checkpoint directory does not exist.")

# ------------------------------------------------------------
# IMPORTANT: SEARCH OTHER AETHERION RUN CHECKPOINTS
# ------------------------------------------------------------

print()
print("=" * 90)
print("[ALL GAMAX1 CHECKPOINTS UNDER MAIN AETHERION DRIVE]")
print("=" * 90)

all_ckpts = sorted(
    BASE.rglob("gamax1_step_*.pt")
)

print(f"Total found: {len(all_ckpts)}")
print()

for p in all_ckpts:
    size_gb = p.stat().st_size / (1024**3)
    print(
        f"{size_gb:.3f} GB  |  "
        f"{p.relative_to(BASE)}"
    )

# ------------------------------------------------------------
# LATEST CHECKPOINTS
# ------------------------------------------------------------

print()
print("=" * 90)
print("[LATEST FILES]")
print("=" * 90)

latest_files = sorted(
    BASE.rglob("*latest*.pt")
)

print(f"Found: {len(latest_files)}")

for p in latest_files:
    size_gb = p.stat().st_size / (1024**3)
    print(
        f"{size_gb:.3f} GB  |  "
        f"{p.relative_to(BASE)}"
    )

# ------------------------------------------------------------
# FINAL SAFETY MESSAGE
# ------------------------------------------------------------

print()
print("=" * 90)
print("✓ AUDIT COMPLETE")
print("=" * 90)
print("READ-ONLY: Nothing was deleted.")
print("READ-ONLY: Nothing was moved.")
print("READ-ONLY: Nothing was overwritten.")
print("READ-ONLY: Training was NOT started.")
print("=" * 90)

In [ ]:
# ============================================================
# GAMAX1 BULK CACHE — TOKEN BACKUP AUDIT
# READ-ONLY — NOTHING WILL BE DELETED/MOVED/OVERWRITTEN
# ============================================================

from pathlib import Path

CACHE_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/gamax1_bulk"
)

EXPECTED_TOKENS = 1_170_998_016

print("=" * 110)
print("GAMAX1 BULK CACHE — TOKEN BACKUP AUDIT")
print("=" * 110)
print(f"Directory: {CACHE_DIR}")
print()

if not CACHE_DIR.exists():
    raise FileNotFoundError(f"Cache directory not found:\n{CACHE_DIR}")

# ------------------------------------------------------------
# Find ALL token binary files and backups
# ------------------------------------------------------------

files = sorted(
    CACHE_DIR.glob("tokens.int32.bin*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

print(f"Token files found: {len(files)}")
print()

print("-" * 110)
print(f"{'FILE':<75} {'SIZE (GB)':>10} {'TOKENS':>18} {'MATCH':>8}")
print("-" * 110)

matches = []

for path in files:
    size_bytes = path.stat().st_size

    # int32 = 4 bytes/token
    if size_bytes % 4 == 0:
        token_count = size_bytes // 4
    else:
        token_count = -1

    size_gb = size_bytes / (1024 ** 3)

    is_match = token_count == EXPECTED_TOKENS

    if is_match:
        matches.append(path)

    match_text = "YES" if is_match else ""

    print(
        f"{path.name:<75} "
        f"{size_gb:>10.3f} "
        f"{token_count:>18,} "
        f"{match_text:>8}"
    )

print("-" * 110)

# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

print()
print("=" * 110)
print("RESULT")
print("=" * 110)

if matches:
    print()
    print("🎯 FOUND 1.17B-TOKEN CACHE:")
    print()

    for p in matches:
        size_gb = p.stat().st_size / (1024 ** 3)
        print(f"✓ {p}")
        print(f"  Size   : {size_gb:.3f} GB")
        print(f"  Tokens : {EXPECTED_TOKENS:,}")
        print()

    print("IMPORTANT:")
    print("Do NOT replace tokens.int32.bin yet.")
    print("Do NOT delete any .bak file.")
    print("First verify which backup is the correct cache.")
else:
    print()
    print("❌ No file with exactly 1,170,998,016 tokens was found.")
    print()
    print("The audit above shows the exact size/token count")
    print("of every tokens.int32.bin* file.")

print()
print("=" * 110)
print("READ-ONLY AUDIT COMPLETE")
print("✓ Nothing deleted")
print("✓ Nothing moved")
print("✓ Nothing renamed")
print("✓ Nothing overwritten")
print("=" * 110)

In [ ]:
# ============================================================
# GAMAX1 — 1.17B CACHE COMPATIBILITY AUDIT
# READ-ONLY — NO COPY / NO MOVE / NO DELETE / NO OVERWRITE
# ============================================================

from pathlib import Path
import json
import hashlib

CACHE_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/gamax1_bulk"
)

OLD_CACHE = CACHE_DIR / (
    "tokens.int32.bin.changed_files.20260929_075826.bak"
)

CURRENT_TOKENS = CACHE_DIR / "tokens.int32.bin"
FILE_INDEX = CACHE_DIR / "file_index.json"
METADATA = CACHE_DIR / "metadata.json"
TOKENIZER = CACHE_DIR / "tokenizer.json"

EXPECTED_TOKENS = 1_170_998_016

print("=" * 100)
print("GAMAX1 — 1.17B CACHE COMPATIBILITY AUDIT")
print("=" * 100)
print("READ-ONLY MODE")
print()

# ------------------------------------------------------------
# 1. EXISTENCE
# ------------------------------------------------------------

paths = {
    "OLD 1.17B TOKEN CACHE": OLD_CACHE,
    "CURRENT TOKEN CACHE": CURRENT_TOKENS,
    "FILE INDEX": FILE_INDEX,
    "METADATA": METADATA,
    "TOKENIZER": TOKENIZER,
}

print("[1] FILE EXISTENCE")
print("-" * 100)

for name, path in paths.items():
    status = "✓ EXISTS" if path.exists() else "❌ MISSING"
    print(f"{name:<28}: {status}")
    print(f"  {path}")

# ------------------------------------------------------------
# 2. TOKEN COUNTS
# ------------------------------------------------------------

print()
print("[2] TOKEN COUNTS")
print("-" * 100)

for name, path in [
    ("OLD 1.17B CACHE", OLD_CACHE),
    ("CURRENT CACHE", CURRENT_TOKENS),
]:

    if not path.exists():
        print(f"{name:<20}: MISSING")
        continue

    size = path.stat().st_size

    if size % 4 != 0:
        print(f"{name:<20}: INVALID int32 size ({size:,} bytes)")
        continue

    tokens = size // 4

    print(f"{name:<20}: {tokens:,} tokens")
    print(f"{'':<20}  {size / (1024**3):.3f} GB")

    if tokens == EXPECTED_TOKENS:
        print(f"{'':<20}  ✓ EXACT 1.17B MATCH")
    else:
        print(f"{'':<20}  — different token count")

# ------------------------------------------------------------
# 3. FILE SIZES
# ------------------------------------------------------------

print()
print("[3] SUPPORT FILE SIZES")
print("-" * 100)

for name, path in [
    ("file_index.json", FILE_INDEX),
    ("metadata.json", METADATA),
    ("tokenizer.json", TOKENIZER),
]:

    if path.exists():
        size = path.stat().st_size
        print(f"{name:<22}: {size:,} bytes ({size/(1024**2):.3f} MB)")
    else:
        print(f"{name:<22}: ❌ MISSING")

# ------------------------------------------------------------
# 4. JSON VALIDATION
# ------------------------------------------------------------

print()
print("[4] JSON VALIDATION")
print("-" * 100)

json_objects = {}

for name, path in [
    ("file_index.json", FILE_INDEX),
    ("metadata.json", METADATA),
    ("tokenizer.json", TOKENIZER),
]:

    if not path.exists():
        print(f"{name:<22}: ❌ MISSING")
        continue

    try:
        with open(path, "r", encoding="utf-8") as f:
            obj = json.load(f)

        json_objects[name] = obj

        if isinstance(obj, dict):
            print(
                f"{name:<22}: ✓ VALID JSON "
                f"(dict, {len(obj):,} top-level keys)"
            )
        elif isinstance(obj, list):
            print(
                f"{name:<22}: ✓ VALID JSON "
                f"(list, {len(obj):,} items)"
            )
        else:
            print(f"{name:<22}: ✓ VALID JSON")

    except Exception as e:
        print(f"{name:<22}: ❌ INVALID")
        print(f"  Error: {e}")

# ------------------------------------------------------------
# 5. SEARCH METADATA FOR TOKEN INFORMATION
# ------------------------------------------------------------

print()
print("[5] METADATA TOKEN INFORMATION")
print("-" * 100)

def find_token_values(obj, prefix=""):
    results = []

    if isinstance(obj, dict):
        for k, v in obj.items():
            key = f"{prefix}.{k}" if prefix else str(k)

            if isinstance(v, (int, float)):
                key_lower = key.lower()

                if any(x in key_lower for x in [
                    "token",
                    "count",
                    "length",
                    "size",
                ]):
                    results.append((key, v))

            elif isinstance(v, (dict, list)):
                results.extend(find_token_values(v, key))

    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            if isinstance(v, (dict, list)):
                results.extend(
                    find_token_values(v, f"{prefix}[{i}]")
                )

    return results


if "metadata.json" in json_objects:

    values = find_token_values(
        json_objects["metadata.json"]
    )

    if values:
        for key, value in values[:100]:
            print(f"{key:<60}: {value}")
    else:
        print("No obvious token/count metadata fields found.")

# ------------------------------------------------------------
# 6. HASH OLD CACHE
# ------------------------------------------------------------

print()
print("[6] SHA256 OF 1.17B CACHE")
print("-" * 100)

if OLD_CACHE.exists():

    print("Calculating SHA256...")
    print("This is READ-ONLY and may take some time for 4.36 GB.")

    sha = hashlib.sha256()

    with open(OLD_CACHE, "rb") as f:
        while True:
            chunk = f.read(16 * 1024 * 1024)

            if not chunk:
                break

            sha.update(chunk)

    print()
    print("OLD CACHE SHA256:")
    print(sha.hexdigest())

# ------------------------------------------------------------
# 7. FINAL VERDICT
# ------------------------------------------------------------

print()
print("=" * 100)
print("AUDIT RESULT")
print("=" * 100)

old_ok = (
    OLD_CACHE.exists()
    and OLD_CACHE.stat().st_size % 4 == 0
    and OLD_CACHE.stat().st_size // 4 == EXPECTED_TOKENS
)

support_ok = all(
    p.exists()
    for p in [FILE_INDEX, METADATA, TOKENIZER]
)

json_ok = all(
    name in json_objects
    for name in [
        "file_index.json",
        "metadata.json",
        "tokenizer.json",
    ]
)

print()
print(f"1.17B token cache found : {'✓ YES' if old_ok else '❌ NO'}")
print(f"Support files present   : {'✓ YES' if support_ok else '❌ NO'}")
print(f"JSON files readable     : {'✓ YES' if json_ok else '❌ NO'}")

print()
print("IMPORTANT:")
print("This audit does NOT modify the cache.")
print("Do NOT restore/copy the .bak yet.")
print("Do NOT start training from the backup yet.")

if old_ok and support_ok and json_ok:
    print()
    print("✓ The 1.17B token backup exists and all current support")
    print("  files are present/readable.")
    print()
    print("Next step: compare cache identity/metadata before restoring.")
else:
    print()
    print("⚠ Compatibility is NOT fully established yet.")

print()
print("=" * 100)
print("READ-ONLY AUDIT COMPLETE")
print("✓ Nothing deleted")
print("✓ Nothing moved")
print("✓ Nothing renamed")
print("✓ Nothing overwritten")
print("=" * 100)

In [ ]:
# ============================================================
# GAMAX1 — 1.17B CACHE IDENTITY / COMPATIBILITY CHECK
# READ-ONLY — NOTHING WILL BE MODIFIED
# ============================================================

from pathlib import Path
import json
import hashlib
import struct

CACHE_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/gamax1_bulk"
)

OLD_TOKENS = CACHE_DIR / (
    "tokens.int32.bin.changed_files.20260929_075826.bak"
)

CURRENT_TOKENS = CACHE_DIR / "tokens.int32.bin"
FILE_INDEX = CACHE_DIR / "file_index.json"
METADATA = CACHE_DIR / "metadata.json"
TOKENIZER = CACHE_DIR / "tokenizer.json"

print("=" * 100)
print("GAMAX1 — 1.17B CACHE IDENTITY / COMPATIBILITY CHECK")
print("=" * 100)
print("READ-ONLY MODE")
print()

# ------------------------------------------------------------
# 1. EXISTENCE
# ------------------------------------------------------------

files = {
    "OLD 1.17B TOKEN CACHE": OLD_TOKENS,
    "CURRENT TOKEN CACHE": CURRENT_TOKENS,
    "FILE INDEX": FILE_INDEX,
    "METADATA": METADATA,
    "TOKENIZER": TOKENIZER,
}

print("[1] FILE EXISTENCE")
print("-" * 100)

for name, path in files.items():
    status = "✓ EXISTS" if path.exists() else "✗ MISSING"
    print(f"{name:<25}: {status}")
    print(f"  {path}")

if not all(p.exists() for p in files.values()):
    raise RuntimeError("SAFETY STOP: Required file missing.")

# ------------------------------------------------------------
# 2. TOKEN COUNTS
# ------------------------------------------------------------

print()
print("[2] TOKEN COUNTS")
print("-" * 100)

old_tokens = OLD_TOKENS.stat().st_size // 4
current_tokens = CURRENT_TOKENS.stat().st_size // 4

print(f"OLD 1.17B BACKUP : {old_tokens:,}")
print(f"CURRENT CACHE    : {current_tokens:,}")

EXPECTED = 1_170_998_016

old_ok = old_tokens == EXPECTED

print()
print(f"Expected 1.17B   : {EXPECTED:,}")
print(f"Old backup match : {'✓ YES' if old_ok else '✗ NO'}")

# ------------------------------------------------------------
# 3. LOAD JSON — READ ONLY
# ------------------------------------------------------------

print()
print("[3] JSON / METADATA")
print("-" * 100)

with open(METADATA, "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(FILE_INDEX, "r", encoding="utf-8") as f:
    file_index = json.load(f)

with open(TOKENIZER, "r", encoding="utf-8") as f:
    tokenizer = json.load(f)

metadata_token_count = metadata.get("token_count")
metadata_vocab = metadata.get("vocab_size")
metadata_file_count = metadata.get("file_count")

print(f"metadata.token_count : {metadata_token_count:,}")
print(f"metadata.vocab_size  : {metadata_vocab}")
print(f"metadata.file_count  : {metadata_file_count:,}")

# ------------------------------------------------------------
# 4. TOKEN COUNT IDENTITY
# ------------------------------------------------------------

print()
print("[4] TOKEN COUNT IDENTITY")
print("-" * 100)

checks = []

check = old_tokens == metadata_token_count
checks.append(check)

print(
    "OLD backup token count == metadata.token_count : "
    + ("✓ MATCH" if check else "✗ MISMATCH")
)

check = old_tokens == EXPECTED
checks.append(check)

print(
    "OLD backup == expected 1,170,998,016           : "
    + ("✓ MATCH" if check else "✗ MISMATCH")
)

# ------------------------------------------------------------
# 5. SOURCE TOKEN COUNTS
# ------------------------------------------------------------

print()
print("[5] SOURCE TOKEN COUNTS")
print("-" * 100)

source_counts = metadata.get("source_token_counts", {})

source_total = 0

for source, count in source_counts.items():
    print(f"{source:<55}: {count:,}")
    source_total += int(count)

print("-" * 100)
print(f"{'SUM OF SOURCE TOKENS':<55}: {source_total:,}")
print(f"{'OLD BACKUP TOKENS':<55}: {old_tokens:,}")

source_match = source_total == old_tokens

print()
print(
    "Source-token total == old backup : "
    + ("✓ MATCH" if source_match else "✗ MISMATCH")
)

checks.append(source_match)

# ------------------------------------------------------------
# 6. FILE COUNT / FILE INDEX
# ------------------------------------------------------------

print()
print("[6] FILE INDEX IDENTITY")
print("-" * 100)

print("file_index type :", type(file_index).__name__)

if isinstance(file_index, dict):
    print("file_index keys :", list(file_index.keys()))

# Try to discover file-count information without modifying anything
possible_counts = {}

for key, value in file_index.items() if isinstance(file_index, dict) else []:
    if isinstance(value, (list, tuple, dict)):
        possible_counts[key] = len(value)

for key, count in possible_counts.items():
    print(f"{key:<35}: {count:,}")

# ------------------------------------------------------------
# 7. TOKENIZER IDENTITY
# ------------------------------------------------------------

print()
print("[7] TOKENIZER IDENTITY")
print("-" * 100)

print("tokenizer type :", type(tokenizer).__name__)

if isinstance(tokenizer, dict):
    print("tokenizer keys :", list(tokenizer.keys()))

tokenizer_vocab = None

# HuggingFace-style tokenizer.json normally contains model.vocab
if isinstance(tokenizer, dict):
    model = tokenizer.get("model")

    if isinstance(model, dict):
        vocab = model.get("vocab")

        if isinstance(vocab, dict):
            tokenizer_vocab = len(vocab)

if tokenizer_vocab is not None:
    print(f"Tokenizer vocab entries : {tokenizer_vocab:,}")
    print(f"Metadata vocab size      : {metadata_vocab:,}")

    vocab_match = tokenizer_vocab == metadata_vocab

    print(
        "Tokenizer ↔ metadata vocab : "
        + ("✓ MATCH" if vocab_match else "✗ MISMATCH")
    )

    checks.append(vocab_match)
else:
    print("⚠ Could not determine tokenizer vocabulary count automatically.")

# ------------------------------------------------------------
# 8. SHA256 — OLD BACKUP
# ------------------------------------------------------------

print()
print("[8] OLD 1.17B CACHE SHA256")
print("-" * 100)

print("Calculating SHA256 of 4.36 GB backup...")
print("This is READ-ONLY.")

sha256 = hashlib.sha256()

with open(OLD_TOKENS, "rb") as f:
    while True:
        chunk = f.read(64 * 1024 * 1024)

        if not chunk:
            break

        sha256.update(chunk)

old_hash = sha256.hexdigest()

print()
print("OLD BACKUP SHA256:")
print(old_hash)

# ------------------------------------------------------------
# 9. FINAL DECISION
# ------------------------------------------------------------

print()
print("=" * 100)
print("COMPATIBILITY RESULT")
print("=" * 100)

print()
print(f"1.17B token backup exists       : {'✓ YES' if old_tokens else '✗ NO'}")
print(f"Exact 1,170,998,016 tokens      : {'✓ YES' if old_ok else '✗ NO'}")
print(f"Metadata token count matches    : "
      f"{'✓ YES' if old_tokens == metadata_token_count else '✗ NO'}")
print(f"Source token total matches      : "
      f"{'✓ YES' if source_match else '✗ NO'}")

print()

if all(checks):
    print("🎯 RESULT: CACHE IDENTITY CHECK PASSED")
    print()
    print("✓ The 1.17B-token backup matches the current metadata")
    print("✓ Source token totals match")
    print("✓ No files were modified")
    print()
    print("IMPORTANT:")
    print("The backup has NOT been restored.")
    print("Training has NOT been started.")
else:
    print("⚠ RESULT: COMPATIBILITY CHECK DID NOT FULLY PASS")
    print()
    print("DO NOT restore the .bak yet.")
    print("DO NOT start training.")

print()
print("=" * 100)
print("READ-ONLY CHECK COMPLETE")
print("✓ Nothing deleted")
print("✓ Nothing moved")
print("✓ Nothing renamed")
print("✓ Nothing overwritten")
print("✓ Training NOT started")
print("=" * 100)

-------------------------------------------------------------Test---------------------

In [ ]:
# ============================================================
# GAMAX1 — VERIFIED 1.17B TOKEN CACHE RESTORE
# SAFE RESTORE — NO CORPUS REBUILD
# ============================================================

from pathlib import Path
import shutil
from datetime import datetime

print("=" * 100)
print("GAMAX1 — VERIFIED CACHE RESTORE")
print("=" * 100)

# ------------------------------------------------------------
# ESTABLISHED PATH
# ------------------------------------------------------------

CACHE_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/gamax1_bulk"
)

ACTIVE = CACHE_DIR / "tokens.int32.bin"

# Exact verified 1.17B-token backup
VERIFIED_BACKUP = CACHE_DIR / (
    "tokens.int32.bin.changed_files.20260929_075826.bak"
)

EXPECTED_TOKENS = 1_170_998_016
BYTES_PER_TOKEN = 4

# ------------------------------------------------------------
# CHECKS
# ------------------------------------------------------------

print("\n[SAFETY CHECK]")

if not CACHE_DIR.exists():
    raise FileNotFoundError(
        f"Cache directory not found:\n{CACHE_DIR}"
    )

if not VERIFIED_BACKUP.exists():
    raise FileNotFoundError(
        f"Verified 1.17B-token backup not found:\n"
        f"{VERIFIED_BACKUP}"
    )

backup_size = VERIFIED_BACKUP.stat().st_size
backup_tokens = backup_size // BYTES_PER_TOKEN

print("Verified backup:")
print(" ", VERIFIED_BACKUP)
print()
print("Size        :", f"{backup_size / (1024**3):.2f} GB")
print("Token count :", f"{backup_tokens:,}")

if backup_tokens != EXPECTED_TOKENS:
    raise RuntimeError(
        "SAFETY STOP!\n"
        f"Expected : {EXPECTED_TOKENS:,}\n"
        f"Found    : {backup_tokens:,}"
    )

print("✓ Verified 1.17B-token backup")

# ------------------------------------------------------------
# VERIFY METADATA
# ------------------------------------------------------------

METADATA = CACHE_DIR / "metadata.json"
FILE_INDEX = CACHE_DIR / "file_index.json"

if not METADATA.exists():
    raise FileNotFoundError("metadata.json missing")

if not FILE_INDEX.exists():
    raise FileNotFoundError("file_index.json missing")

import json

with open(METADATA, "r") as f:
    metadata = json.load(f)

with open(FILE_INDEX, "r") as f:
    file_index = json.load(f)

if metadata.get("token_count") != EXPECTED_TOKENS:
    raise RuntimeError(
        "Metadata token count does not match verified cache."
    )

if file_index.get("token_count") != EXPECTED_TOKENS:
    raise RuntimeError(
        "file_index token count does not match verified cache."
    )

if len(file_index.get("files", {})) != 34_436:
    raise RuntimeError(
        "file_index file count does not match expected 34,436."
    )

print("✓ metadata.json matches 1.17B tokens")
print("✓ file_index.json matches 1.17B tokens / 34,436 files")

# ------------------------------------------------------------
# BACKUP CURRENT ACTIVE FILE
# ------------------------------------------------------------

if ACTIVE.exists():

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    CURRENT_BACKUP = CACHE_DIR / (
        f"tokens.int32.bin.before_restore.{timestamp}.bak"
    )

    print()
    print("Backing up current active token file:")
    print(" ", ACTIVE)
    print("→", CURRENT_BACKUP)

    shutil.copy2(ACTIVE, CURRENT_BACKUP)

    print("✓ Current active file preserved")

else:
    print()
    print("⚠️ Active tokens.int32.bin does not exist.")
    print("No active file needs to be backed up.")

# ------------------------------------------------------------
# RESTORE VERIFIED STREAM
# ------------------------------------------------------------

print()
print("=" * 100)
print("RESTORING VERIFIED 1.17B-TOKEN STREAM")
print("=" * 100)

shutil.copy2(
    VERIFIED_BACKUP,
    ACTIVE
)

print("✓ Restore completed")

# ------------------------------------------------------------
# FINAL VERIFICATION
# ------------------------------------------------------------

print()
print("=" * 100)
print("RESTORE VERIFICATION")
print("=" * 100)

active_size = ACTIVE.stat().st_size
active_tokens = active_size // BYTES_PER_TOKEN

print("Active file:")
print(" ", ACTIVE)
print()
print("Active size   :", f"{active_size / (1024**3):.2f} GB")
print("Active tokens :", f"{active_tokens:,}")

if active_tokens != EXPECTED_TOKENS:
    raise RuntimeError(
        "RESTORE VERIFICATION FAILED!\n"
        f"Expected : {EXPECTED_TOKENS:,}\n"
        f"Found    : {active_tokens:,}"
    )

print()
print("🎉 VERIFIED CACHE RESTORED")
print()
print("✓ 1,170,998,016 tokens active")
print("✓ 34,436-file index preserved")
print("✓ metadata preserved")
print("✓ Original verified .bak preserved")
print("✓ Previous active token file preserved")
print("✓ NO corpus rebuild performed")
print()
print("=" * 100)
print("DO NOT RUN --rebuild_bulk_cache")
print("=" * 100)

In [ ]:
# ============================================================
# GAMAX1 / AETHERION — FINAL LIVE RESUME
# 43,500 -> 100,000 STEPS
# VERIFIED 1.17B CACHE
# LIVE STDOUT/STDERR
# NO CACHE REBUILD
# ============================================================

from pathlib import Path
import subprocess
import sys
import os
import torch

# ------------------------------------------------------------
# PATHS — VERIFIED NEW ACCOUNT
# ------------------------------------------------------------

REPO_DIR = Path("/content/gamax1_project")

DATA_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/data"
)

CACHE_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/gamax1_bulk"
)

OUT_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full"
)

EXPERIMENT_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/experiments"
)

CHECKPOINT = OUT_DIR / "gamax1_latest.pt"

MAX_STEPS = 100_000
EXPECTED_TOKENS = 1_170_998_016

# ------------------------------------------------------------
# HEADER
# ------------------------------------------------------------

print("=" * 100)
print("GAMAX1 / AETHERION — FINAL LIVE RESUME")
print("=" * 100)

print("Repository :", REPO_DIR)
print("Data       :", DATA_DIR)
print("Cache      :", CACHE_DIR)
print("Output     :", OUT_DIR)
print("Checkpoint :", CHECKPOINT)
print()

# ------------------------------------------------------------
# BASIC PATH SAFETY
# ------------------------------------------------------------

for name, path in {
    "Repository": REPO_DIR,
    "Data": DATA_DIR,
    "Cache": CACHE_DIR,
    "Output": OUT_DIR,
    "Checkpoint": CHECKPOINT,
}.items():

    if not path.exists():
        raise FileNotFoundError(
            f"{name} not found:\n{path}"
        )

    print(f"✓ {name}: {path}")

# ------------------------------------------------------------
# CACHE VERIFICATION
# ------------------------------------------------------------

TOKENS = CACHE_DIR / "tokens.int32.bin"
TOKENIZER = CACHE_DIR / "tokenizer.json"
FILE_INDEX = CACHE_DIR / "file_index.json"
METADATA = CACHE_DIR / "metadata.json"

print()
print("=" * 100)
print("CACHE VERIFICATION")
print("=" * 100)

if not TOKENS.exists():
    raise FileNotFoundError(
        f"Active token file missing:\n{TOKENS}"
    )

token_count = TOKENS.stat().st_size // 4

print(
    "Active token file :",
    f"{TOKENS.stat().st_size / (1024**3):.2f} GB"
)

print(
    "Token count       :",
    f"{token_count:,}"
)

if token_count != EXPECTED_TOKENS:
    raise RuntimeError(
        "SAFETY STOP — ACTIVE CACHE IS NOT THE VERIFIED CACHE\n"
        f"Expected : {EXPECTED_TOKENS:,}\n"
        f"Found    : {token_count:,}"
    )

for p in [TOKENIZER, FILE_INDEX, METADATA]:

    if not p.exists():
        raise FileNotFoundError(
            f"Required cache file missing:\n{p}"
        )

print("✓ 1,170,998,016-token cache VERIFIED")
print("✓ tokenizer.json present")
print("✓ file_index.json present")
print("✓ metadata.json present")
print("✓ NO cache rebuild")

# ------------------------------------------------------------
# CHECKPOINT VERIFICATION
# ------------------------------------------------------------

print()
print("=" * 100)
print("CHECKPOINT VERIFICATION")
print("=" * 100)

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

current_step = int(
    checkpoint.get("step", 0)
)

print("Checkpoint step :", f"{current_step:,}")
print("Target step     :", f"{MAX_STEPS:,}")
print(
    "Remaining       :",
    f"{MAX_STEPS - current_step:,}"
)

if current_step != 95_000:
    raise RuntimeError(
        "SAFETY STOP — unexpected checkpoint step.\n"
        f"Expected : 95,000\n"
        f"Found    : {current_step:,}"
    )

print("✓ Correct 43,500 checkpoint verified")

# ------------------------------------------------------------
# CUDA
# ------------------------------------------------------------

print()
print("=" * 100)
print("DEVICE VERIFICATION")
print("=" * 100)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available. Training NOT started."
    )

print("✓ CUDA available")
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# TRAINING COMMAND
# ------------------------------------------------------------

cmd = [
    sys.executable,
    "-u",
    "-m",
    "gamax1.train",

    "--data_dir",
    str(DATA_DIR),

    "--bulk_cache_dir",
    str(CACHE_DIR),

    "--out_dir",
    str(OUT_DIR),

    "--resume_from",
    str(CHECKPOINT),

    "--d_model",
    "768",

    "--n_heads",
    "12",

    "--n_layers",
    "12",

    "--n_features",
    "3072",

    "--block_size",
    "512",

    "--batch_size",
    "16",

    "--lr",
    "0.0003",

    "--weight_decay",
    "0.01",

    "--dropout",
    "0.1",

    "--max_steps",
    str(MAX_STEPS),

    "--warmup_steps",
    "500",

    "--eval_interval",
    "200",

    "--eval_batches",
    "10",

    "--checkpoint_interval",
    "500",

    "--validation_split",
    "tail",

    "--validation_split_seed",
    "42",

    "--tokenizer",
    "bpe",

    "--bpe_vocab_size",
    "16000",

    "--bpe_sample_chars",
    "3000000",

    "--experiment_dir",
    str(EXPERIMENT_DIR),

    "--run_name",
    "full_v7_master",

    "--device",
    "cuda",
]

# ------------------------------------------------------------
# HARD SAFETY CHECK
# ------------------------------------------------------------

if "--rebuild_bulk_cache" in cmd:
    raise RuntimeError(
        "SAFETY STOP: rebuild flag detected!"
    )

print()
print("=" * 100)
print("TRAINING CONFIGURATION")
print("=" * 100)

print("Resume       :", f"{current_step:,}")
print("Target       :", f"{MAX_STEPS:,}")
print("Remaining    :", f"{MAX_STEPS-current_step:,}")
print("Cache        :", "VERIFIED 1.17B TOKENS")
print("Rebuild      :", "DISABLED")
print("Device       :", "CUDA")
print("Live output  :", "ENABLED")
print()
print("✓ --rebuild_bulk_cache NOT present")

# ------------------------------------------------------------
# ENVIRONMENT FOR LIVE OUTPUT
# ------------------------------------------------------------

env = os.environ.copy()

# Force Python/unbuffered output
env["PYTHONUNBUFFERED"] = "1"

# Prevent some terminal buffering
env["TERM"] = "xterm-256color"

print()
print("=" * 100)
print("🚀 STARTING TRAINING")
print("=" * 100)
print()

# ------------------------------------------------------------
# LIVE PROCESS STREAM
# ------------------------------------------------------------

process = subprocess.Popen(
    cmd,
    cwd=str(REPO_DIR),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Print every line as soon as it reaches the parent process
try:

    for line in iter(process.stdout.readline, ""):

        if line:
            print(
                line,
                end="",
                flush=True
            )

finally:

    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()

# ------------------------------------------------------------
# RESULT
# ------------------------------------------------------------

print()
print("=" * 100)

if return_code == 0:

    print("🎉 GAMAX1 TRAINING FINISHED")
    print("=" * 100)

else:

    print(
        f"❌ Training exited with code {return_code}"
    )
    print("=" * 100)

    raise RuntimeError(
        f"gamax1.train exited with code {return_code}"
    )

In [ ]:
import subprocess, sys, torch

CKPT = RUN_ROOT / "full" / "gamax1.pt"   # final 100k checkpoint
dev = "cuda" if torch.cuda.is_available() else "cpu"

def gen(prompt, chat=False, temp=0.8, top_k=40, rep=1.2, n=80, label=""):
    cmd = [sys.executable, "-u", "-m", "gamax1.generate",
           "--ckpt", str(CKPT), "--prompt", prompt,
           "--max_new_tokens", str(n),
           "--temperature", str(temp), "--top_k", str(top_k),
           "--repetition_penalty", str(rep), "--device", dev]
    if chat:
        cmd += ["--chat", "--stop_at_eos"]
    r = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    print(f"\n{'='*70}\n[{label}] PROMPT: {prompt!r}\n{'='*70}")
    print(r.stdout)
    if r.returncode != 0:
        print("--- stderr ---\n", r.stderr[-1500:])


# PART A: Normal sampling — directly comparable to earlier 30k/44k tests
gen("What is the capital of India?", chat=True, label="A1: Chat fact - India capital")
gen("Who wrote Hamlet?", chat=True, label="A2: Chat fact - Hamlet author")
gen("Once upon a time", label="A3: Base fluency")
gen("The capital of India is", label="A4: Base fact")

# PART B: Low-noise / near-deterministic
gen("We hold these truths to be self-evident, that all men are",
    temp=0.3, top_k=10, rep=1.0, n=60, label="B1: DOI - exact continuation")
gen("2 + 2 =", temp=0.3, top_k=10, rep=1.0, n=20, label="B2: Arithmetic")

# PART C: Training-data recall (exact ultrachat Q, exact answer exists in corpus)
gen("Which famous landmarks should I visit in London, beyond the usual ones?",
    chat=True, temp=0.3, top_k=10, rep=1.0, n=80, label="C1: ultrachat London Q")
gen("Can you guide me through the process of enabling the secondary image hover feature on my Collection pages and Featured Collections sections?",
    chat=True, temp=0.3, top_k=10, rep=1.0, n=100, label="C2: ultrachat Shopify Q")

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO_DIR = Path("/content/gamax1_project")
BASE_CKPT = RUN_ROOT / "full" / "gamax1.pt"          # final 100k checkpoint

QA_DATA = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA/")
FT_OUT_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA")
FT_OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"Base checkpoint": BASE_CKPT, "QA data": QA_DATA}.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found:\n{path}")
    print(f"✓ {name}: {path}")

if QA_DATA.is_dir():
    found = list(QA_DATA.rglob("*.jsonl")) + list(QA_DATA.rglob("*.json"))
    print(f"  → {len(found)} Q&A file(s) found: {[f.name for f in found]}")

cmd = [
    sys.executable, "-u", "-m", "gamax1.finetune",
    "--init_from", str(BASE_CKPT),
    "--data", str(QA_DATA),
    "--out_dir", str(FT_OUT_DIR),
    "--max_len", "128",
    "--batch_size", "8",
    "--epochs", "3",
    "--lr", "3e-5",
    "--warmup_steps", "50",
    "--val_fraction", "0.05",
    "--eval_interval", "50",
    "--checkpoint_interval", "100",
    "--freeze_sparsity",
    "--seed", "0",
    "--device", "cuda",
]

print("\n" + "=" * 100)
print("STARTING FINE-TUNE: Q&A (QnA folder) on final 100k checkpoint")
print("=" * 100)
print(" ".join(cmd))
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    cmd, cwd=str(REPO_DIR), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in iter(process.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)
finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()
print("\n" + "=" * 100)
if return_code == 0:
    print("🎉 FINE-TUNE FINISHED")
else:
    print(f"❌ Fine-tune exited with code {return_code}")
print("=" * 100)

In [ ]:
import subprocess, sys, torch
from pathlib import Path

FT_CKPT = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA/gamax1_finetune_best.pt"
)
dev = "cuda" if torch.cuda.is_available() else "cpu"

def gen(prompt, chat=True, temp=0.6, top_k=30, rep=1.15, n=100, label=""):
    cmd = [sys.executable, "-u", "-m", "gamax1.generate",
           "--ckpt", str(FT_CKPT), "--prompt", prompt,
           "--max_new_tokens", str(n),
           "--temperature", str(temp), "--top_k", str(top_k),
           "--repetition_penalty", str(rep), "--device", dev]
    if chat:
        cmd += ["--chat", "--stop_at_eos"]
    r = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    print(f"\n{'='*70}\n[{label}] PROMPT: {prompt!r}\n{'='*70}")
    print(r.stdout)
    if r.returncode != 0:
        print("--- stderr ---\n", r.stderr[-1500:])


# Same questions jo base-model pe galat aaye the -- seedha before/after compare
gen("What is the capital of India?", label="1: India capital (was wrong on base model)")
gen("Who wrote Hamlet?", label="2: Hamlet author (was wrong on base model)")

# In-domain: geography/polity/science (training-data category ke andar)
gen("What is the capital of Rajasthan?", label="3: In-domain geography")
gen("Who is the President of India?", label="4: In-domain polity")
gen("What is the chemical symbol for gold?", label="5: In-domain science")
gen("How many fundamental rights are there in the Indian Constitution?", label="6: In-domain polity")

# Fluency sanity check -- general writing ability bachi hai ya nahi
gen("Once upon a time", chat=False, temp=0.8, top_k=40, rep=1.2, n=80,
    label="7: Base-mode fluency check (not a QA prompt)")

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO_DIR = Path("/content/gamax1_project")

INIT_CKPT = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA/gamax1_finetune_best.pt"
)
QA_DATA = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA/")
FT_OUT_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v2")
FT_OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"Init checkpoint": INIT_CKPT, "QA data": QA_DATA}.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found:\n{path}")
    print(f"✓ {name}: {path}")

cmd = [
    sys.executable, "-u", "-m", "gamax1.finetune",
    "--init_from", str(INIT_CKPT),
    "--data", str(QA_DATA),
    "--out_dir", str(FT_OUT_DIR),
    "--max_len", "128",
    "--batch_size", "8",
    "--epochs", "20",
    "--lr", "3e-5",
    "--warmup_steps", "50",
    "--val_fraction", "0.05",
    "--eval_interval", "100",
    "--checkpoint_interval", "300",
    "--freeze_sparsity",
    "--seed", "0",
    "--device", "cuda",
]

print("\n" + "=" * 100)
print("EXTENDED FINE-TUNE (20 epochs): pushing for fact memorization")
print("=" * 100)
print(" ".join(cmd))
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    cmd, cwd=str(REPO_DIR), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in iter(process.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)
finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()
print("\n" + "=" * 100)
if return_code == 0:
    print("🎉 EXTENDED FINE-TUNE FINISHED")
else:
    print(f"❌ Extended fine-tune exited with code {return_code}")
print("=" * 100)

In [ ]:
import subprocess, sys, torch
from pathlib import Path

FT_CKPT = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v2/gamax1_finetune_best.pt"
)
dev = "cuda" if torch.cuda.is_available() else "cpu"

def gen(prompt, chat=True, temp=0.6, top_k=30, rep=1.15, n=100, label=""):
    cmd = [sys.executable, "-u", "-m", "gamax1.generate",
           "--ckpt", str(FT_CKPT), "--prompt", prompt,
           "--max_new_tokens", str(n),
           "--temperature", str(temp), "--top_k", str(top_k),
           "--repetition_penalty", str(rep), "--device", dev]
    if chat:
        cmd += ["--chat", "--stop_at_eos"]
    r = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    print(f"\n{'='*70}\n[{label}] PROMPT: {prompt!r}\n{'='*70}")
    print(r.stdout)
    if r.returncode != 0:
        print("--- stderr ---\n", r.stderr[-1500:])

gen("What is the capital of India?", label="1: India capital")
gen("Who wrote Hamlet?", label="2: Hamlet author")
gen("What is the capital of Rajasthan?", label="3: Geography")
gen("Who is the President of India?", label="4: Polity")
gen("What is the chemical symbol for gold?", label="5: Science")
gen("How many fundamental rights are there in the Indian Constitution?", label="6: Polity")
gen("Once upon a time", chat=False, temp=0.8, top_k=40, rep=1.2, n=80, label="7: Fluency check")

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO_DIR = Path("/content/gamax1_project")

# Fresh start -- original 100k pretrained checkpoint se, v2 (overfit) se chain nahi
BASE_CKPT = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt")
QA_DATA = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA/")

# Replay source: books_cleaned_v1 (plain prose) -- "Once upon a time" jaisi
# fluency ko protect karne ke liye sabse relevant, poore 1.17B corpus se halka
REPLAY_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/books_cleaned_v1")
REPLAY_CACHE = Path("/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache")

FT_OUT_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3")
FT_OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"Base checkpoint": BASE_CKPT, "QA data": QA_DATA, "Replay dir": REPLAY_DIR}.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found:\n{path}")
    print(f"✓ {name}: {path}")

cmd = [
    sys.executable, "-u", "-m", "gamax1.finetune",
    "--init_from", str(BASE_CKPT),
    "--data", str(QA_DATA),
    "--out_dir", str(FT_OUT_DIR),
    "--max_len", "128",
    "--batch_size", "8",
    "--epochs", "6",
    "--lr", "3e-5",
    "--warmup_steps", "50",
    "--val_fraction", "0.05",
    "--eval_interval", "50",
    "--checkpoint_interval", "100",
    "--freeze_sparsity",
    "--replay_data_dir", str(REPLAY_DIR),
    "--replay_ratio", "0.3",
    "--replay_cache_dir", str(REPLAY_CACHE),
    "--seed", "0",
    "--device", "cuda",
]

print("\n" + "=" * 100)
print("FINE-TUNE v3: fresh base + replay (books) to protect fluency, 6 epochs")
print("=" * 100)
print(" ".join(cmd))
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    cmd, cwd=str(REPO_DIR), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in iter(process.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)
finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()
print("\n" + "=" * 100)
if return_code == 0:
    print("🎉 FINE-TUNE v3 FINISHED")
else:
    print(f"❌ Fine-tune v3 exited with code {return_code}")
print("=" * 100)

In [ ]:
finetune_path = REPO_DIR / "gamax1" / "finetune.py"
content = finetune_path.read_text()

old_block = '''        from .bulk_corpus import build_or_load_bulk_tokens
        from .train import get_batch
        replay_data, replay_starts, _sources = build_or_load_bulk_tokens(
            args.replay_data_dir, args.replay_cache_dir, tok, rebuild=False,
        )
        print(f"Replay corpus: {len(replay_data):,} tokens from {args.replay_data_dir} "
              f"(mixed in at ratio {args.replay_ratio:g})")'''

new_block = '''        from .bulk_corpus import build_or_load_bulk_tokens
        from .train import get_batch
        _replay_tok, _replay_store, _sources = build_or_load_bulk_tokens(
            args.replay_data_dir, args.replay_cache_dir,
            tokenizer=tok, rebuild=False,
        )
        replay_data = _replay_store.tensor
        replay_starts = None   # None -> get_batch samples fully random windows
        print(f"Replay corpus: {len(replay_data):,} tokens from {args.replay_data_dir} "
              f"(mixed in at ratio {args.replay_ratio:g})")'''

assert old_block in content, "Buggy block not found -- file may already be patched or differ from expected."
content = content.replace(old_block, new_block)
finetune_path.write_text(content)
print(f"✓ Patched: {finetune_path}")

# Verify
import subprocess, sys
r = subprocess.run([sys.executable, "-c", "import ast; ast.parse(open(r'%s').read())" % finetune_path],
                    capture_output=True, text=True)
print("Syntax check:", "OK" if r.returncode == 0 else r.stderr)

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO_DIR = Path("/content/gamax1_project")

BASE_CKPT = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt")
QA_DATA = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA/")
REPLAY_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/books_cleaned_v1")
REPLAY_CACHE = Path("/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache")

FT_OUT_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3")
FT_OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"Base checkpoint": BASE_CKPT, "QA data": QA_DATA, "Replay dir": REPLAY_DIR}.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found:\n{path}")
    print(f"✓ {name}: {path}")

cmd = [
    sys.executable, "-u", "-m", "gamax1.finetune",
    "--init_from", str(BASE_CKPT),
    "--data", str(QA_DATA),
    "--out_dir", str(FT_OUT_DIR),
    "--max_len", "128",
    "--batch_size", "8",
    "--epochs", "6",
    "--lr", "3e-5",
    "--warmup_steps", "50",
    "--val_fraction", "0.05",
    "--eval_interval", "50",
    "--checkpoint_interval", "100",
    "--freeze_sparsity",
    "--replay_data_dir", str(REPLAY_DIR),
    "--replay_ratio", "0.3",
    "--replay_cache_dir", str(REPLAY_CACHE),
    "--seed", "0",
    "--device", "cuda",
]

print("\n" + "=" * 100)
print("FINE-TUNE v3: fresh base + replay (books) to protect fluency, 6 epochs")
print("=" * 100)
print(" ".join(cmd))
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    cmd, cwd=str(REPO_DIR), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in iter(process.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)
finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()
print("\n" + "=" * 100)
if return_code == 0:
    print("🎉 FINE-TUNE v3 FINISHED")
else:
    print(f"❌ Fine-tune v3 exited with code {return_code}")
print("=" * 100)

In [ ]:
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/gamax1_project")

# ------------------------------------------------------------
# PATCH 1: finetune.py -- replay call args + return-value unpacking
# ------------------------------------------------------------
ft_path = REPO_DIR / "gamax1" / "finetune.py"
ft_content = ft_path.read_text()

ft_old = '''        replay_data, replay_starts, _sources = build_or_load_bulk_tokens(
            args.replay_data_dir, args.replay_cache_dir, tok, rebuild=False,
        )'''
ft_new = '''        _replay_tok, _replay_store, _sources = build_or_load_bulk_tokens(
            args.replay_data_dir, args.replay_cache_dir,
            tokenizer=tok, rebuild=False,
        )
        replay_data = _replay_store.tensor
        replay_starts = None   # None -> get_batch samples fully random windows'''

if ft_old in ft_content:
    ft_content = ft_content.replace(ft_old, ft_new)
    ft_path.write_text(ft_content)
    print("✓ finetune.py: patched")
elif ft_new in ft_content:
    print("✓ finetune.py: already patched, skipping")
else:
    print("⚠ finetune.py: expected block not found -- check manually")

# ------------------------------------------------------------
# PATCH 2: bulk_corpus.py -- path_to_source[key] wrong lookup
# ------------------------------------------------------------
bc_path = REPO_DIR / "gamax1" / "bulk_corpus.py"
bc_content = bc_path.read_text()

bc_old = '''                key = _stable_file_key(data_dir, path_to_source[str(path)], path)
                stat = _file_stat(path)
                files_record[key] = {
                    "source": path_to_source[key],
                    "size": stat["size"],
                    "token_start": token_count,
                    "token_count": len(encoded),
                }'''
bc_new = '''                key = _stable_file_key(data_dir, path_to_source[str(path)], path)
                stat = _file_stat(path)
                files_record[key] = {
                    "source": source_name,
                    "size": stat["size"],
                    "token_start": token_count,
                    "token_count": len(encoded),
                }'''

if bc_old in bc_content:
    bc_content = bc_content.replace(bc_old, bc_new)
    bc_path.write_text(bc_content)
    print("✓ bulk_corpus.py: patched")
elif bc_new in bc_content:
    print("✓ bulk_corpus.py: already patched, skipping")
else:
    print("⚠ bulk_corpus.py: expected block not found -- check manually")

# ------------------------------------------------------------
# Syntax-check both before committing
# ------------------------------------------------------------
import sys
for p in [ft_path, bc_path]:
    r = subprocess.run([sys.executable, "-c", f"import ast; ast.parse(open(r'{p}').read())"],
                        capture_output=True, text=True)
    print(f"Syntax check {p.name}:", "OK" if r.returncode == 0 else r.stderr)

# ------------------------------------------------------------
# COMMIT + PUSH to GitHub -- this makes the fix permanent
# ------------------------------------------------------------
print("\n--- git status ---")
print(subprocess.run(["git", "status", "--short"], cwd=str(REPO_DIR), capture_output=True, text=True).stdout)

r = subprocess.run(["git", "add", "gamax1/finetune.py", "gamax1/bulk_corpus.py"],
                    cwd=str(REPO_DIR), capture_output=True, text=True)
print(r.stdout, r.stderr)

r = subprocess.run(
    ["git", "commit", "-m",
     "Fix finetune.py replay call-args/unpacking + bulk_corpus.py path_to_source KeyError"],
    cwd=str(REPO_DIR), capture_output=True, text=True,
)
print(r.stdout, r.stderr)

r = subprocess.run(["git", "push"], cwd=str(REPO_DIR), capture_output=True, text=True)
print("--- git push ---")
print(r.stdout)
print(r.stderr)

In [ ]:
import shutil
from pathlib import Path

REPO_DIR = Path("/content/gamax1_project")

# Apne upload kiye hue path yahan daal do
shutil.copy("/content/drive/MyDrive/<jahan upload kiya>/finetune.py", REPO_DIR / "gamax1" / "finetune.py")
shutil.copy("/content/drive/MyDrive/<jahan upload kiya>/bulk_corpus.py", REPO_DIR / "gamax1" / "bulk_corpus.py")

import subprocess
subprocess.run(["git", "add", "gamax1/finetune.py", "gamax1/bulk_corpus.py"], cwd=str(REPO_DIR))
subprocess.run(["git", "commit", "-m", "Fix finetune replay bug + bulk_corpus KeyError"], cwd=str(REPO_DIR))
r = subprocess.run(["git", "push"], cwd=str(REPO_DIR), capture_output=True, text=True)
print(r.stdout, r.stderr)

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO_DIR = Path("/content/gamax1_project")

BASE_CKPT = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt")
QA_DATA = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA/")
REPLAY_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/books_cleaned_v1")
REPLAY_CACHE = Path("/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache")

FT_OUT_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3")
FT_OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"Base checkpoint": BASE_CKPT, "QA data": QA_DATA, "Replay dir": REPLAY_DIR}.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found:\n{path}")
    print(f"✓ {name}: {path}")

cmd = [
    sys.executable, "-u", "-m", "gamax1.finetune",
    "--init_from", str(BASE_CKPT),
    "--data", str(QA_DATA),
    "--out_dir", str(FT_OUT_DIR),
    "--max_len", "128",
    "--batch_size", "8",
    "--epochs", "6",
    "--lr", "3e-5",
    "--warmup_steps", "50",
    "--val_fraction", "0.05",
    "--eval_interval", "50",
    "--checkpoint_interval", "100",
    "--freeze_sparsity",
    "--replay_data_dir", str(REPLAY_DIR),
    "--replay_ratio", "0.3",
    "--replay_cache_dir", str(REPLAY_CACHE),
    "--seed", "0",
    "--device", "cuda",
]

print("\n" + "=" * 100)
print("FINE-TUNE v3: fresh base + replay (books) to protect fluency, 6 epochs")
print("=" * 100)
print(" ".join(cmd))
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    cmd, cwd=str(REPO_DIR), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in iter(process.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)
finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()
print("\n" + "=" * 100)
if return_code == 0:
    print("🎉 FINE-TUNE v3 FINISHED")
else:
    print(f"❌ Fine-tune v3 exited with code {return_code}")
print("=" * 100)

In [ ]:
import subprocess, sys, torch
from pathlib import Path

FT_CKPT = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/gamax1_finetune_best.pt"
)
dev = "cuda" if torch.cuda.is_available() else "cpu"

def gen(prompt, chat=True, temp=0.6, top_k=30, rep=1.15, n=100, label=""):
    cmd = [sys.executable, "-u", "-m", "gamax1.generate",
           "--ckpt", str(FT_CKPT), "--prompt", prompt,
           "--max_new_tokens", str(n),
           "--temperature", str(temp), "--top_k", str(top_k),
           "--repetition_penalty", str(rep), "--device", dev]
    if chat:
        cmd += ["--chat", "--stop_at_eos"]
    r = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    print(f"\n{'='*70}\n[{label}] PROMPT: {prompt!r}\n{'='*70}")
    print(r.stdout)
    if r.returncode != 0:
        print("--- stderr ---\n", r.stderr[-1500:])

gen("What is the capital of India?", label="1: India capital")
gen("Who wrote Hamlet?", label="2: Hamlet author")
gen("What is the capital of Rajasthan?", label="3: Geography")
gen("Who is the President of India?", label="4: Polity")
gen("What is the chemical symbol for gold?", label="5: Science")
gen("How many fundamental rights are there in the Indian Constitution?", label="6: Polity")
gen("Once upon a time", chat=False, temp=0.8, top_k=40, rep=1.2, n=80, label="7: Fluency check")

In [ ]:
import json
from pathlib import Path

QA_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA")
for f in sorted(QA_DIR.glob("*.jsonl")):
    print(f"\n{'='*60}\n{f.name}\n{'='*60}")
    lines = f.read_text(encoding="utf-8").splitlines()
    for line in lines[:3]:
        try:
            rec = json.loads(line)
            print(json.dumps(rec, indent=2, ensure_ascii=False)[:500])
            print("---")
        except Exception as e:
            print(f"[parse error: {e}] raw: {line[:200]}")

In [ ]:
import subprocess, sys, torch
from pathlib import Path

FT_CKPT = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/gamax1_finetune_best.pt"
)
dev = "cuda" if torch.cuda.is_available() else "cpu"

def gen(prompt, expected, label=""):
    cmd = [sys.executable, "-u", "-m", "gamax1.generate",
           "--ckpt", str(FT_CKPT), "--prompt", prompt,
           "--max_new_tokens", "60",
           "--temperature", "0.6", "--top_k", "30", "--repetition_penalty", "1.15",
           "--chat", "--stop_at_eos", "--device", dev]
    r = subprocess.run(cmd, cwd=str(REPO_DIR), capture_output=True, text=True)
    print(f"\n{'='*70}\n[{label}] Q: {prompt!r}\nEXPECTED: {expected!r}\n{'='*70}")
    print(r.stdout)
    if r.returncode != 0:
        print("--- stderr ---\n", r.stderr[-1000:])

# Exact questions from the actual training data (in-distribution test)
gen("In which Article of the Constitution of India is there a provision for the abolition of untouchability?",
    "In Article 17", label="1: Polity (seen in training)")
gen("Which river forms the Sundarbans Delta?",
    "Ganga, Brahmaputra, and Meghna", label="2: Geography (seen in training)")
gen("Carbon dating method is used to determine the age of?",
    "Fossils", label="3: Science (seen in training)")
gen("Which organ acts as the blood bank in the body?",
    "Spleen", label="4: Science (seen in training)")
gen("In which state is the largest network of rivers found in India?",
    "In Uttar Pradesh", label="5: Geography (seen in training)")

In [ ]:

from pathlib import Path

PROJECT = Path("/content/gamax1_project")

BEST_CKPT = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/"
    "runs/gamax1/finetune-QNA-v3/"
    "gamax1_finetune_best.pt"
)

QUESTIONS = [
    {
        "question": (
            "In which Article of the Constitution of India "
            "is there a provision for the abolition of untouchability?"
        ),
        "expected": "In Article 17",
    },
    {
        "question": "Which river forms the Sundarbans Delta?",
        "expected": "Ganga, Brahmaputra, and Meghna",
    },
    {
        "question": "Carbon dating method is used to determine the age of?",
        "expected": "Fossils",
    },
    {
        "question": "Which organ acts as the blood bank in the body?",
        "expected": "Spleen",
    },
    {
        "question": (
            "In which state is the largest network of rivers "
            "found in India?"
        ),
        "expected": "In Uttar Pradesh",
    },
]

if not BEST_CKPT.is_file():
    raise FileNotFoundError(
        f"Best checkpoint not found: {BEST_CKPT}"
    )

print("Checkpoint exists:", BEST_CKPT)
print("Size (bytes):", BEST_CKPT.stat().st_size)
print("Questions to evaluate:", len(QUESTIONS))

for i, item in enumerate(QUESTIONS, start=1):
    print(f"\n{i}. {item['question']}")
    print("Expected:", item["expected"])


In [ ]:
%cd /content/gamax1_project
!ls

In [ ]:
!python -m gamax1.generate \
  --ckpt "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/gamax1_finetune_best.pt" \
  --prompt "In which Article of the Constitution of India is there a provision for the abolition of untouchability?" \
  --max_new_tokens 20 \
  --temperature 0.1 \
  --top_k 1 \
  --device cuda

In [ ]:
!python3 -m gamax1.generate \
  --ckpt "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/gamax1_finetune_best.pt" \
  --prompt "Question: In which Article of the Constitution of India is there a provision for the abolition of untouchability? Answer:" \
  --max_new_tokens 20 \
  --temperature 0.1 \
  --top_k 1 \
  --device cuda

some test

In [ ]:
import argparse
import os
import subprocess
import sys
from pathlib import Path

import torch


def inspect_checkpoint(path):
    path = Path(path)

    print("\n" + "=" * 65)
    print(f"CHECKPOINT: {path}")
    print("=" * 65)

    if not path.is_file():
        print(f"[ERROR] Checkpoint not found: {path}")
        return False

    try:
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

        if not isinstance(ckpt, dict):
            print("[ERROR] Checkpoint is not a dictionary.")
            return False

        print("Available keys:", list(ckpt.keys()))

        config = ckpt.get("config")
        if isinstance(config, dict):
            print("\nModel configuration:")
            for key in (
                "d_model",
                "n_heads",
                "n_layers",
                "n_features",
                "block_size",
                "tokenizer",
            ):
                print(f"  {key}: {config.get(key, 'NOT FOUND')}")

        state = ckpt.get("model_state")
        if isinstance(state, dict):
            print("\nModel state:")
            print("  Tensor count:", len(state))
            print(
                "  Total parameters:",
                sum(
                    value.numel()
                    for value in state.values()
                    if isinstance(value, torch.Tensor)
                ),
            )
        else:
            print("[WARNING] No model_state dictionary found.")

        if "merges" not in ckpt and (
            not isinstance(config, dict)
            or config.get("tokenizer") == "bpe"
        ):
            print("[WARNING] BPE merges are missing or tokenizer is unclear.")

        return True

    except Exception as exc:
        print(f"[ERROR] Could not read checkpoint: {exc}")
        return False


def generate_answer(repo, checkpoint, question, label, max_tokens):
    repo = Path(repo).resolve()
    checkpoint = Path(checkpoint).resolve()

    print("\n" + "=" * 65)
    print(f"GENERATION TEST: {label}")
    print("=" * 65)

    if not checkpoint.is_file():
        print(f"[ERROR] Checkpoint not found: {checkpoint}")
        return

    env = os.environ.copy()
    env["PYTHONPATH"] = (
        str(repo)
        + os.pathsep
        + env.get("PYTHONPATH", "")
    )

    command = [
        sys.executable,
        "-m",
        "gamax1.generate",
        "--ckpt", str(checkpoint),
        "--prompt", f"Question: {question}\nAnswer:",
        "--max_new_tokens", str(max_tokens),
        "--temperature", "1.0",
        "--top_k", "1",
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
    ]

    print("Command:", " ".join(command))
    print("\nModel output:\n")

    try:
        result = subprocess.run(
            command,
            cwd=repo,
            env=env,
            text=True,
            capture_output=True,
            timeout=300,
        )

        if result.stdout:
            print(result.stdout)

        if result.stderr:
            print("\nDiagnostics / errors:")
            print(result.stderr)

        print("Exit code:", result.returncode)

    except subprocess.TimeoutExpired:
        print("[ERROR] Generation timed out after 300 seconds.")
    except Exception as exc:
        print(f"[ERROR] Could not run generation: {exc}")


def main(override_args=None):
    parser = argparse.ArgumentParser(
        description="Diagnose GamaX1 base vs fine-tuned checkpoints."
    )

    parser.add_argument(
        "--repo",
        default="/content/gamax1_project",
    )
    parser.add_argument(
        "--base",
        required=True,
        help="Path to the original pretraining checkpoint",
    )
    parser.add_argument(
        "--finetuned",
        required=True,
        help="Path to the fine-tuned checkpoint",
    )
    parser.add_argument(
        "--question",
        default="In which Article of the Constitution of India "
                "is untouchability abolished?",
    )
    parser.add_argument(
        "--max_tokens",
        type=int,
        default=60,
    )

    # Allow overriding args within Notebook executions
    args = parser.parse_args(args=override_args)

    repo = Path(args.repo).resolve()

    if not (repo / "gamax1" / "generate.py").is_file():
        raise SystemExit(
            f"Could not find gamax1/generate.py inside {repo}"
        )

    print("GamaX1 Diagnostic Test")
    print("Repository:", repo)
    print("Question:", args.question)
    print("Device:", "CUDA" if torch.cuda.is_available() else "CPU")

    base_ok = inspect_checkpoint(args.base)
    ft_ok = inspect_checkpoint(args.finetuned)

    if not base_ok or not ft_ok:
        print(
            "\n[STOP] At least one checkpoint could not be inspected. "
            "Fix the reported checkpoint error before comparing outputs."
        )
        return

    generate_answer(
        repo,
        args.base,
        args.question,
        "BASE MODEL",
        args.max_tokens,
    )

    generate_answer(
        repo,
        args.finetuned,
        args.question,
        "FINE-TUNED MODEL",
        args.max_tokens,
    )

    print("\n" + "=" * 65)
    print("TEST COMPLETE")
    print("Compare both outputs and save any errors shown above.")
    print("=" * 65)


if __name__ == "__main__":
    # Example usage inside Colab:
    # main([
    #     "--base", "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt",
    #     "--finetuned", "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/gamax1_finetune_best.pt"
    # ])
    # Note: Replace paths below as needed.
    main([
        "--base", "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt",
        "--finetuned", "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/gamax1_finetune_best.pt"
    ])

In [ ]:

# ============================================================
# TEST 12A — CHECKPOINT & Q/A DATA AUDIT
# Read-only diagnostic. No training. No checkpoint changes.
# ============================================================

import os
import json
import torch

BASE_CKPT = (
    "/content/drive/MyDrive/Aetherion_GamaX1/"
    "runs/gamax1/full/gamax1.pt"
)

FT_CKPT = (
    "/content/drive/MyDrive/Aetherion_GamaX1/"
    "runs/gamax1/finetune-QNA-v3/"
    "gamax1_finetune_best.pt"
)

QNA_PATH = (
    "/content/drive/MyDrive/shareDev/"
    "Aetherion_GamaX1/data/QnA/"
    "indian_polity_qa_cleaned.jsonl"
)

def inspect_checkpoint(path):
    print("\n" + "=" * 65)
    print("CHECKPOINT:", path)
    print("=" * 65)

    if not os.path.isfile(path):
        print("ERROR: Checkpoint not found")
        return

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    print("File size (GB):",
          round(os.path.getsize(path) / 1e9, 3))
    print("Training step:", ckpt.get("step"))
    print("Config:", json.dumps(
        ckpt.get("config", {}),
        indent=2,
        default=str,
    ))

    state = ckpt.get("model_state", {})
    print("Model tensors:", len(state))
    print("Tensor elements:",
          sum(v.numel() for v in state.values()
              if torch.is_tensor(v)))

    # Use 'or []' to prevent NoneType len() error if keys exist but are None
    print("Tokenizer vocab entries:",
          len(ckpt.get("vocab") or []))
    print("Tokenizer merges:",
          len(ckpt.get("merges") or []))

    del ckpt


print("CUDA available:", torch.cuda.is_available())

inspect_checkpoint(BASE_CKPT)
inspect_checkpoint(FT_CKPT)

print("\n" + "=" * 65)
print("Q/A DATASET AUDIT")
print("=" * 65)

if not os.path.isfile(QNA_PATH):
    print("ERROR: Q/A dataset not found:", QNA_PATH)
else:
    total = 0
    malformed = 0
    empty = 0
    examples = []

    with open(QNA_PATH, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            try:
                obj = json.loads(line)
                total += 1

                # Inspect common field names without assuming
                # which schema the dataset uses.
                q = next(
                    (obj[k] for k in
                     ("question", "prompt", "instruction", "input")
                     if isinstance(obj.get(k), str)),
                    None,
                )
                a = next(
                    (obj[k] for k in
                     ("answer", "output", "response", "completion")
                     if isinstance(obj.get(k), str)),
                    None,
                )

                if q is None or a is None:
                    malformed += 1
                    if len(examples) < 5:
                        examples.append(
                            ("SCHEMA", line_no, obj)
                        )
                    continue

                if not q.strip() or not a.strip():
                    empty += 1
                    if len(examples) < 5:
                        examples.append(
                            ("EMPTY", line_no, obj)
                        )

            except Exception as e:
                malformed += 1
                if len(examples) < 5:
                    examples.append(
                        ("JSON_ERROR", line_no, str(e))
                    )

    print("Records parsed:", total)
    print("Malformed/schema-mismatch:", malformed)
    print("Empty question/answer:", empty)

    print("\nFirst records requiring inspection:")
    for item in examples:
        print(repr(item)[:1200])

print("\nTEST 12A COMPLETE")


In [ ]:

import os

ROOT = "/content/drive/MyDrive/Aetherion_GamaX1/data/QnA"

print("Folder exists:", os.path.isdir(ROOT))

if os.path.isdir(ROOT):
    print("\nFiles found:")
    for name in sorted(os.listdir(ROOT)):
        path = os.path.join(ROOT, name)
        if os.path.isfile(path):
            print(f"{name}  ({os.path.getsize(path):,} bytes)")
else:
    print("Folder not found. Check whether Google Drive is mounted.")


In [ ]:

import os
import json

ROOT = "/content/drive/MyDrive/Aetherion_GamaX1/data/QnA"

files = [
    "indian_polity_qa_cleaned.jsonl",
    "indian_world_geography_english_qa_cleaned.jsonl",
    "science_qa_cleaned.jsonl",
]

for filename in files:
    path = os.path.join(ROOT, filename)
    print("\n" + "=" * 60)
    print("FILE:", filename)

    count = 0
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            obj = json.loads(line)
            count += 1

            # Print only the first two records per dataset.
            if count <= 2:
                print("Record", count)
                print("Keys:", list(obj.keys()))
                print("Content:", json.dumps(
                    obj, ensure_ascii=False
                )[:1200])

    print("Total non-empty records:", count)

print("\nTEST 12B COMPLETE")


In [ ]:

# TEST 12C — Verify the project's actual Q&A loader
# Read-only. No training or checkpoint modifications.

import sys
import os

PROJECT = "/content/gamax1_project"
QNA_PATH = (
    "/content/drive/MyDrive/Aetherion_GamaX1/"
    "data/QnA/indian_polity_qa_cleaned.jsonl"
)

if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)

from gamax1.instruction_data import load_pairs

pairs = load_pairs(QNA_PATH)

print("Loader returned:", type(pairs).__name__)
print("Number of pairs:", len(pairs))

print("\nFirst three pairs:")
for i, pair in enumerate(pairs[:3], 1):
    print(f"\nExample {i}:")
    print(repr(pair)[:1500])


In [ ]:

from pathlib import Path

path = Path("/content/gamax1_project/gamax1/generate.py")

print("File exists:", path.exists())

if path.exists():
    print(path.read_text(encoding="utf-8"))


In [ ]:
!cd /content/gamax1_project

!/usr/bin/python3 -m gamax1.generate \
  --ckpt /content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/gamax1_finetune_best.pt \
  --prompt "In which Article of the Constitution of India is there a provision for the abolition of untouchability?" \
  --chat \
  --max_new_tokens 32 \
  --temperature 1.0 \
  --top_k 1 \
  --device cuda

In [ ]:

!cd /content/gamax1_project

!/usr/bin/python3 -m gamax1.generate \
  --ckpt /content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt \
  --prompt "In which Article of the Constitution of India is there a provision for the abolition of untouchability?" \
  --chat \
  --max_new_tokens 32 \
  --temperature 1.0 \
  --top_k 1 \
  --device cuda


In [ ]:

import sys
sys.path.insert(0, "/content/gamax1_project")

import torch
from gamax1.generate import load_model
from gamax1.instruction_data import encode_example

CKPT = (
    "/content/drive/MyDrive/Aetherion_GamaX1/"
    "runs/gamax1/finetune-QNA-v3/"
    "gamax1_finetune_best.pt"
)

Q = (
    "In which Article of the Constitution of India is there "
    "a provision for the abolition of untouchability?"
)
A = "In Article 17"

device = "cuda" if torch.cuda.is_available() else "cpu"
model, tok = load_model(CKPT, device)

ids, mask = encode_example(tok, Q, A, max_len=512)

print("Tokenizer:", type(tok).__name__)
print("Vocab size:", tok.vocab_size)
print("USER id:", getattr(tok, "user_id", None))
print("ASSISTANT id:", getattr(tok, "assistant_id", None))
print("EOS id:", getattr(tok, "eos_id", None))
print("Encoded length:", len(ids))
print("Masked answer tokens:", sum(mask))

print("\nEncoded training example:")
print(tok.decode_with_boundaries(ids))

print("\nToken-by-token trace:")
for i, token_id in enumerate(ids):
    print(
        i,
        token_id,
        repr(tok.decode([token_id])),
        "ANSWER" if mask[i] else "CONTEXT",
    )


In [ ]:

import sys
sys.path.insert(0, "/content/gamax1_project")

import torch
from gamax1.generate import load_model

BASE = (
    "/content/drive/MyDrive/Aetherion_GamaX1/"
    "runs/gamax1/full/gamax1.pt"
)
FT = (
    "/content/drive/MyDrive/Aetherion_GamaX1/"
    "runs/gamax1/finetune-QNA-v3/"
    "gamax1_finetune_best.pt"
)

Q = (
    "In which Article of the Constitution of India is there "
    "a provision for the abolition of untouchability?"
)
A = "In Article 17"

device = "cuda" if torch.cuda.is_available() else "cpu"

def audit(path, label):
    model, tok = load_model(path, device)

    # Match the training encoder's exact token sequence.
    prompt = [tok.user_id] + tok.encode(Q) + [tok.assistant_id]
    answer = tok.encode(A) + [tok.eos_id]
    ids = prompt + answer

    x = torch.tensor([ids[:-1]], dtype=torch.long, device=device)

    with torch.inference_mode():
        logits, _ = model(
            x,
            targets=None,
            k=model.sparsity_ctrl.k,
            use_ptm=False,
        )

    print("\n" + "=" * 60)
    print(label)
    print("=" * 60)

    # At each position, predict the next correct answer token.
    start = len(prompt) - 1

    for j, target_id in enumerate(answer):
        position = start + j
        scores = logits[0, position].float()
        probs = torch.softmax(scores, dim=-1)
        pred_id = int(torch.argmax(scores).item())

        target_rank = int(
            (scores > scores[target_id]).sum().item()
        ) + 1

        print(
            f"target={tok.decode([target_id])!r} | "
            f"top1={tok.decode([pred_id])!r} | "
            f"target_prob={probs[target_id].item():.6g} | "
            f"target_rank={target_rank}"
        )

    del model

audit(BASE, "BASE MODEL")
audit(FT, "FINE-TUNED MODEL")


In [ ]:

from pathlib import Path

root = Path("/content/gamax1_project/gamax1")

for filename in ["train.py", "instruction_data.py"]:
    path = root / filename
    print("\n" + "=" * 70)
    print(filename, "exists:", path.exists())
    print("=" * 70)

    if path.exists():
        lines = path.read_text(encoding="utf-8").splitlines()

        keywords = (
            "encode_example",
            "cross_entropy",
            "ignore_index",
            "answer_mask",
            "mask",
            "optimizer.step",
            "gamax1_finetune_best",
            "best_val",
            "val_loss",
            "shift",
            "logits",
            "targets",
        )

        matches = [
            i for i, line in enumerate(lines)
            if any(k in line for k in keywords)
        ]

        shown = set()
        for i in matches:
            lo, hi = max(0, i - 5), min(len(lines), i + 8)
            for j in range(lo, hi):
                if j not in shown:
                    print(f"{j+1:4}: {lines[j]}")
                    shown.add(j)
            print("----")


In [ ]:

from pathlib import Path

ROOT = Path("/content/gamax1_project")

print("Possible fine-tuning files:")
for p in sorted(ROOT.rglob("*.py")):
    if any(part in {".git", "__pycache__"} for part in p.parts):
        continue
    name = p.name.lower()
    if any(word in name for word in ("finetune", "fine_tune", "instruction", "sft")):
        print(p)

print("\n" + "=" * 65)
print("Fine-tuning loss and checkpoint references")
print("=" * 65)

terms = (
    "loss_mask",
    "cross_entropy",
    "ignore_index",
    "masked_fill",
    "finetune_best",
    "best_val",
    "val_loss",
    "optimizer.step",
    "scaler.scale",
)

for p in sorted(ROOT.rglob("*.py")):
    if any(part in {".git", "__pycache__"} for part in p.parts):
        continue

    try:
        lines = p.read_text(encoding="utf-8").splitlines()
    except Exception:
        continue

    hits = [
        i for i, line in enumerate(lines)
        if any(term in line.lower() for term in terms)
    ]

    if not hits:
        continue

    print(f"\n--- {p} ---")
    shown = set()

    for i in hits:
        for j in range(max(0, i - 4), min(len(lines), i + 6)):
            if j not in shown:
                print(f"{j+1:4}: {lines[j]}")
                shown.add(j)
        print("...")


In [ ]:
%cd /content/gamax1_project

!echo "===== LOSS + EVALUATION ====="
!sed -n '55,110p' gamax1/finetune.py

!echo "===== MODEL / OPTIMIZER SETUP ====="
!sed -n '107,222p' gamax1/finetune.py

!echo "===== TRAINING + CHECKPOINT SAVING ====="
!sed -n '223,283p' gamax1/finetune.py

!echo "===== MODEL use_ptm IMPLEMENTATION ====="
!grep -n -C 5 "use_ptm" gamax1/model.py

In [ ]:
import torch
import torch.nn.functional as F

@torch.no_grad()
def evaluate(model, val_loader, device, k, use_amp):
    """Compute exact mean loss over all unmasked answer/EOS tokens."""
    model.eval()

    total_loss = 0.0
    total_tokens = 0

    for xb, yb, mask in val_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        mask = mask.to(device).bool()

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp,
        ):
            logits, _ = model(
                xb,
                targets=None,
                k=k,
                use_ptm=False,
            )

            flat_logits = logits.reshape(-1, logits.size(-1))
            flat_targets = yb.reshape(-1)
            flat_mask = mask.reshape(-1)

            # Sum losses only at answer/EOS positions.
            token_losses = F.cross_entropy(
                flat_logits,
                flat_targets,
                reduction="none",
            )

            batch_loss_sum = token_losses[flat_mask].sum()

        count = int(flat_mask.sum().item())
        total_loss += batch_loss_sum.item()
        total_tokens += count

    if total_tokens == 0:
        raise ValueError("Validation set contains no unmasked answer tokens.")

    return total_loss / total_tokens

In [ ]:
!cd /content/gamax1_project

!grep -n -A 65 -B 5 "def save_checkpoint" gamax1/finetune.py

!grep -n -A 35 -B 5 "class ProbationaryMemoryTracker" gamax1/model.py

!grep -n -A 45 -B 5 "def update" gamax1/model.py

In [ ]:

!cd /content/gamax1_project && grep -nE "save_checkpoint|ProbationaryMemoryTracker|def update|def save" gamax1/finetune.py gamax1/model.py


In [ ]:

import subprocess

result = subprocess.run(
    [
        "grep", "-nE",
        "save_checkpoint|ProbationaryMemoryTracker|def update|def save",
        "/content/gamax1_project/gamax1/finetune.py",
        "/content/gamax1_project/gamax1/model.py",
    ],
    capture_output=True,
    text=True,
)

print(result.stdout or result.stderr or "Koi matching line nahi mili.")


In [ ]:
%cd /content/gamax1_project

!grep -n -A 55 -B 5 "def save_checkpoint" gamax1/train.py

!grep -n -A 60 -B 5 "class ProbationaryMemoryTracker" gamax1/layers.py

In [ ]:
%cd /content/gamax1_project

!sed -n '225,290p' gamax1/train.py

!sed -n '145,190p' gamax1/layers.py

In [ ]:
import torch
from pathlib import Path

paths = [
    Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt"),
    Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/gamax1_finetune_best.pt"),
]

for path in paths:
    print("\n" + "=" * 70)
    print("Checkpoint:", path)
    print("Exists:", path.exists())

    if not path.exists():
        continue

    ckpt = torch.load(path, map_location="cpu", weights_only=False)

    print("Step:", ckpt.get("step"))
    print("Top-level keys:", sorted(ckpt.keys()))
    print("Best validation loss:", ckpt.get("best_val_loss"))
    print("Config:", ckpt.get("config"))
    print("Model tensors:", len(ckpt.get("model_state", {})))
    print("PTM state count:", len(ckpt.get("ptm_states") or []))

In [ ]:
%cd /content/gamax1_project

!grep -nE "load_base_checkpoint|save_checkpoint|best_val_loss|best_loss|val_loss|config|resume|optimizer.load_state_dict|load_state_dict|args.lr|run_name|finetune" gamax1/finetune.py

In [ ]:
!grep -nE "def load|resume_from|optimizer.load_state_dict|scaler.load_state_dict|set_rng_state|set_rng_state_all|ptm_states|sparsity_controller_state" gamax1/train.py

In [ ]:
%cd /content/gamax1_project

!sed -n '145,285p' gamax1/finetune.py
!sed -n '245,280p' gamax1/train.py

In [ ]:
%cd /content/gamax1_project
!sed -n '1,145p' gamax1/finetune.py

In [ ]:

@torch.no_grad()
def evaluate(model, val_loader, device, k, use_amp):
    """Compute mean cross-entropy over all unmasked answer tokens."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for xb, yb, mask in val_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        mask = mask.to(device).bool()

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp,
        ):
            logits, _ = model(
                xb, targets=None, k=k, use_ptm=False
            )

            flat_logits = logits.reshape(-1, logits.size(-1))
            flat_targets = yb.reshape(-1)
            flat_mask = mask.reshape(-1)

            # Sum individual token losses; do not average batches equally.
            token_losses = F.cross_entropy(
                flat_logits,
                flat_targets,
                reduction="none",
            )
            loss_sum = token_losses[flat_mask].sum()

        count = int(flat_mask.sum().item())
        total_loss += loss_sum.item()
        total_tokens += count

    if total_tokens == 0:
        raise ValueError(
            "Validation set contains no unmasked answer tokens."
        )

    return total_loss / total_tokens


In [ ]:
model, tok, base_cfg = load_base_checkpoint(args.init_from, device)

In [ ]:

# Preserve the pretrained architecture settings, but record the actual
# fine-tuning run separately. Do not overwrite the base checkpoint.
ft_cfg = dict(base_cfg)
ft_cfg["out_dir"] = os.path.abspath(args.out_dir)
ft_cfg["run_name"] = os.path.basename(
    os.path.abspath(args.out_dir)
)
ft_cfg["fine_tuning"] = vars(args).copy()
ft_cfg["fine_tuning"]["best_val_loss"] = None


In [ ]:
%cd /content/gamax1_project

!grep -n -A 15 -B 3 "def evaluate" gamax1/finetune.py
!grep -n -A 12 -B 5 "model, tok, base_cfg" gamax1/finetune.py

In [ ]:

%cd /content/gamax1_project

from pathlib import Path
import shutil

path = Path("gamax1/finetune.py")
source = path.read_text()

# 1. Replace evaluate() with token-weighted validation.
start = source.index("@torch.no_grad()\ndef evaluate(")
end = source.index("\n\ndef main():", start)

new_evaluate = '''@torch.no_grad()
def evaluate(model, val_loader, device, k, use_amp):
    """Mean cross-entropy across all unmasked answer tokens."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for xb, yb, mask in val_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        mask = mask.to(device).bool()

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=use_amp,
        ):
            logits, _ = model(
                xb, targets=None, k=k, use_ptm=False
            )
            flat_logits = logits.reshape(-1, logits.size(-1))
            flat_targets = yb.reshape(-1)
            flat_mask = mask.reshape(-1)

            token_losses = F.cross_entropy(
                flat_logits, flat_targets, reduction="none"
            )
            loss_sum = token_losses[flat_mask].sum()

        count = int(flat_mask.sum().item())
        total_loss += loss_sum.item()
        total_tokens += count

    if total_tokens == 0:
        raise ValueError("Validation set has no answer tokens.")

    return total_loss / total_tokens
'''

# 2. Back up the source before editing.
backup = path.with_suffix(".py.backup_before_metadata_fix")
if not backup.exists():
    shutil.copy2(path, backup)

updated = source[:start] + new_evaluate + source[end:]

# 3. Build a fine-tuning config inside main(), where args/base_cfg exist.
needle = '    model, tok, base_cfg = load_base_checkpoint(args.init_from, device)\n'
assert updated.count(needle) == 1, "Expected model-loading line not found uniquely."

insert = needle + '''
    # Keep the base architecture config but record this fine-tuning run.
    ft_cfg = dict(base_cfg)
    ft_cfg["out_dir"] = os.path.abspath(args.out_dir)
    ft_cfg["run_name"] = os.path.basename(os.path.abspath(args.out_dir))
    ft_cfg["fine_tuning"] = vars(args).copy()
    ft_cfg["fine_tuning"]["best_val_loss"] = None
'''
updated = updated.replace(needle, insert, 1)

# 4. Save ft_cfg rather than the stale base_cfg in all fine-tuning checkpoints.
updated = updated.replace(
    'model, optimizer, tok, base_cfg, step, scaler=scaler,',
    'model, optimizer, tok, ft_cfg, step, scaler=scaler,'
)

# 5. Record the best score before saving the best checkpoint.
needle_best = '''                if val_loss < best_val:
                    best_val = val_loss
                    save_checkpoint(
'''
replacement_best = '''                if val_loss < best_val:
                    best_val = val_loss
                    ft_cfg["fine_tuning"]["best_val_loss"] = best_val
                    save_checkpoint(
'''
assert updated.count(needle_best) == 1, "Best-checkpoint block did not match."
updated = updated.replace(needle_best, replacement_best, 1)

# Verify all three checkpoint save calls now use ft_cfg.
assert updated.count(
    'model, optimizer, tok, ft_cfg, step, scaler=scaler,'
) == 3, "Not all checkpoint calls were updated; file left unchanged."

# Write only after every check has passed.
path.write_text(updated)

# Syntax check without starting training.
import ast
ast.parse(updated)

print("SUCCESS: finetune.py updated and syntax-checked.")
print("Backup:", backup)
print("Validation now uses a token-weighted mean.")
print("Fine-tuning checkpoints now store the fine-tuning config.")
print("Existing checkpoint files were not modified.")


In [ ]:
%cd /content/gamax1_project
!python -m py_compile gamax1/finetune.py
!grep -n -E "ft_cfg|best_val_loss|def evaluate" gamax1/finetune.py

In [ ]:
%cd /content/gamax1_project

!python -m py_compile gamax1/finetune.py
!sed -n '175,200p' gamax1/finetune.py
!sed -n '285,318p' gamax1/finetune.py

In [ ]:
from gamax1.finetune import load_base_checkpoint
import torch

# Run the shell backups and compilation checks using correct IPython syntax
!cd /content/gamax1_project && cp gamax1/finetune.py gamax1/finetune.py.backup
!cd /content/gamax1_project && python -m py_compile gamax1/finetune.py

print("Import and compilation check: SUCCESS")

In [ ]:
!cd /content/gamax1_project && python -m gamax1.finetune --resume --out_dir "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3"

In [ ]:

%cd /content/gamax1_project
!python -m gamax1.finetune --help


In [ ]:
%cd /content/gamax1_project
!cp gamax1/finetune.py gamax1/finetune.py.backup_before_resume


In [ ]:
%cd /content/gamax1_project

In [ ]:
from pathlib import Path

p = Path("/content/gamax1_project/gamax1/finetune.py")
print("File exists:", p.exists())
print("File size:", p.stat().st_size if p.exists() else "Not found")

In [ ]:
%cd /content/gamax1_project
!python -m py_compile gamax1/finetune.py
print("Syntax check complete")

In [ ]:
!python -m gamax1.finetune --help

In [ ]:
!python -m py_compile gamax1/finetune.py
!python -m gamax1.finetune --help

In [ ]:
%cd /content/gamax1_project
!git status
!git pull
!python -m py_compile gamax1/finetune.py
!python -m gamax1.finetune --help

In [ ]:
%cd /content/gamax1_project

!git stash push -m "backup-local-finetune-before-pull" -- gamax1/finetune.py
!git pull --ff-only
!python -m py_compile gamax1/finetune.py
!python -m gamax1.finetune --help

In [ ]:
from pathlib import Path

root = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3")

for p in sorted(root.glob("*.pt")):
    print(f"{p.name} — {p.stat().st_size / (1024 * 1024):.2f} MB")

In [ ]:
import torch
from pathlib import Path

p = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/"
    "finetune-QNA-v3/gamax1_finetune_latest.pt"
)

ckpt = torch.load(p, map_location="cpu", weights_only=False)

print("Checkpoint type:", type(ckpt).__name__)

if isinstance(ckpt, dict):
    print("\nTop-level keys:")
    for k, v in ckpt.items():
        if isinstance(v, dict):
            print(f"  {k}: dict with {len(v)} keys")
        else:
            print(f"  {k}: {type(v).__name__}")

    for section in ("optimizer_state", "optimizer", "scaler_state",
                    "fine_tune_state", "ft_cfg"):
        value = ckpt.get(section)
        if isinstance(value, dict):
            print(f"\n{section} keys:", list(value.keys()))

    print("\nStep fields:")
    for k in ("step", "global_step", "epoch", "batch_idx"):
        if k in ckpt:
            print(f"  {k}:", ckpt[k])

In [ ]:
import torch
from pathlib import Path

p = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/"
    "finetune-QNA-v3/gamax1_finetune_latest.pt"
)

ckpt = torch.load(p, map_location="cpu", weights_only=False)
cfg = ckpt.get("config", {})

print("Checkpoint step:", ckpt.get("step"))
print("Saved fine-tuning configuration:")
print(cfg.get("fine_tuning", "No fine-tuning metadata found"))

print("\nResume position metadata:")
print(ckpt.get("fine_tune_state", "Not present in this older checkpoint"))

In [ ]:
%cd /content/gamax1_project

from pathlib import Path
from gamax1.instruction_data import load_pairs, split_train_val

data_dir = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA")
out_dir = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3")
latest = out_dir / "gamax1_finetune_latest.pt"

assert data_dir.exists(), f"Dataset directory missing: {data_dir}"
assert latest.exists(), f"Latest checkpoint missing: {latest}"

pairs, stats = load_pairs(str(data_dir))
train_pairs, val_pairs = split_train_val(pairs, 0.05, seed=0)
steps_per_epoch = (len(train_pairs) + 8 - 1) // 8

print("Dataset records:", stats["records"])
print("Usable Q&A pairs:", len(pairs))
print("Train examples:", len(train_pairs))
print("Validation examples:", len(val_pairs))
print("Steps per epoch (batch_size=8):", steps_per_epoch)
print("Checkpoint step: 2658")
print("Estimated completed epochs:", 2658 / steps_per_epoch)
print("Output directory:", out_dir)
print("Preflight only — no training started.")

In [ ]:
%cd /content/gamax1_project

!python -m gamax1.finetune \
  --resume_from "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/gamax1_finetune_latest.pt" \
  --data "/content/drive/MyDrive/Aetherion_GamaX1/data/QnA" \
  --out_dir "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3" \
  --epochs 9 \
  --batch_size 8 \
  --max_len 512 \
  --lr 3e-5 \
  --warmup_steps 50 \
  --weight_decay 0 \
  --val_fraction 0.05 \
  --eval_interval 200 \
  --checkpoint_interval 500 \
  --freeze_sparsity \
  --seed 0

In [ ]:

%cd /content/gamax1_project

from pathlib import Path

root = Path("/content/gamax1_project")
out = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3")

print("=== Generation files ===")
for p in sorted(root.rglob("*.py")):
    if any(word in p.name.lower() for word in ["generate", "inference", "chat", "eval"]):
        print(p.relative_to(root))

print("\n=== Checkpoints ===")
for name in ["gamax1_finetune_best.pt", "gamax1_finetune_latest.pt"]:
    p = out / name
    print(f"{name}: {'FOUND' if p.exists() else 'MISSING'}")


In [ ]:

%cd /content/gamax1_project
!python -m gamax1.generate --help


In [ ]:

%cd /content/gamax1_project

!python -m gamax1.generate \
  --ckpt "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/gamax1_finetune_best.pt" \
  --prompt "Question: What is 2 + 2? Answer:" \
  --max_new_tokens 40 \
  --temperature 0 \
  --top_k 1 \
  --device cuda


In [ ]:

!python -m gamax1.generate \
  --ckpt "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/gamax1_finetune_best.pt" \
  --prompt "Question: In which Article of the Constitution of India is untouchability abolished? Answer:" \
  --max_new_tokens 60 \
  --temperature 0 \
  --top_k 1 \
  --device cuda


In [ ]:
%cd /content/gamax1_project

!python -m gamax1.generate \
  --ckpt "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/gamax1_finetune_best.pt" \
  --prompt "In which Article of the Constitution of India is there a provision for the abolition of untouchability?" \
  --max_new_tokens 40 \
  --temperature 0.1 \
  --top_k 1 \
  --chat --stop_at_eos \
  --device cuda

In [ ]:
%cd /content/gamax1_project

questions = [
    ("Which river forms the Sundarbans Delta?", "Ganga, Brahmaputra, and Meghna"),
    ("Carbon dating method is used to determine the age of?", "Fossils"),
    ("Which organ acts as the blood bank in the body?", "Spleen"),
    ("In which state is the largest network of rivers found in India?", "In Uttar Pradesh"),
]

for q, expected in questions:
    print(f"\n{'='*70}\nQ: {q}\nEXPECTED: {expected}\n{'='*70}")
    !python -m gamax1.generate \
      --ckpt "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/gamax1_finetune_best.pt" \
      --prompt "{q}" \
      --max_new_tokens 40 \
      --temperature 0.1 \
      --top_k 1 \
      --chat --stop_at_eos \
      --device cuda

In [ ]:
import json, re
from pathlib import Path

QA_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA")
OUT_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA_augmented")
OUT_DIR.mkdir(parents=True, exist_ok=True)

REWRITE_RULES = [
    (r'^Which\b', ["What"]),
    (r'^What is\b', ["Which is", "Name"]),
    (r'^In which\b', ["Which"]),
    (r'^Who\b', ["Name the person who"]),
    (r'^How many\b', ["What is the number of"]),
    (r'^When\b', ["In which year/period"]),
]

def reword(question):
    variants = []
    for pattern, alts in REWRITE_RULES:
        if re.match(pattern, question):
            for alt in alts:
                new_q = re.sub(pattern, alt, question, count=1)
                if new_q != question:
                    variants.append(new_q)
            break
    if not variants:
        variants.append(f"Answer this: {question}")
    return variants[:2]

total_in, total_out = 0, 0
for f in sorted(QA_DIR.glob("*.jsonl")):
    out_lines = []
    for line in f.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        msgs = rec.get("messages", [])
        if len(msgs) != 2:
            continue
        q, a = msgs[0]["content"], msgs[1]["content"]
        total_in += 1
        out_lines.append(json.dumps(rec, ensure_ascii=False))
        total_out += 1
        for variant_q in reword(q):
            new_rec = {"messages": [{"role": "user", "content": variant_q},
                                     {"role": "assistant", "content": a}]}
            out_lines.append(json.dumps(new_rec, ensure_ascii=False))
            total_out += 1
    out_path = OUT_DIR / f.name
    out_path.write_text("\n".join(out_lines) + "\n", encoding="utf-8")
    print(f"{f.name}: wrote {len(out_lines)} records -> {out_path}")

print(f"\nTotal: {total_in} original -> {total_out} augmented ({total_out/max(total_in,1):.1f}x exposure)")

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO_DIR = Path("/content/gamax1_project")

BASE_CKPT = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt")
QA_DATA = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA_augmented/")   # <-- naya augmented data
REPLAY_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/books_cleaned_v1")
REPLAY_CACHE = Path("/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache")  # already built, reuse hoga

FT_OUT_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v4")
FT_OUT_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {"Base checkpoint": BASE_CKPT, "QA data": QA_DATA, "Replay dir": REPLAY_DIR}.items():
    if not path.exists():
        raise FileNotFoundError(f"{name} not found:\n{path}")
    print(f"✓ {name}: {path}")

cmd = [
    sys.executable, "-u", "-m", "gamax1.finetune",
    "--init_from", str(BASE_CKPT),
    "--data", str(QA_DATA),
    "--out_dir", str(FT_OUT_DIR),
    "--max_len", "128",
    "--batch_size", "8",
    "--epochs", "5",              # ~2.2x more data/epoch, so fewer epochs for similar total steps
    "--lr", "3e-5",
    "--warmup_steps", "50",
    "--val_fraction", "0.05",
    "--eval_interval", "50",
    "--checkpoint_interval", "150",
    "--freeze_sparsity",
    "--replay_data_dir", str(REPLAY_DIR),
    "--replay_ratio", "0.3",
    "--replay_cache_dir", str(REPLAY_CACHE),
    "--seed", "0",
    "--device", "cuda",
]

print("\n" + "=" * 100)
print("FINE-TUNE v4: augmented data (2.2x) + replay, 5 epochs")
print("=" * 100)
print(" ".join(cmd))
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

process = subprocess.Popen(
    cmd, cwd=str(REPO_DIR), env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
try:
    for line in iter(process.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)
finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()
print("\n" + "=" * 100)
if return_code == 0:
    print("🎉 FINE-TUNE v4 FINISHED")
else:
    print(f"❌ Fine-tune v4 exited with code {return_code}")
print("=" * 100)

In [ ]:
%cd /content/gamax1_project

questions = [
    ("In which Article of the Constitution of India is there a provision for the abolition of untouchability?", "In Article 17"),
    ("Which river forms the Sundarbans Delta?", "Ganga, Brahmaputra, and Meghna"),
    ("Carbon dating method is used to determine the age of?", "Fossils"),
    ("Which organ acts as the blood bank in the body?", "Spleen"),
    ("In which state is the largest network of rivers found in India?", "In Uttar Pradesh"),
]

CKPT = "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v4/gamax1_finetune_best.pt"

for q, expected in questions:
    print(f"\n{'='*70}\nQ: {q}\nEXPECTED: {expected}\n{'='*70}")
    !python -m gamax1.generate \
      --ckpt "{CKPT}" \
      --prompt "{q}" \
      --max_new_tokens 40 \
      --temperature 0.1 \
      --top_k 1 \
      --chat --stop_at_eos \
      --device cuda

In [ ]:
!python -m gamax1.generate \
  --ckpt "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v4/gamax1_finetune_best.pt" \
  --prompt "Once upon a time" \
  --max_new_tokens 80 \
  --temperature 0.8 \
  --top_k 40 \
  --repetition_penalty 1.2 \
  --device cuda

In [ ]:
import json, random, re
from pathlib import Path

QA_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA")
AUG_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA_augmented_v2")
AUG_DIR.mkdir(parents=True, exist_ok=True)

REWRITE_RULES = [
    (r'^Which\b', ["What"]),
    (r'^What is\b', ["Which is", "Name"]),
    (r'^In which\b', ["Which"]),
    (r'^Who\b', ["Name the person who"]),
    (r'^How many\b', ["What is the number of"]),
    (r'^When\b', ["In which year/period"]),
]
def reword(question):
    variants = []
    for pattern, alts in REWRITE_RULES:
        if re.match(pattern, question):
            for alt in alts:
                new_q = re.sub(pattern, alt, question, count=1)
                if new_q != question:
                    variants.append(new_q)
            break
    if not variants:
        variants.append(f"Answer this: {question}")
    return variants[:2]

random.seed(0)
VAL_FRACTION = 0.05
total_train_out, total_val_out = 0, 0

for f in sorted(QA_DIR.glob("*.jsonl")):
    records = []
    for line in f.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        if len(rec.get("messages", [])) == 2:
            records.append(rec)

    # Group by ANSWER text -- variants of the same fact always share the
    # exact same answer, so this keeps originals + their reworded variants
    # together on the same side of the split (prevents leakage).
    groups = {}
    for rec in records:
        answer = rec["messages"][1]["content"]
        groups.setdefault(answer, []).append(rec)

    group_keys = list(groups.keys())
    random.shuffle(group_keys)

    n_val_target = max(1, int(len(records) * VAL_FRACTION))
    val_groups, train_groups = [], []
    val_count = 0
    for key in group_keys:
        if val_count < n_val_target:
            val_groups.append(key)
            val_count += len(groups[key])
        else:
            train_groups.append(key)

    out_lines = []
    for key in train_groups:
        for rec in groups[key]:
            q, a = rec["messages"][0]["content"], rec["messages"][1]["content"]
            out_lines.append(json.dumps(rec, ensure_ascii=False))
            total_train_out += 1
            for vq in reword(q):
                out_lines.append(json.dumps(
                    {"messages": [{"role": "user", "content": vq},
                                   {"role": "assistant", "content": a}]},
                    ensure_ascii=False))
                total_train_out += 1
    for key in val_groups:
        for rec in groups[key]:
            # Val stays PURE -- original phrasing only, no augmentation
            out_lines.append(json.dumps(rec, ensure_ascii=False))
            total_val_out += 1

    out_path = AUG_DIR / f.name
    out_path.write_text("\n".join(out_lines) + "\n", encoding="utf-8")
    print(f"{f.name}: {len(train_groups)} train-groups (augmented) + "
          f"{len(val_groups)} val-groups (pure) -> {out_path}")

print(f"\nTotal: {total_train_out} train-side records, {total_val_out} val-side records")

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO_DIR = Path("/content/gamax1_project")

BASE_CKPT = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt")
QA_DATA = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA_augmented/")   # original 2.2x augmented
REPLAY_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/books_cleaned_v1")
REPLAY_CACHE = Path("/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache")

FT_OUT_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v5")
FT_OUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "-m", "gamax1.finetune",
    "--init_from", str(BASE_CKPT),
    "--data", str(QA_DATA),
    "--out_dir", str(FT_OUT_DIR),
    "--max_len", "128",
    "--batch_size", "8",
    "--epochs", "6",              # thoda zyada rakha, kyunki ab val_loss trustworthy hai, overfit-onset khud dikh jayega
    "--lr", "3e-5",
    "--warmup_steps", "50",
    "--val_fraction", "0.05",
    "--eval_interval", "50",
    "--checkpoint_interval", "150",
    "--freeze_sparsity",
    "--replay_data_dir", str(REPLAY_DIR),
    "--replay_ratio", "0.3",
    "--replay_cache_dir", str(REPLAY_CACHE),
    "--seed", "0",
    "--device", "cuda",
]

print("FINE-TUNE v5: augmented data + leak-free split + replay, 6 epochs")
print(" ".join(cmd))
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
process = subprocess.Popen(cmd, cwd=str(REPO_DIR), env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
try:
    for line in iter(process.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)
finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()
print("\n🎉 FINE-TUNE v5 FINISHED" if return_code == 0 else f"\n❌ exited with code {return_code}")

In [ ]:
import shutil
from pathlib import Path

REPO_DIR = Path("/content/gamax1_project")

# Apna upload kiya hua patched-file path yahan daalo
SRC = Path("/content/drive/MyDrive/<jahan upload kiya>/instruction_data.py")
DST = REPO_DIR / "gamax1" / "instruction_data.py"

if not SRC.exists():
    raise FileNotFoundError(f"Pehle patched instruction_data.py ko Drive pe upload karo: {SRC}")

shutil.copy(SRC, DST)
print(f"✓ Copied: {SRC} -> {DST}")

# VERIFY: naya grouping-logic file mein hai ya nahi
content = DST.read_text()
if "Group pairs by answer text" in content:
    print("✓ CONFIRMED: patched version (group-aware split) is now active")
else:
    print("⚠ WARNING: patch marker not found -- old version may still be in place")

In [ ]:
import subprocess
# Poore Drive mein instruction_data.py dhoondo
r = subprocess.run(["find", "/content/drive/MyDrive", "-iname", "instruction_data.py"],
                    capture_output=True, text=True)
print(r.stdout)
print(r.stderr)

In [ ]:
from google.colab import files
uploaded = files.upload()  # Ye ek "Choose File" button dikhayega

In [ ]:
import shutil
from pathlib import Path

REPO_DIR = Path("/content/gamax1_project")
DST = REPO_DIR / "gamax1" / "instruction_data.py"

# Jo abhi upload hui, uska naam 'instruction_data.py' hona chahiye
shutil.copy("/content/instruction_data.py", DST)
print(f"✓ Copied to: {DST}")

content = DST.read_text()
if "Group pairs by answer text" in content:
    print("✓ CONFIRMED: patched version (group-aware split) is now active")
else:
    print("⚠ WARNING: patch marker not found")

In [ ]:
import subprocess
r1 = subprocess.run(["git", "add", "gamax1/instruction_data.py"], cwd="/content/gamax1_project", capture_output=True, text=True)
r2 = subprocess.run(["git", "commit", "-m", "Fix split_train_val to be answer-grouped, preventing augmented-variant leakage"], cwd="/content/gamax1_project", capture_output=True, text=True)
r3 = subprocess.run(["git", "push"], cwd="/content/gamax1_project", capture_output=True, text=True)
print("--- commit ---"); print(r2.stdout, r2.stderr)
print("--- push ---"); print(r3.stdout, r3.stderr)

In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO_DIR = Path("/content/gamax1_project")

BASE_CKPT = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt")
QA_DATA = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA_augmented/")
REPLAY_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/books_cleaned_v1")
REPLAY_CACHE = Path("/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache")

FT_OUT_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v7")
FT_OUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "-m", "gamax1.finetune",
    "--init_from", str(BASE_CKPT),
    "--data", str(QA_DATA),
    "--out_dir", str(FT_OUT_DIR),
    "--max_len", "128",
    "--batch_size", "8",
    "--epochs", "6",
    "--lr", "3e-5",
    "--warmup_steps", "50",
    "--val_fraction", "0.05",
    "--eval_interval", "50",
    "--checkpoint_interval", "150",
    "--freeze_sparsity",
    "--replay_data_dir", str(REPLAY_DIR),
    "--replay_ratio", "0.3",
    "--replay_cache_dir", str(REPLAY_CACHE),
    "--seed", "0",
    "--device", "cuda",
]

print("FINE-TUNE v6: augmented data + TRUE leak-free split (patch confirmed) + replay, 6 epochs")
print(" ".join(cmd))
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
process = subprocess.Popen(cmd, cwd=str(REPO_DIR), env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
try:
    for line in iter(process.stdout.readline, ""):
        if line:
            print(line, end="", flush=True)
finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()
print("\n🎉 FINE-TUNE v7 FINISHED" if return_code == 0 else f"\n❌ exited with code {return_code}")

In [ ]:
%cd /content/gamax1_project

questions = [
    ("In which Article of the Constitution of India is there a provision for the abolition of untouchability?", "In Article 17"),
    ("Which river forms the Sundarbans Delta?", "Ganga, Brahmaputra, and Meghna"),
    ("Carbon dating method is used to determine the age of?", "Fossils"),
    ("Which organ acts as the blood bank in the body?", "Spleen"),
    ("In which state is the largest network of rivers found in India?", "In Uttar Pradesh"),
]

CKPT = "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v6/gamax1_finetune_best.pt"

for q, expected in questions:
    print(f"\n{'='*70}\nQ: {q}\nEXPECTED: {expected}\n{'='*70}")
    !python -m gamax1.generate \
      --ckpt "{CKPT}" \
      --prompt "{q}" \
      --max_new_tokens 40 \
      --temperature 0.1 \
      --top_k 1 \
      --chat --stop_at_eos \
      --device cuda

In [ ]:
import json
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

QA_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA")  # original, non-augmented

# Saare original Q&A records load karo
records = []
for f in sorted(QA_DIR.glob("*.jsonl")):
    for line in f.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        msgs = rec.get("messages", [])
        if len(msgs) == 2:
            records.append({
                "question": msgs[0]["content"],
                "answer": msgs[1]["content"],
                "source": f.name,
            })

print(f"Total records loaded: {len(records)}")

# Lightweight, accurate embedding model -- Colab GPU pe fast chalega
embedder = SentenceTransformer("all-MiniLM-L6-v2")

questions = [r["question"] for r in records]
print("Encoding all questions (one-time, ~30-60s for 3700 records)...")
question_embeddings = embedder.encode(questions, convert_to_numpy=True, show_progress_bar=True, normalize_embeddings=True)
print(f"✓ Embeddings ready: {question_embeddings.shape}")

def retrieve(query, top_k=3, min_similarity=0.55):
    """Return the best-matching (question, answer, similarity) tuples."""
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = question_embeddings @ q_emb   # cosine similarity (vectors already normalized)
    top_idx = np.argsort(-sims)[:top_k]
    results = []
    for i in top_idx:
        if sims[i] >= min_similarity:
            results.append({
                "question": records[i]["question"],
                "answer": records[i]["answer"],
                "similarity": float(sims[i]),
                "source": records[i]["source"],
            })
    return results

def answer(query):
    hits = retrieve(query, top_k=1, min_similarity=0.55)
    if not hits:
        return "❌ No confident match found in the knowledge base."
    best = hits[0]
    return f"✅ {best['answer']}  (matched: {best['question']!r}, similarity={best['similarity']:.2f})"

In [ ]:
test_questions = [
    "In which Article of the Constitution of India is there a provision for the abolition of untouchability?",
    "Which river forms the Sundarbans Delta?",
    "Carbon dating method is used to determine the age of?",
    "Which organ acts as the blood bank in the body?",
    "In which state is the largest network of rivers found in India?",
]
for q in test_questions:
    print(f"\nQ: {q}")
    print(answer(q))

# Bonus: thoda alag-phrasing mein bhi test karo (semantic-match ka asli test)
print("\n--- Rephrased queries (semantic match test) ---")
for q in [
    "Spleen's role is called what in the human body?",
    "What rivers join to form the Sundarbans?",
]:
    print(f"\nQ: {q}")
    print(answer(q))

In [ ]:
def smart_answer(query, similarity_threshold=0.65):
    """Retrieval-first, model-fallback hybrid answering."""
    hits = retrieve(query, top_k=1, min_similarity=similarity_threshold)
    if hits:
        return {
            "source": "retrieval",
            "answer": hits[0]["answer"],
            "confidence": hits[0]["similarity"],
            "matched_question": hits[0]["question"],
        }
    else:
        # Fallback to the fine-tuned model for out-of-dataset questions
        import subprocess, sys
        CKPT = "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v6/gamax1_finetune_best.pt"
        cmd = [sys.executable, "-u", "-m", "gamax1.generate",
               "--ckpt", CKPT, "--prompt", query,
               "--max_new_tokens", "40", "--temperature", "0.6", "--top_k", "30",
               "--repetition_penalty", "1.15", "--chat", "--stop_at_eos", "--device", "cuda"]
        r = subprocess.run(cmd, cwd="/content/gamax1_project", capture_output=True, text=True)
        return {"source": "model (no confident match found)", "raw_output": r.stdout}

# Test
for q in ["Who is the President of India?", "What is the capital of India?"]:
    print(f"\nQ: {q}")
    print(smart_answer(q))


Q: Who is the President of India?


NameError: name 'retrieve' is not defined

In [ ]:
import re

def extract_key_terms(text):
    """Extract proper nouns, numbers, and distinguishing capitalized words
    (excluding common sentence-starters like Who/What/Which/In)."""
    stopwords = {"who", "what", "which", "in", "is", "the", "of", "are",
                 "india", "indian"}  # "India/Indian" too common in this dataset to be distinguishing
    words = re.findall(r"[A-Za-z]+|\d+", text)
    key = set()
    for w in words:
        wl = w.lower()
        if wl in stopwords:
            continue
        if w[0].isupper() or w.isdigit():
            key.add(wl)
    return key

def retrieve_verified(query, top_k=5, min_similarity=0.55, min_entity_overlap=0.5):
    """Embedding retrieval + entity-overlap verification to reject
    same-template-different-fact false positives."""
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = question_embeddings @ q_emb
    top_idx = np.argsort(-sims)[:top_k]

    query_entities = extract_key_terms(query)

    for i in top_idx:
        if sims[i] < min_similarity:
            break
        candidate_q = records[i]["question"]
        cand_entities = extract_key_terms(candidate_q)

        if not query_entities and not cand_entities:
            # Neither has distinguishing entities -- trust pure similarity
            overlap_ok = True
        elif not query_entities or not cand_entities:
            overlap_ok = False
        else:
            overlap = len(query_entities & cand_entities) / len(query_entities | cand_entities)
            overlap_ok = overlap >= min_entity_overlap

        if overlap_ok:
            return {
                "question": candidate_q,
                "answer": records[i]["answer"],
                "similarity": float(sims[i]),
                "source": records[i]["source"],
            }
    return None

def answer_v2(query):
    hit = retrieve_verified(query)
    if not hit:
        return "❌ No confident, verified match found in the knowledge base."
    return f"✅ {hit['answer']}  (matched: {hit['question']!r}, similarity={hit['similarity']:.2f})"

In [ ]:
print("--- Should be correct (in training data) ---")
for q in [
    "In which Article of the Constitution of India is there a provision for the abolition of untouchability?",
    "Which river forms the Sundarbans Delta?",
]:
    print(f"\nQ: {q}")
    print(answer_v2(q))

print("\n--- Should be correct (rephrased, in training data) ---")
for q in ["What rivers join to form the Sundarbans?"]:
    print(f"\nQ: {q}")
    print(answer_v2(q))

print("\n--- Should REJECT (not in training data, different entity) ---")
for q in ["Who is the President of India?", "What is the capital of India?"]:
    print(f"\nQ: {q}")
    print(answer_v2(q))

In [ ]:
STOPWORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "of", "in",
    "on", "at", "to", "for", "by", "do", "does", "did", "and", "or", "this",
    "that", "these", "those", "which", "what", "who", "whom", "whose", "how",
    "when", "where", "why", "there", "it", "its", "as", "with", "from",
    "into", "about", "india", "indian",
}

def extract_key_terms(text):
    """Content words (not just capitalized ones), lightly stemmed via
    4-char prefix truncation so plurals/tenses (river/rivers, form/forms)
    still match."""
    words = re.findall(r"[A-Za-z]+|\d+", text)
    terms = set()
    for w in words:
        wl = w.lower()
        if wl in STOPWORDS or len(wl) < 3:
            continue
        stem = wl[:4] if len(wl) > 4 else wl
        terms.add(stem)
    return terms

def retrieve_verified(query, top_k=5, min_similarity=0.55, min_lexical_overlap=0.35):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = question_embeddings @ q_emb
    top_idx = np.argsort(-sims)[:top_k]

    query_terms = extract_key_terms(query)

    for i in top_idx:
        if sims[i] < min_similarity:
            break
        candidate_q = records[i]["question"]
        cand_terms = extract_key_terms(candidate_q)

        if not query_terms or not cand_terms:
            continue  # can't verify -- reject rather than risk a false positive

        overlap = len(query_terms & cand_terms) / len(query_terms | cand_terms)
        if overlap >= min_lexical_overlap:
            return {
                "question": candidate_q,
                "answer": records[i]["answer"],
                "similarity": float(sims[i]),
                "lexical_overlap": overlap,
                "source": records[i]["source"],
            }
    return None

def answer_v3(query):
    hit = retrieve_verified(query)
    if not hit:
        return "❌ No confident, verified match found in the knowledge base."
    return (f"✅ {hit['answer']}  (matched: {hit['question']!r}, "
            f"sim={hit['similarity']:.2f}, lex_overlap={hit['lexical_overlap']:.2f})")

In [ ]:
print("--- Should be correct ---")
for q in [
    "In which Article of the Constitution of India is there a provision for the abolition of untouchability?",
    "Which river forms the Sundarbans Delta?",
    "What rivers join to form the Sundarbans?",
]:
    print(f"\nQ: {q}")
    print(answer_v3(q))

print("\n--- Should REJECT ---")
for q in ["Who is the President of India?", "What is the capital of India?"]:
    print(f"\nQ: {q}")
    print(answer_v3(q))

In [ ]:
import re

SENTENCE_STARTERS = {"what","which","who","whom","whose","in","how","when","where","why","is","are","do","does"}
ORDINAL_WORDS = {"first","second","third","fourth","fifth","sixth","seventh","eighth",
                  "ninth","tenth","eleventh","twelfth","thirteenth","fourteenth","fifteenth"}

def extract_proper_nouns(text):
    """Capitalized words, excluding sentence-initial word and question-starters."""
    words = re.findall(r"[A-Za-z]+", text)
    entities = set()
    for idx, w in enumerate(words):
        if idx == 0:
            continue
        if w[0].isupper() and w.lower() not in SENTENCE_STARTERS:
            entities.add(w.lower())
    return entities

def extract_ordinals_numbers(text):
    words = re.findall(r"[A-Za-z]+|\d+", text.lower())
    return {w for w in words if w.isdigit() or w in ORDINAL_WORDS}

def retrieve_verified(query, top_k=5, min_similarity=0.55):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = question_embeddings @ q_emb
    top_idx = np.argsort(-sims)[:top_k]

    q_ent = extract_proper_nouns(query)
    q_num = extract_ordinals_numbers(query)

    for i in top_idx:
        if sims[i] < min_similarity:
            break
        candidate_q = records[i]["question"]
        c_ent = extract_proper_nouns(candidate_q)
        c_num = extract_ordinals_numbers(candidate_q)

        entity_ok = q_ent.issubset(c_ent) if q_ent else True
        number_ok = (q_num == c_num)

        if entity_ok and number_ok:
            return {
                "question": candidate_q,
                "answer": records[i]["answer"],
                "similarity": float(sims[i]),
                "source": records[i]["source"],
            }
    return None

def answer_v4(query):
    hit = retrieve_verified(query)
    if not hit:
        return "❌ No confident, verified match found in the knowledge base."
    return f"✅ {hit['answer']}  (matched: {hit['question']!r}, sim={hit['similarity']:.2f})"

In [ ]:
print("--- Should be correct ---")
for q in [
    "In which Article of the Constitution of India is there a provision for the abolition of untouchability?",
    "Which river forms the Sundarbans Delta?",
    "What rivers join to form the Sundarbans?",
]:
    print(f"\nQ: {q}")
    print(answer_v4(q))

print("\n--- Should REJECT ---")
for q in ["Who is the President of India?", "What is the capital of India?"]:
    print(f"\nQ: {q}")
    print(answer_v4(q))

In [ ]:
import re

SENTENCE_STARTERS = {"what","which","who","whom","whose","in","how","when","where","why","is","are","do","does"}
ORDINAL_WORDS = {"first","second","third","fourth","fifth","sixth","seventh","eighth",
                  "ninth","tenth","eleventh","twelfth","thirteenth","fourteenth","fifteenth"}
LINK_WORDS = {"of","is","are","the","a","an","in","to","by","and","or","that","this",
              "for","as","with","from","into","about","there","was","were"}
COUNTRY_SELF = {"india","indian"}

def extract_proper_nouns(text):
    words = re.findall(r"[A-Za-z]+", text)
    entities = set()
    for idx, w in enumerate(words):
        if idx == 0:
            continue
        if w[0].isupper() and w.lower() not in SENTENCE_STARTERS:
            entities.add(w.lower())
    return entities

def extract_ordinals_numbers(text):
    words = re.findall(r"[A-Za-z]+|\d+", text.lower())
    return {w for w in words if w.isdigit() or w in ORDINAL_WORDS}

def extract_topic_words(text):
    words = re.findall(r"[A-Za-z]+", text.lower())
    terms = set()
    for i, w in enumerate(words):
        if i == 0 or w in SENTENCE_STARTERS or w in LINK_WORDS or w in COUNTRY_SELF:
            continue
        if len(w) < 3:
            continue
        terms.add(w[:4] if len(w) > 4 else w)
    return terms

def retrieve_verified(query, top_k=5, min_similarity=0.55, topic_threshold=0.4):
    q_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]
    sims = question_embeddings @ q_emb
    top_idx = np.argsort(-sims)[:top_k]

    q_ent = extract_proper_nouns(query)
    q_num = extract_ordinals_numbers(query)
    q_top = extract_topic_words(query)

    for i in top_idx:
        if sims[i] < min_similarity:
            break
        candidate_q = records[i]["question"]
        c_ent = extract_proper_nouns(candidate_q)
        c_num = extract_ordinals_numbers(candidate_q)
        c_top = extract_topic_words(candidate_q)

        entity_ok = q_ent.issubset(c_ent) if q_ent else True
        number_ok = (q_num == c_num)
        overlap = len(q_top & c_top) / len(q_top | c_top) if (q_top | c_top) else 1.0
        topic_ok = overlap >= topic_threshold

        if entity_ok and number_ok and topic_ok:
            return {
                "question": candidate_q,
                "answer": records[i]["answer"],
                "similarity": float(sims[i]),
                "source": records[i]["source"],
            }
    return None

def answer_v5(query):
    hit = retrieve_verified(query)
    if not hit:
        return "❌ No confident, verified match found in the knowledge base."
    return f"✅ {hit['answer']}  (matched: {hit['question']!r}, sim={hit['similarity']:.2f})"

In [ ]:
print("--- Should be correct ---")
for q in [
    "In which Article of the Constitution of India is there a provision for the abolition of untouchability?",
    "Which river forms the Sundarbans Delta?",
    "What rivers join to form the Sundarbans?",
]:
    print(f"\nQ: {q}")
    print(answer_v5(q))

print("\n--- Should REJECT ---")
for q in ["Who is the President of India?", "What is the capital of India?"]:
    print(f"\nQ: {q}")
    print(answer_v5(q))

In [ ]:
import subprocess, sys

FALLBACK_CKPT = "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v6/gamax1_finetune_best.pt"

def smart_answer_v2(query):
    """Retrieval-first (verified, high-precision), model-fallback for
    anything not confidently found in the knowledge base."""
    hit = retrieve_verified(query)
    if hit:
        return {
            "source": "retrieval",
            "answer": hit["answer"],
            "confidence": round(hit["similarity"], 2),
            "matched_question": hit["question"],
            "dataset_file": hit["source"],
        }

    # Fallback: generate with the fine-tuned model (less reliable, but
    # better than a flat refusal for genuinely out-of-dataset questions)
    cmd = [sys.executable, "-u", "-m", "gamax1.generate",
           "--ckpt", FALLBACK_CKPT, "--prompt", query,
           "--max_new_tokens", "40", "--temperature", "0.6", "--top_k", "30",
           "--repetition_penalty", "1.15", "--chat", "--stop_at_eos", "--device", "cuda"]
    r = subprocess.run(cmd, cwd="/content/gamax1_project", capture_output=True, text=True)

    # Extract just the assistant's reply from decode_with_boundaries output
    raw = r.stdout
    reply = raw.split("<|assistant|>")[-1].replace("<|eos|>", "").strip() if "<|assistant|>" in raw else raw.strip()

    return {
        "source": "model (fallback, no confident retrieval match)",
        "answer": reply,
        "confidence": None,
        "matched_question": None,
        "dataset_file": None,
    }

In [ ]:
test_cases = [
    "In which Article of the Constitution of India is there a provision for the abolition of untouchability?",  # in-dataset
    "What rivers join to form the Sundarbans?",   # in-dataset, rephrased
    "Who is the President of India?",             # out-of-dataset
    "What is the capital of India?",               # out-of-dataset
    "What is 5 + 7?",                               # totally unrelated
]

for q in test_cases:
    print(f"\n{'='*70}\nQ: {q}")
    result = smart_answer_v2(q)
    print(f"Source: {result['source']}")
    print(f"Answer: {result['answer']}")
    if result['confidence'] is not None:
        print(f"Confidence: {result['confidence']} | Matched: {result['matched_question']!r}")

In [ ]:
import random

random.seed(42)
sample_size = 100
sample_records = random.sample(records, min(sample_size, len(records)))

correct = 0
no_match = 0
wrong = 0
results_log = []

for rec in sample_records:
    hit = retrieve_verified(rec["question"])
    if hit is None:
        no_match += 1
        status = "NO_MATCH"
    elif hit["answer"].strip() == rec["answer"].strip():
        correct += 1
        status = "CORRECT"
    else:
        wrong += 1
        status = "WRONG_ANSWER"  # matched something, but answer differs from ground truth
    results_log.append({
        "question": rec["question"],
        "expected": rec["answer"],
        "got": hit["answer"] if hit else None,
        "status": status,
    })

print(f"Sample size: {len(sample_records)}")
print(f"✅ Correct:    {correct}  ({correct/len(sample_records)*100:.1f}%)")
print(f"❌ No match:   {no_match}  ({no_match/len(sample_records)*100:.1f}%)")
print(f"⚠️  Wrong:      {wrong}  ({wrong/len(sample_records)*100:.1f}%)")

if wrong > 0:
    print("\n--- Wrong-answer cases (worth inspecting) ---")
    for r in results_log:
        if r["status"] == "WRONG_ANSWER":
            print(f"\nQ: {r['question']}")
            print(f"Expected: {r['expected']!r}")
            print(f"Got:      {r['got']!r}")

if no_match > 0:
    print(f"\n--- Sample of NO_MATCH cases (first 5) ---")
    shown = 0
    for r in results_log:
        if r["status"] == "NO_MATCH" and shown < 5:
            print(f"\nQ: {r['question']}")
            print(f"Expected: {r['expected']!r}")
            shown += 1

In [ ]:
import pickle
from pathlib import Path

KB_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/knowledge_base")
KB_DIR.mkdir(parents=True, exist_ok=True)

# Save embeddings (numpy array) + records (metadata) together
np.save(KB_DIR / "question_embeddings.npy", question_embeddings)
with open(KB_DIR / "records.pkl", "wb") as f:
    pickle.dump(records, f)

print(f"✓ Saved {len(records)} records + embeddings to {KB_DIR}")
print(f"  question_embeddings.npy: {(KB_DIR / 'question_embeddings.npy').stat().st_size / 1024:.1f} KB")
print(f"  records.pkl: {(KB_DIR / 'records.pkl').stat().st_size / 1024:.1f} KB")

In [ ]:
def load_knowledge_base():
    """One-call setup for a fresh Colab session -- loads pre-computed
    embeddings instead of re-encoding all questions from scratch."""
    import numpy as np, pickle
    from pathlib import Path
    from sentence_transformers import SentenceTransformer

    KB_DIR = Path("/content/drive/MyDrive/Aetherion_GamaX1/knowledge_base")
    embedder = SentenceTransformer("all-MiniLM-L6-v2")
    question_embeddings = np.load(KB_DIR / "question_embeddings.npy")
    with open(KB_DIR / "records.pkl", "rb") as f:
        records = pickle.load(f)
    print(f"✓ Loaded {len(records)} records, embeddings shape {question_embeddings.shape}")
    return embedder, question_embeddings, records

# Quick test: confirm reload works and gives identical results
embedder2, question_embeddings2, records2 = load_knowledge_base()
assert np.allclose(question_embeddings, question_embeddings2), "Mismatch after reload!"
print("✓ Reload verified -- identical to original")

In [ ]:
import sys
from pathlib import Path

# Ensure the project directory is in Python's search path
REPO_DIR = "/content/gamax1_project"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from gamax1.knowledge_qa import load_knowledge_base, smart_answer

embedder, question_embeddings, records = load_knowledge_base()
result = smart_answer("Which river forms the Sundarbans Delta?", embedder, question_embeddings, records)
print(result)

In [ ]:
%cd /content/gamax1_project
!python -m gamax1.generate \
  --ckpt "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v6/gamax1_finetune_best.pt" \
  --prompt "Which river forms the Sundarbans Delta?" \
  --chat --knowledge_base --stop_at_eos --device cuda

/content/gamax1_project
modules.json: 100% 349/349 [00:00<00:00, 1.80MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 703kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 31.4MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 347kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.91MB/s]

model.safetensors: downloading bytes:  78% 70.5M/90.9M [00:01<00:00, 64.9MB/s, 5.58MB/s  ]
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 45.6MB/s, 8.03MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 48.7MB/s, 8.70MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 4618.44it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 1.42MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 64.5MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 110MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 520kB/s]
config.json: 100% 190/190 [00:00<00:00, 1.04MB/s]
✓ Loaded 3724 records, embeddings shape (3724, 384)
[knowledge_base] matched 'W

In [ ]:
import torch
from gamax1.tokenizer import BPETokenizer

ckpt = torch.load(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt",
    map_location="cpu", weights_only=False,
)
tok = BPETokenizer(merges=ckpt["merges"])

# Test: kya digits isolated hain ya merge ho sakte hain?
test_cases = ["2", "22", "222", "17", "1947", "100000"]
for t in test_cases:
    ids = tok.encode(t)
    tokens = [tok.decode([i]) for i in ids]
    print(f"{t!r:12} -> {len(ids)} tokens -> {tokens}")

print(f"\nTotal vocab size: {tok.vocab_size}")

'2'          -> 1 tokens -> ['2']
'22'         -> 2 tokens -> ['2', '2']
'222'        -> 3 tokens -> ['2', '2', '2']
'17'         -> 2 tokens -> ['1', '7']
'1947'       -> 4 tokens -> ['1', '9', '4', '7']
'100000'     -> 6 tokens -> ['1', '0', '0', '0', '0', '0']

Total vocab size: 16006


# fine tuning start

In [ ]:
%cd /content/gamax1_project

from pathlib import Path
from gamax1.instruction_data import load_pairs

data_dir = Path("/content/drive/MyDrive/Aetherion_GamaX1/data/QnA")
pairs, stats = load_pairs(str(data_dir))

print("Total pairs:", len(pairs))
for i, pair in enumerate(pairs[:5], 1):
    print(f"\nExample {i}:", repr(pair))

/content/gamax1_project
Total pairs: 3724

Example 1: ('In an important judgment, the Supreme Court has brought Muslim husbands also under the purview of Section 125 of the Criminal Procedure Code; what is in this judgment?', 'The father is responsible for the expenses of children living with a divorced wife until they become adults.')

Example 2: ('When and as President of which Congress session did Womesh Chandra Bonnerjee, the first President of the Indian National Congress, become President again?', '1892, Allahabad Session')

Example 3: ('In which Article of the Constitution of India is there a provision for the abolition of untouchability?', 'In Article 17')

Example 4: ('When does the Comptroller and Auditor General retire?', 'On completion of 6 years of appointment or 65 years of age')

Example 5: ('Who appoints the Chief Election Commissioner?', 'The President')


In [ ]:
%cd /content/gamax1_project
!grep -n -E "prompt|encode|decode|temperature|top_k|generate" gamax1/generate.py

/content/gamax1_project
2:gamax1/generate.py
4:Load a trained GamaX1 checkpoint and generate text.
53:    parser.add_argument("--prompt", type=str, default="\n")
55:    parser.add_argument("--temperature", type=float, default=0.8)
56:    parser.add_argument("--top_k", type=int, default=20)
71:                         help="Wrap --prompt as a user turn using the tokenizer's reserved "
73:                              "\"User:\"/\"Assistant:\") and generate the assistant's reply. "
80:                         help="With --chat, generate the full --max_new_tokens instead of "
98:    if args.temperature <= 0:
99:        parser.error("--temperature must be > 0")
100:    if args.top_k < 0:
101:        parser.error("--top_k must be >= 0")
115:        hit = retrieve_verified(args.prompt, embedder, question_embeddings, records)
124:            print(f"\n\n<|user|>\n\n{args.prompt}\n\n<|assistant|>\n\n{hit['answer']}\n\n<|eos|>\n\n")
142:        prompt_ids = [tok.user_id] + tok.encode(args.promp

In [ ]:
%cd /content/gamax1_project

print("========== GENERATION CODE ==========")
!sed -n '100,175p' gamax1/generate.py

print("\n========== FINETUNING CODE ==========")
!grep -n -E "tok.encode|tokenizer|prompt|answer|model_input|input_ids|labels|loss|eos_id|user_id|assistant_id" gamax1/finetune.py

print("\n========== DATA LOADING CODE ==========")
!grep -n -E "def load_pairs|def split_train_val|return pairs|json|question|answer" gamax1/instruction_data.py

/content/gamax1_project
========== GENERATION CODE ==========
    if args.top_k < 0:
        parser.error("--top_k must be >= 0")
    if args.repetition_penalty <= 0:
        parser.error("--repetition_penalty must be > 0")
    if args.no_stop_at_eos and not args.chat and not args.stop_at_eos:
        parser.error("--no_stop_at_eos is only meaningful with --chat")
    if args.knowledge_base and not args.chat:
        parser.error("--knowledge_base is only meaningful with --chat")

    if args.knowledge_base:
        # Retrieval-first: try this BEFORE loading the (much heavier) GamaX1
        # checkpoint, so a knowledge-base hit skips the model load entirely.
        from .knowledge_qa import load_knowledge_base, retrieve_verified
        import sys as _sys
        embedder, question_embeddings, records = load_knowledge_base()
        hit = retrieve_verified(args.prompt, embedder, question_embeddings, records)
        if hit is not None:
            print(f"[knowledge_base] matched {hi

In [ ]:

%cd /content/gamax1_project

from pathlib import Path
import inspect
import gamax1.instruction_data as data
import gamax1.generate as gen

print("===== DATASET FUNCTIONS =====")
print([
    name for name, obj in inspect.getmembers(data, inspect.isfunction)
    if not name.startswith("_")
])

print("\n===== ENCODING FUNCTION SOURCE =====")
for name in ["encode_instruction", "encode_example", "encode_pair", "load_pairs"]:
    obj = getattr(data, name, None)
    if obj is not None and inspect.isfunction(obj):
        print(f"\n--- {name} ---")
        print(inspect.getsource(obj))

print("\n===== GENERATION DEFAULTS =====")
print(inspect.getsource(gen.main))

print("\n===== TOKENIZER FILE =====")
p = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v3/tokenizer.json"
)
print("Exists:", p.exists())


/content/gamax1_project
===== DATASET FUNCTIONS =====
['encode_example', 'iter_files', 'load_pairs', 'make_collate_fn', 'split_train_val']

===== ENCODING FUNCTION SOURCE =====

--- encode_example ---
def encode_example(
    tokenizer,
    prompt_text: str,
    answer_text: str,
    max_len: int,
):
    """
    Encode one prompt/answer pair.

    Final structure:

        <|user|>
        prompt tokens
        <|assistant|>
        answer tokens
        <|eos|>

    Token structure:

        ids =
            [
                user_id,
                prompt_tokens...,
                assistant_id,
                answer_tokens...,
                eos_id,
            ]

    Loss mask:

        [
            0,                 # <|user|>
            0, 0, 0, ...       # prompt
            0,                 # <|assistant|>
            1, 1, 1, ...       # answer
            1,                 # <|eos|>
        ]

    IMPORTANT
    ---------

    The tokenizer is responsible for ordinary

In [ ]:

%cd /content/gamax1_project

!python -m gamax1.generate \
  --ckpt "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v6/gamax1_finetune_best.pt" \
  --prompt "What is 'Apsara' completed 34 years ago?" \
  --chat \
  --max_new_tokens 400 \
  --temperature 1.0 \
  --top_k 1 \
  --stop_at_eos \
  --device cuda


/content/gamax1_project


<|user|>

What is 'Apsara' completed 34 years ago?

<|assistant|>

According to the National Development Council of India

<|eos|>




In [ ]:
from gamax1.instruction_data import load_pairs

pairs, stats = load_pairs(
    "/content/drive/MyDrive/Aetherion_GamaX1/data/QnA"
)

target = "Who appoints the Chief Election Commissioner?"

for q, a in pairs:
    if target.lower().strip() in q.lower().strip():
        print("QUESTION:", q)
        print("EXPECTED ANSWER:", a)
        break
else:
    print("Exact question not found in training data.")

QUESTION: Who appoints the Chief Election Commissioner?
EXPECTED ANSWER: The President


In [ ]:

from pathlib import Path
import subprocess
import sys
import os

REPO_DIR = Path("/content/gamax1_project")
SCRIPT = REPO_DIR / "gamax1" / "finetune.py"

BASE_CKPT = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt"
)
QA_DATA = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/data/QnA_augmented"
)
REPLAY_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/data/books_cleaned_v1"
)
REPLAY_CACHE = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache"
)
FT_OUT_DIR = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v7"
)

TOTAL_STEPS = 10_000

# Check prerequisites before modifying anything.
for path in (SCRIPT, BASE_CKPT, QA_DATA, REPLAY_DIR):
    if not path.exists():
        raise FileNotFoundError(f"Required path not found: {path}")

# Add --max_steps support to this repository version.
source = SCRIPT.read_text()

parser_anchor = (
    '    parser.add_argument("--epochs", type=int, default=None)\n'
)
if "--max_steps" not in source:
    if parser_anchor not in source:
        raise RuntimeError("Could not find the epochs argument; no changes made.")

    source = source.replace(
        parser_anchor,
        parser_anchor
        + '    parser.add_argument("--max_steps", type=int, default=None, '
        'help="Stop after this many Q&A training steps.")\n',
        1,
    )

validation_anchor = (
    '    if args.epochs < 1:\n'
    '        parser.error("--epochs must be >= 1")\n'
)
if 'if args.max_steps is not None and args.max_steps < 1:' not in source:
    if validation_anchor not in source:
        raise RuntimeError("Could not find the epochs validation block.")

    source = source.replace(
        validation_anchor,
        validation_anchor
        + '    if args.max_steps is not None and args.max_steps < 1:\n'
        '        parser.error("--max_steps must be >= 1")\n',
        1,
    )

old_steps = (
    '    steps_per_epoch = max(1, (len(train_dataset) + args.batch_size - 1) // args.batch_size)\n'
    '    max_steps = steps_per_epoch * args.epochs\n'
)
new_steps = (
    '    steps_per_epoch = max(1, (len(train_dataset) + args.batch_size - 1) // args.batch_size)\n'
    '    max_steps = args.max_steps if args.max_steps is not None else steps_per_epoch * args.epochs\n'
)
if old_steps in source:
    source = source.replace(old_steps, new_steps, 1)
elif new_steps not in source:
    raise RuntimeError("Could not find the step-count block.")

old_loop = '    for epoch in range(start_epoch, args.epochs + 1):\n'
new_loop = (
    '    epoch_limit = (max(args.epochs, start_epoch + (max_steps // steps_per_epoch) + 1)\n'
    '                   if args.max_steps is not None else args.epochs)\n'
    '    for epoch in range(start_epoch, epoch_limit + 1):\n'
)
if old_loop in source:
    source = source.replace(old_loop, new_loop, 1)
elif new_loop not in source:
    raise RuntimeError("Could not find the epoch loop.")

# Stop at the requested Q&A step count.
loop_anchor = (
    '        for batch_idx, (xb, yb, mask) in enumerate(epoch_loader):\n'
    '            if epoch == start_epoch and batch_idx < start_batch:\n'
    '                continue\n'
)
loop_replacement = loop_anchor + (
    '\n            if step >= max_steps:\n'
    '                break\n'
)
if '            if step >= max_steps:\n                break\n' not in source:
    if loop_anchor not in source:
        raise RuntimeError("Could not find the training batch loop.")
    source = source.replace(loop_anchor, loop_replacement, 1)

# Exit the outer epoch loop as well once the target is reached.
outer_anchor = (
    '            last_batch_in_epoch = batch_in_epoch\n'
)
outer_replacement = outer_anchor
# The outer-loop break belongs after the inner loop, so insert before final save.
save_anchor = (
    '    # If resuming from an already-completed checkpoint with the same --epochs,\n'
)
if '    if step >= max_steps:\n        pass  # requested step limit reached' not in source:
    if save_anchor not in source:
        raise RuntimeError("Could not find the final-save section.")
    # The inner loop ends immediately before the final-save comment.
    source = source.replace(
        save_anchor,
        '        if step >= max_steps:\n'
        '            break\n\n'
        + save_anchor,
        1,
    )

# Save a backup, then update the repository script.
backup = SCRIPT.with_suffix(".py.before_max_steps.bak")
if not backup.exists():
    backup.write_text(SCRIPT.read_text())
SCRIPT.write_text(source)

FT_OUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, "-u", "-m", "gamax1.finetune",
    "--init_from", str(BASE_CKPT),
    "--data", str(QA_DATA),
    "--out_dir", str(FT_OUT_DIR),
    "--max_len", "128",
    "--batch_size", "8",
    "--epochs", "6",
    "--max_steps", str(TOTAL_STEPS),
    "--lr", "3e-5",
    "--warmup_steps", "50",
    "--val_fraction", "0.05",
    "--eval_interval", "50",
    "--checkpoint_interval", "150",
    "--freeze_sparsity",
    "--replay_data_dir", str(REPLAY_DIR),
    "--replay_ratio", "0.3",
    "--replay_cache_dir", str(REPLAY_CACHE),
    "--seed", "0",
    "--device", "cuda",
]

print("Starting Gamma X1 fine-tuning")
print("Target Q&A steps:", f"{TOTAL_STEPS:,}")
print("Output:", FT_OUT_DIR)
print("Backup:", backup)
print()

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

# Verify the patched CLI before launching training.
help_result = subprocess.run(
    [sys.executable, "-m", "gamax1.finetune", "--help"],
    cwd=str(REPO_DIR),
    capture_output=True,
    text=True,
)
if help_result.returncode != 0 or "--max_steps" not in help_result.stdout:
    raise RuntimeError(
        "CLI verification failed. Training was not started.\n"
        + help_result.stdout + help_result.stderr
    )

process = subprocess.Popen(
    cmd,
    cwd=str(REPO_DIR),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in iter(process.stdout.readline, ""):
        print(line, end="", flush=True)
finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()
print(
    "\nFine-tuning process finished."
    if return_code == 0
    else f"\nFine-tuning failed with exit code {return_code}."
)


Starting Gamma X1 fine-tuning
Target Q&A steps: 10,000
Output: /content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v7
Backup: /content/gamax1_project/gamax1/finetune.py.before_max_steps.bak

Loaded checkpoint from /content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt (step ?)
Sparsity k: 384 (frozen: True)
Instruction data: 3 file(s), 8023 record(s) -> 8023 (prompt, answer) example(s); 0 record(s) skipped (unrecognized format)
Train examples: 7621 | Val examples: 402
Reusing bulk token cache: /content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache/tokens.int32.bin (701,873,500 tokens) -- no new or changed files
Bulk corpus ready: 4,593 files, 701,873,500 tokens | by source: {'books': 701873500}
Replay corpus: 701,873,500 tokens from /content/drive/MyDrive/Aetherion_GamaX1/data/books_cleaned_v1 (mixed in at ratio 0.3)
Starting new fine-tuning run from /content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt
epoch 1 | step     1/10000 | lr 

In [ ]:
%cd /content/gamax1_project

!python -m gamax1.generate \
  --ckpt "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v7/gamax1_finetune_best.pt" \
  --prompt "Who appoints the Chief Election Commissioner?" \
  --chat \
  --max_new_tokens 40 \
  --temperature 0.1 \
  --top_k 1 \
  --stop_at_eos \
  --device cuda

/content/gamax1_project


<|user|>

Who appoints the Chief Election Commissioner?

<|assistant|>

The President

<|eos|>




In [ ]:
%cd /content/gamax1_project

!python -m gamax1.generate \
  --ckpt "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v7/gamax1_finetune_best.pt" \
  --prompt "In which Part of the Constitution are Fundamental Rights described?" \
  --chat \
  --max_new_tokens 40 \
  --temperature 0.1 \
  --top_k 1 \
  --stop_at_eos \
  --device cuda

/content/gamax1_project


<|user|>

In which Part of the Constitution are Fundamental Rights described?

<|assistant|>

In Part 3

<|eos|>




In [ ]:
%cd /content/gamax1_project

CKPT = "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v7/gamax1_finetune_best.pt"

questions = [
    "Who appoints the Chief Election Commissioner?",
    "In which Part of the Constitution are Fundamental Rights described?",
    "Which State has the highest representation in Lok Sabha and Rajya Sabha?",
    "What process is mentioned in Article 368 of the Constitution?",
    "Who was the first Indian Governor General of Independent India?",
    "Name the person who operates the Contingency Fund of the State?"
]

for i, question in enumerate(questions, 1):
    print(f"\n{'='*60}")
    print(f"QUESTION {i}: {question}")
    print('='*60)

    !python -m gamax1.generate --ckpt "$CKPT" --prompt "$question" --chat --max_new_tokens 80 --temperature 0.1 --top_k 1 --stop_at_eos --device cuda

/content/gamax1_project

QUESTION 1: Who appoints the Chief Election Commissioner?


<|user|>

Who appoints the Chief Election Commissioner?

<|assistant|>

The President

<|eos|>



QUESTION 2: In which Part of the Constitution are Fundamental Rights described?


<|user|>

In which Part of the Constitution are Fundamental Rights described?

<|assistant|>

In Part 3

<|eos|>



QUESTION 3: Which State has the highest representation in Lok Sabha and Rajya Sabha?


<|user|>

Which State has the highest representation in Lok Sabha and Rajya Sabha?

<|assistant|>

In Lok Sabha

<|eos|>



QUESTION 4: What process is mentioned in Article 368 of the Constitution?


<|user|>

What process is mentioned in Article 368 of the Constitution?

<|assistant|>

In Article 368

<|eos|>



QUESTION 5: Who was the first Indian Governor General of Independent India?


<|user|>

Who was the first Indian Governor General of Independent India?

<|assistant|>

Dr. Rajendra Prasad

<|eos|>



QUESTION 6: Name 

In [ ]:
%cd /content/gamax1_project

!python -m gamax1.generate \
  --ckpt "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/finetune-QNA-v7/gamax1_finetune_best.pt" \
  --prompt "Who operates the Contingency Fund of the State?" \
  --chat \
  --max_new_tokens 40 \
  --temperature 0.1 \
  --top_k 1 \
  --stop_at_eos \
  --device cuda

/content/gamax1_project


<|user|>

Who operates the Contingency Fund of the State?

<|assistant|>

The President

<|eos|>




In [ ]:
"""
Colab mein chalane ke liye: Discord-Dialogues aur Reddit-Constructive se
limited random files symlink karta hai (copy nahi), aur purane 4 sources
(books/math/conversations/QnA) ko bhi same combined data_dir mein link
karta hai -- taaki train.py ko ek hi --data_dir point karna kaafi ho.

Symlinks fixed seed ke saath banti hain, isliye baar-baar chalane par
wahi files milengi (cache-stability ke liye zaroori).
"""

import os
import random
from pathlib import Path

# ---- INPUT: apne hisaab se ye 2 paths badlo ----
DISCORD_SRC = "/content/drive/MyDrive/Aetherion_GamaX1/data/Discord-Dialogues"
REDDIT_SRC = "/content/drive/MyDrive/Aetherion_GamaX1/data/Reddit-Constructive"

# purane 4 sources jahan se already training chal rahi thi
OLD_DATA_ROOT = "/content/drive/MyDrive/Aetherion_GamaX1/data"
OLD_SOURCES = {
    "books_cleaned_v1": "books_cleaned_v1",
    "Math_Reasoning_train": "Math_Reasoning",       # poora folder link hoga
    "Conversations-200k_clean": "Conversations-200k_clean",
    "Q&A": "QnA",
}

# Naya combined data_dir jo train.py ko diya jayega
DATA_DIR = "/content/gamax1_data_subset"

N_DISCORD = 300
N_REDDIT = 150
SEED = 1234  # mat badalna -- isi se wahi subset dobara milta hai
# --------------------------------------------------


def pick_and_link(src: str, dst_folder_name: str, n: int) -> None:
    src_path = Path(src)
    if not src_path.is_dir():
        raise FileNotFoundError(f"Source folder nahi mila: {src}")

    files = sorted(src_path.glob("*.txt"))
    if not files:
        raise FileNotFoundError(f"{src} mein koi .txt file nahi mili")

    rng = random.Random(SEED)
    chosen = sorted(rng.sample(files, min(n, len(files))))

    dst = Path(DATA_DIR) / dst_folder_name
    dst.mkdir(parents=True, exist_ok=True)

    linked, already = 0, 0
    for f in chosen:
        link = dst / f.name
        if link.exists() or link.is_symlink():
            already += 1
            continue
        os.symlink(f, link)
        linked += 1

    print(
        f"{dst_folder_name}: {len(files):,} files available, "
        f"{len(chosen)} chosen ({linked} new links, {already} already present)"
    )


def link_whole_folder(src: str, dst_folder_name: str) -> None:
    src_path = Path(src)
    dst = Path(DATA_DIR) / dst_folder_name
    if dst.exists() or dst.is_symlink():
        print(f"{dst_folder_name}: already linked, skipping")
        return
    if not src_path.is_dir():
        print(f"[WARNING] {dst_folder_name}: source not found at {src}, skipping")
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(src_path, dst)
    print(f"{dst_folder_name}: linked whole folder -> {src}")


def main() -> None:
    Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

    pick_and_link(DISCORD_SRC, "Discord-Dialogues", N_DISCORD)
    pick_and_link(REDDIT_SRC, "Reddit-Constructive", N_REDDIT)

    for dst_name, sub_path in OLD_SOURCES.items():
        link_whole_folder(str(Path(OLD_DATA_ROOT) / sub_path), dst_name)

    print(f"\nDone. Combined data_dir ready at: {DATA_DIR}")
    print("train.py ko --data_dir isi path ke saath chalana.")


if __name__ == "__main__":
    main()

Discord-Dialogues: 3,651 files available, 300 chosen (300 new links, 0 already present)
Reddit-Constructive: 803 files available, 150 chosen (150 new links, 0 already present)
books_cleaned_v1: linked whole folder -> /content/drive/MyDrive/Aetherion_GamaX1/data/books_cleaned_v1
Math_Reasoning_train: linked whole folder -> /content/drive/MyDrive/Aetherion_GamaX1/data/Math_Reasoning
Conversations-200k_clean: linked whole folder -> /content/drive/MyDrive/Aetherion_GamaX1/data/Conversations-200k_clean
Q&A: linked whole folder -> /content/drive/MyDrive/Aetherion_GamaX1/data/QnA

Done. Combined data_dir ready at: /content/gamax1_data_subset
train.py ko --data_dir isi path ke saath chalana.


In [ ]:
from pathlib import Path
import os

candidates = [
    Path("/content/gamax1_project"),
    Path("/content/gamax1_data_subset"),
    Path("/content/drive/MyDrive/Aetherion_GamaX1"),
]

for root in candidates:
    print(f"\n=== {root} ===")
    print("exists:", root.exists())
    if root.exists():
        if root.name == "gamax1_project":
            for p in [root / "gamax1" / "finetune.py",
                      root / "gamax1" / "train.py",
                      root / "gamax1" / "instruction_data.py"]:
                print(("FOUND " if p.exists() else "MISSING "), p)
        elif root.name == "gamax1_data_subset":
            for p in sorted(root.iterdir())[:30]:
                print(("DIR  " if p.is_dir() else "FILE "), p.name)
        else:
            for p in [
                root / "runs/gamax1/full/gamax1.pt",
                root / "data/books_cleaned_v1",
                root / "data/Conversations-200k_clean",
                root / "data/QnA",
            ]:
                print(("FOUND " if p.exists() else "MISSING "), p)



=== /content/gamax1_project ===
exists: True
FOUND  /content/gamax1_project/gamax1/finetune.py
FOUND  /content/gamax1_project/gamax1/train.py
FOUND  /content/gamax1_project/gamax1/instruction_data.py

=== /content/gamax1_data_subset ===
exists: True
DIR   Conversations-200k_clean
DIR   Discord-Dialogues
DIR   Math_Reasoning_train
DIR   Q&A
DIR   Reddit-Constructive
DIR   books_cleaned_v1

=== /content/drive/MyDrive/Aetherion_GamaX1 ===
exists: True
FOUND  /content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt
FOUND  /content/drive/MyDrive/Aetherion_GamaX1/data/books_cleaned_v1
FOUND  /content/drive/MyDrive/Aetherion_GamaX1/data/Conversations-200k_clean
FOUND  /content/drive/MyDrive/Aetherion_GamaX1/data/QnA


In [ ]:
from pathlib import Path

root = Path("/content/gamax1_data_subset")

for folder_name in ["Discord-Dialogues", "Reddit-Constructive"]:
    folder = root / folder_name
    files = [p for p in folder.rglob("*") if p.is_file()]
    extensions = {}
    for p in files:
        ext = p.suffix.lower() or "[no extension]"
        extensions[ext] = extensions.get(ext, 0) + 1

    print(f"\n{folder_name}")
    print("Total files:", len(files))
    print("Formats:", extensions)
    for p in files[:3]:
        print("Sample:", p.name, "|", p.stat().st_size, "bytes")

print("\nQ&A folder:")
qa = root / "Q&A"
print("Exists:", qa.exists())
if qa.exists():
    for p in list(qa.rglob("*"))[:20]:
        if p.is_file():
            print(p.name, "|", p.suffix, "|", p.stat().st_size, "bytes")


Discord-Dialogues
Total files: 300
Formats: {'.txt': 300}
Sample: discord_01970.txt | 192903 bytes
Sample: discord_00590.txt | 216587 bytes
Sample: discord_00059.txt | 393341 bytes

Reddit-Constructive
Total files: 150
Formats: {'.txt': 150}
Sample: reddit_00479.txt | 948266 bytes
Sample: reddit_00716.txt | 1154557 bytes
Sample: reddit_00710.txt | 1027116 bytes

Q&A folder:
Exists: True
science_qa_cleaned.jsonl | .jsonl | 250502 bytes
indian_world_geography_english_qa_cleaned.jsonl | .jsonl | 223279 bytes
indian_polity_qa_cleaned.jsonl | .jsonl | 182145 bytes


In [ ]:
from pathlib import Path

root = Path("/content/gamax1_data_subset")

samples = [
    root / "Discord-Dialogues" / "discord_01970.txt",
    root / "Reddit-Constructive" / "reddit_00479.txt",
]

for path in samples:
    print("\n" + "=" * 80)
    print("FILE:", path)
    print("EXISTS:", path.exists())

    if path.exists():
        with path.open("r", encoding="utf-8", errors="replace") as f:
            for i, line in enumerate(f):
                if i >= 35:
                    break
                print(line.rstrip()[:500])


FILE: /content/gamax1_data_subset/Discord-Dialogues/discord_01970.txt
EXISTS: True
User: they named the place bug invasion?
Assistant: Yeah cause of like Volkswagen beetles

---

User: it’s needed for your industry
Assistant: Training in dust sounds so annoying reeeee

---

User: aya delete your account you only been here barely a month
Assistant: talk your talk

---

User: I hate the goodbyes people give before they switch profiles
Assistant: same account

---

User: Meanwhile Finland has like 4 hours a day and no homework
Assistant: absolute beauty

---

User: What are you talking about
Assistant: The people who know will understand Inshallah

---

User: I hope the pope is black
Assistant: Most of the american south would become atheist

---


FILE: /content/gamax1_data_subset/Reddit-Constructive/reddit_00479.txt
EXISTS: True
Speaker 0: Epic Thunder Sound Mod IV Play RB Ground M1A2 Abrams
Speaker 1: Does ETSM fix the extremely loud turboprop and jet engine noises?
Speaker 0: plane s

In [ ]:
from pathlib import Path

p = Path("/content/gamax1_project/gamax1/instruction_data.py")

print("File exists:", p.exists())

if p.exists():
    print(p.read_text(encoding="utf-8", errors="replace"))
else:
    print("instruction_data.py नहीं मिली।")

File exists: True
"""
gamax1/instruction_data.py

Loads instruction/Q&A-style training data (JSON or JSONL files) for the
fine-tuning stage.

This module keeps instruction examples DISCRETE, unlike bulk_corpus.py,
which works with one continuous token stream.

Main responsibilities:

1. Read JSON / JSONL instruction and Q&A datasets.
2. Detect common dataset formats.
3. Convert records into (prompt, answer) pairs.
4. Encode examples with explicit:
       <|user|>
       <|assistant|>
       <|eos|>
   boundaries.
5. Create an answer-only loss mask.
6. Batch variable-length examples with right-padding.
7. Provide deterministic train/validation splitting.

IMPORTANT TRAINING RULE
-----------------------

The loss is calculated ONLY on assistant answer tokens and EOS.

Question/prompt tokens:
    loss_mask = 0

<|user|>:
    loss_mask = 0

<|assistant|>:
    loss_mask = 0

Answer tokens:
    loss_mask = 1

<|eos|>:
    loss_mask = 1

This keeps fine-tuning focused on learning the desired 

In [ ]:
from pathlib import Path
import json
import re

# Discord text files का source folder
SOURCE_DIR = Path("/content/gamax1_data_subset/Discord-Dialogues")

# Converted dataset अलग जगह save होगा
OUTPUT_FILE = Path("/content/gamax1_project/discord_conversations.jsonl")
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# User:/Assistant: के बाद आने वाली multiline बातचीत parse करें
speaker_pattern = re.compile(
    r"^\s*(User|Assistant)\s*:\s*(.*)$",
    re.IGNORECASE
)

def parse_conversation(path):
    messages = []
    current_role = None
    current_text = []

    def save_message():
        nonlocal current_role, current_text

        text = "\n".join(current_text).strip()

        if current_role and text:
            role = current_role.lower()

            # लगातार एक ही speaker के messages जोड़ दें
            if messages and messages[-1]["role"] == role:
                messages[-1]["content"] += "\n" + text
            else:
                messages.append({
                    "role": role,
                    "content": text
                })

        current_role = None
        current_text = []

    try:
        content = path.read_text(
            encoding="utf-8",
            errors="replace"
        )
    except OSError as exc:
        print(f"Could not read {path.name}: {exc}")
        return None

    for line in content.splitlines():
        match = speaker_pattern.match(line)

        if match:
            save_message()
            current_role = match.group(1).lower()
            current_text = [match.group(2)]
        elif current_role is not None:
            current_text.append(line)

    save_message()

    # केवल पूरी user-assistant बातचीत रखें
    if not any(m["role"] == "user" for m in messages):
        return None

    if not any(m["role"] == "assistant" for m in messages):
        return None

    return {"messages": messages}


if not SOURCE_DIR.exists():
    raise FileNotFoundError(
        f"Source folder not found: {SOURCE_DIR}"
    )

files = sorted(SOURCE_DIR.rglob("*.txt"))

written = 0
skipped = 0

with OUTPUT_FILE.open("w", encoding="utf-8") as out:
    for path in files:
        record = parse_conversation(path)

        if record is None:
            skipped += 1
            continue

        out.write(
            json.dumps(record, ensure_ascii=False) + "\n"
        )
        written += 1

print("Source files:", len(files))
print("Conversations written:", written)
print("Files skipped:", skipped)
print("Output:", OUTPUT_FILE)

# कुछ records का preview
with OUTPUT_FILE.open(encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i == 3:
            break
        record = json.loads(line)
        print(f"\nExample {i + 1}:")
        print(json.dumps(
            record,
            ensure_ascii=False,
            indent=2
        )[:1500])

In [ ]:
from pathlib import Path
import json
import re

SOURCE = Path("/content/gamax1_data_subset")
OUTPUT = Path("/content/gamax1_data_subset_converted")
OUTPUT.mkdir(parents=True, exist_ok=True)

def convert_txt(folder_name, pattern, output_name):
    source_dir = SOURCE / folder_name
    output_file = OUTPUT / output_name
    files = sorted(source_dir.rglob("*.txt"))

    written = 0
    skipped = 0

    with output_file.open("w", encoding="utf-8") as out:
        for path in files:
            text = path.read_text(
                encoding="utf-8", errors="replace"
            ).strip()

            # Discord: User: ... / Assistant: ...
            if folder_name == "Discord-Dialogues":
                matches = list(re.finditer(
                    r"(?im)^\s*(User|Assistant)\s*:\s*",
                    text
                ))
                messages = []
                for i, match in enumerate(matches):
                    start = match.end()
                    end = (
                        matches[i + 1].start()
                        if i + 1 < len(matches)
                        else len(text)
                    )
                    content = text[start:end].strip()
                    role = match.group(1).lower()
                    if content:
                        messages.append({
                            "role": role,
                            "content": content
                        })

                # केवल files जिनमें दोनों roles हैं
                if not any(m["role"] == "user" for m in messages) or \
                   not any(m["role"] == "assistant" for m in messages):
                    skipped += 1
                    continue

            # Reddit: Speaker 0 / Speaker 1...
            else:
                matches = list(re.finditer(
                    r"(?im)^\s*Speaker\s+(\d+)\s*:\s*",
                    text
                ))
                messages = []
                for i, match in enumerate(matches):
                    start = match.end()
                    end = (
                        matches[i + 1].start()
                        if i + 1 < len(matches)
                        else len(text)
                    )
                    content = text[start:end].strip()
                    if content:
                        # Speaker 0 को user और बाकी speakers को assistant
                        role = "user" if match.group(1) == "0" else "assistant"
                        messages.append({
                            "role": role,
                            "content": content
                        })

                if not any(m["role"] == "user" for m in messages) or \
                   not any(m["role"] == "assistant" for m in messages):
                    skipped += 1
                    continue

            out.write(json.dumps(
                {"messages": messages},
                ensure_ascii=False
            ) + "\n")
            written += 1

    print(f"{folder_name}: {len(files)} files, "
          f"{written} converted, {skipped} skipped")
    print(f"Output: {output_file}")

convert_txt(
    "Discord-Dialogues", None, "discord.jsonl"
)
convert_txt(
    "Reddit-Constructive", None, "reddit.jsonl"
)

print("\nConversion complete. Original files unchanged.")

Discord-Dialogues: 300 files, 300 converted, 0 skipped
Output: /content/gamax1_data_subset_converted/discord.jsonl
Reddit-Constructive: 150 files, 150 converted, 0 skipped
Output: /content/gamax1_data_subset_converted/reddit.jsonl

Conversion complete. Original files unchanged.


In [ ]:
import json
from pathlib import Path

files = [
    Path("/content/gamax1_data_subset_converted/discord.jsonl"),
    Path("/content/gamax1_data_subset_converted/reddit.jsonl"),
]

for file in files:
    count = 0
    with file.open(encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            assert "messages" in item
            assert any(m["role"] == "user" for m in item["messages"])
            assert any(m["role"] == "assistant" for m in item["messages"])
            count += 1

    print(f"✅ {file.name}: {count} conversations validated")

✅ discord.jsonl: 300 conversations validated
✅ reddit.jsonl: 150 conversations validated


In [ ]:
%cd /content/gamax1_project
!python -m gamax1.finetune --help

/content/gamax1_project
usage: finetune.py [-h] [--init_from INIT_FROM] [--resume |
                   --resume_from RESUME_FROM] [--data DATA]
                   [--out_dir OUT_DIR] [--max_len MAX_LEN]
                   [--batch_size BATCH_SIZE] [--epochs EPOCHS]
                   [--max_steps MAX_STEPS] [--lr LR]
                   [--warmup_steps WARMUP_STEPS] [--weight_decay WEIGHT_DECAY]
                   [--val_fraction VAL_FRACTION]
                   [--eval_interval EVAL_INTERVAL]
                   [--checkpoint_interval CHECKPOINT_INTERVAL]
                   [--freeze_sparsity] [--no_freeze_sparsity]
                   [--replay_data_dir REPLAY_DATA_DIR]
                   [--replay_ratio REPLAY_RATIO]
                   [--replay_cache_dir REPLAY_CACHE_DIR] [--seed SEED]
                   [--no_amp] [--device DEVICE]

Fine-tune a pretrained GamaX1 checkpoint on instruction/Q&A JSON data with
answer-only masked loss.

options:
  -h, --help            show this help mess

In [ ]:
%cd /content/gamax1_project

!python -m gamax1.finetune \
  --init_from "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt" \
  --data "/content/gamax1_data_subset_converted" \
  --out_dir "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/conversation-v1" \
  --max_steps 1000 \
  --epochs 3 \
  --batch_size 4 \
  --lr 0.0001 \
  --warmup_steps 50 \
  --val_fraction 0.05 \
  --eval_interval 100 \
  --checkpoint_interval 100 \
  --replay_data_dir "/content/gamax1_data_subset/books_cleaned_v1" \
  --replay_ratio 0.20

/content/gamax1_project
Loaded checkpoint from /content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt (step ?)
Sparsity k: 384 (frozen: True)
^C


In [ ]:
from pathlib import Path

p = Path("/content/gamax1_data_subset/books_cleaned_v1")
print("Replay directory exists:", p.exists())
print("TXT files:", len(list(p.rglob("*.txt"))) if p.exists() else 0)

out = Path("/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/conversation-v1")
print("Output directory exists:", out.exists())
if out.exists():
    print("Existing files:", [x.name for x in out.iterdir()])

Replay directory exists: True
TXT files: 4593
Output directory exists: False


In [ ]:
import os
from pathlib import Path

print("PID:", os.getpid())
print("CUDA available:", __import__("torch").cuda.is_available())

out = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/conversation-v1"
)

print("Output directory exists:", out.exists())
if out.exists():
    print("Files:", [p.name for p in out.iterdir()])

PID: 779
CUDA available: True
Output directory exists: False


In [ ]:
%cd /content/gamax1_project

!grep -n -E "KeyboardInterrupt|SIGINT|signal|def main|out_dir|for step|for epoch|DataLoader|print\\(" gamax1/finetune.py | head -100

/content/gamax1_project
68:        print("[WARNING] a batch had zero loss-mask positions (fully truncated?); contributing 0 loss.")
137:    print(f"Saved checkpoint to {path} (step={step}, epoch={epoch}, batch={batch_in_epoch})")
149:            print(f"[WARNING] Could not fully restore CUDA RNG state: {exc}")
152:def main():
167:    parser.add_argument("--out_dir", type=str, default="checkpoints_finetune")
206:        resume_path = os.path.join(args.out_dir, "gamax1_finetune_latest.pt")
264:        print("[WARNING] --replay_ratio was set but --replay_data_dir is missing; replay will be disabled.")
276:    print(f"Loaded checkpoint from {source_checkpoint} (step {resume_ckpt.get('step', '?') if resume_ckpt else base_cfg.get('step', '?')})")
277:    print(f"Sparsity k: {model.sparsity_ctrl.k} (frozen: {args.freeze_sparsity})")
280:    print(f"Instruction data: {stats['files']} file(s), {stats['records']} record(s) -> "
284:        print("[WARNING] Sample unmatched record key sets (first

In [ ]:
%cd /content/gamax1_project

!sed -n '250,350p' gamax1/finetune.py

/content/gamax1_project
        parser.error("--batch_size must be >= 1")
    if args.epochs < 1:
        parser.error("--epochs must be >= 1")
    if args.max_steps is not None and args.max_steps < 1:
        parser.error("--max_steps must be >= 1")
    if args.max_len < 2:
        parser.error("--max_len must be >= 2")
    if args.lr <= 0:
        parser.error("--lr must be > 0")
    if not 0.0 <= args.val_fraction < 1.0:
        parser.error("--val_fraction must be in [0, 1)")
    if not 0.0 <= args.replay_ratio <= 1.0:
        parser.error("--replay_ratio must be in [0, 1]")
    if args.replay_ratio > 0 and not args.replay_data_dir:
        print("[WARNING] --replay_ratio was set but --replay_data_dir is missing; replay will be disabled.")
    if args.max_len > base_cfg["block_size"]:
        raise ValueError(f"--max_len={args.max_len} exceeds base checkpoint's block_size="
                          f"{base_cfg['block_size']}; the model was never trained on sequences "
            

In [ ]:
%cd /content/gamax1_project

from pathlib import Path
from gamax1.instruction_data import load_pairs

data_dir = Path("/content/gamax1_data_subset_converted")
print("JSONL files:", [p.name for p in data_dir.glob("*.jsonl")])

pairs, stats = load_pairs(str(data_dir))
print("Dataset stats:", stats)
print("Usable conversation pairs:", len(pairs))

replay_dir = Path("/content/gamax1_data_subset/books_cleaned_v1")
print("Replay directory exists:", replay_dir.exists())
print("Replay TXT files:", len(list(replay_dir.rglob("*.txt")))
      if replay_dir.exists() else 0)

/content/gamax1_project
JSONL files: ['reddit.jsonl', 'discord.jsonl']


In [ ]:
from pathlib import Path
import json
import psutil

cache = Path("/content/drive/MyDrive/Aetherion_GamaX1")
print("RAM available (GB):",
      round(psutil.virtual_memory().available / 1024**3, 2))

for p in cache.rglob("metadata.json"):
    try:
        data = json.loads(p.read_text())
        if "token_count" in data and "source_token_counts" in data:
            print("Replay cache:", p.parent)
            print("Tokens:", f"{data['token_count']:,}")
            print("Sources:", data["source_token_counts"])
    except Exception:
        pass

RAM available (GB): 9.77
Replay cache: /content/drive/MyDrive/Aetherion_GamaX1/cache/gamax1_bulk
Tokens: 1,170,998,016
Sources: {'Conversations-200k_clean': 268768649, 'Math_Reasoning_train': 200249409, 'Q&A': 106457, 'books_cleaned_v1': 701873501}
Replay cache: /content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache
Tokens: 701,873,500
Sources: {'books': 701873500}


In [ ]:
%cd /content/gamax1_project

!grep -n -E "def build_or_load_bulk_tokens|np.memmap|torch.from_numpy|torch.tensor|np.concatenate|np.array|metadata|rebuild" gamax1/bulk_corpus.py

/content/gamax1_project
11:files are tracked separately in the metadata so downstream tooling can
22:merges) still forces a full rebuild, since every existing token ID would be
54:# trigger a 34k-file rebuild.  Cache identity must depend on corpus content
454:    """Read-only memory-mapped token storage with a torch tensor view."""
607:    path candidates.  This is only metadata migration; token bytes are not
636:    """Convert legacy absolute file-index keys to stable keys without rebuilding."""
699:    rebuild rather than a partial reuse.
770:    """Atomically replace a small metadata/tokenizer JSON file."""
801:                        "rebuild the corpus cache with a new cache directory."
846:def build_or_load_bulk_tokens(
853:    rebuild: bool = False,
870:      * A changed/deleted previously-encoded file triggers a clean rebuild,
895:    meta_path = cache / "metadata.json"
911:    if rebuild:
912:        # Explicit rebuild is non-destructive: quarantine old artifacts
915:         

In [ ]:
%cd /content/gamax1_project
!sed -n '846,985p' gamax1/bulk_corpus.py
!sed -n '445,485p' gamax1/bulk_corpus.py

/content/gamax1_project
def build_or_load_bulk_tokens(
    data_dir: str | Path,
    cache_dir: str | Path,
    *,
    bpe_vocab_size: int = 16000,
    bpe_sample_chars: int = 3_000_000,
    tokenizer: Optional[BPETokenizer] = None,
    rebuild: bool = False,
    source_dirs=DEFAULT_SOURCE_DIRS,
    source_formats: dict = None,
) -> tuple[BPETokenizer, BulkTokenStore, dict]:
    """Build/reuse a persistent memory-mapped bulk token cache.

    ``source_formats`` maps each source-folder name to one of
    SOURCE_FORMAT_PROSE / SOURCE_FORMAT_USER_ASSISTANT /
    SOURCE_FORMAT_GENERIC_TURNS (see the module-level comment above
    DEFAULT_SOURCE_FORMATS for what each does and why). Defaults to
    DEFAULT_SOURCE_FORMATS; pass a copy with overrides to customize
    without editing this file.

    IMPORTANT:
      * Changing training settings does NOT invalidate this cache.
      * Existing files are never re-encoded when their size is unchanged.
      * New files are appended only.
      * A

In [ ]:
from pathlib import Path
import json

caches = [
    Path("/content/drive/MyDrive/Aetherion_GamaX1/cache/gamax1_bulk"),
    Path("/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache"),
]

for cache in caches:
    print(f"\n--- {cache.name} ---")

    for name in [
        "metadata.json",
        "tokenizer.json",
        "tokens.int32.bin",
    ]:
        p = cache / name
        print(f"{name}: exists={p.exists()}", end="")
        if p.exists() and p.is_file():
            print(f", size={p.stat().st_size / 1024**3:.2f} GB")
        else:
            print()

    meta = cache / "metadata.json"
    if meta.exists():
        data = json.loads(meta.read_text())
        print("Token count:", data.get("token_count"))
        print("Sources:", data.get("source_token_counts"))


--- gamax1_bulk ---
metadata.json: exists=True, size=0.00 GB
tokenizer.json: exists=True, size=0.00 GB
tokens.int32.bin: exists=True, size=4.36 GB
Token count: 1170998016
Sources: {'Conversations-200k_clean': 268768649, 'Math_Reasoning_train': 200249409, 'Q&A': 106457, 'books_cleaned_v1': 701873501}

--- finetune_replay_cache ---
metadata.json: exists=True, size=0.00 GB
tokenizer.json: exists=True, size=0.00 GB
tokens.int32.bin: exists=True, size=2.61 GB
Token count: 701873500
Sources: {'books': 701873500}


In [ ]:
from pathlib import Path
import json

cache = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache"
)

meta_path = cache / "metadata.json"
token_path = cache / "tokens.int32.bin"
tokenizer_path = cache / "tokenizer.json"

for p in (meta_path, token_path, tokenizer_path):
    print(f"{p.name}: {'OK' if p.is_file() else 'MISSING'}")

if meta_path.is_file() and token_path.is_file() and tokenizer_path.is_file():
    meta = json.loads(meta_path.read_text())
    expected_tokens = int(meta["token_count"])
    actual_tokens = token_path.stat().st_size // 4

    print("Metadata tokens:", f"{expected_tokens:,}")
    print("Binary tokens:", f"{actual_tokens:,}")
    print("Token count matches:", expected_tokens == actual_tokens)

metadata.json: OK
tokens.int32.bin: OK
tokenizer.json: OK
Metadata tokens: 701,873,500
Binary tokens: 701,873,500
Token count matches: True


In [ ]:
%cd /content/gamax1_project

!grep -n -E "replay_cache_dir|build_or_load_bulk_tokens|rebuild=|tokenizer_expected|file_index" gamax1/bulk_corpus.py gamax1/finetune.py

/content/gamax1_project
gamax1/bulk_corpus.py:633:def _migrate_file_index_paths(
gamax1/bulk_corpus.py:682:def _file_index_path(cache_dir: Path) -> Path:
gamax1/bulk_corpus.py:683:    return cache_dir / "file_index.json"
gamax1/bulk_corpus.py:693:def _load_file_index(cache_dir: Path, tokenizer_expected: dict) -> Optional[dict]:
gamax1/bulk_corpus.py:701:    path = _file_index_path(cache_dir)
gamax1/bulk_corpus.py:708:    if index.get("tokenizer") != tokenizer_expected:
gamax1/bulk_corpus.py:713:def _save_file_index(cache_dir: Path, index: dict) -> None:
gamax1/bulk_corpus.py:715:        _file_index_path(cache_dir),
gamax1/bulk_corpus.py:721:    cache_dir: Path, token_path: Path, tokenizer_expected: dict, batch_size: int,
gamax1/bulk_corpus.py:740:    if progress.get("tokenizer") != tokenizer_expected:
gamax1/bulk_corpus.py:846:def build_or_load_bulk_tokens(
gamax1/bulk_corpus.py:903:    tokenizer_expected = {
gamax1/bulk_corpus.py:914:        for path in (token_path, _file_index_path(c

In [ ]:
from pathlib import Path
import json

cache = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache"
)

index_path = cache / "file_index.json"
meta_path = cache / "metadata.json"

print("File index exists:", index_path.is_file())

if index_path.is_file():
    index = json.loads(index_path.read_text())
    print("Indexed token count:", index.get("token_count"))
    print("Indexed files:", len(index.get("files", {})))
    print("Index tokenizer:", index.get("tokenizer"))

if meta_path.is_file():
    metadata = json.loads(meta_path.read_text())
    print("\nCache metadata keys:", list(metadata.keys()))
    print("Cache metadata:", metadata)

File index exists: True
Indexed token count: 701873500
Indexed files: 4593
Index tokenizer: {'tokenizer': 'bpe', 'vocab_size': 16006, 'merges': [[32, 116], [32, 97], [105, 110], [104, 101], [114, 101], [32, 115], [111, 110], [256, 259], [101, 114], [97, 116], [32, 99], [110, 100], [111, 114], [101, 115], [101, 110], [111, 117], [258, 103], [32, 119], [105, 116], [32, 112], [257, 267], [97, 110], [32, 102], [97, 108], [32, 111], [105, 115], [32, 109], [256, 111], [32, 98], [101, 100], [32, 258], [97, 114], [32, 100], [280, 102], [105, 262], [108, 101], [105, 99], [10, 10], [114, 111], [32, 104], [32, 32], [97, 115], [32, 101], [270, 116], [111, 109], [105, 108], [32, 260], [118, 101], [256, 104], [99, 116], [117, 115], [32, 108], [32, 121], [101, 116], [32, 103], [308, 271], [32, 84], [99, 101], [114, 97], [105, 109], [111, 119], [105, 100], [32, 110], [278, 268], [99, 104], [108, 121], [265, 290], [32, 259], [261, 116], [115, 116], [105, 103], [275, 294], [304, 265], [105, 114], [111, 

In [ ]:
%cd /content/gamax1_project

!sed -n '680,715p' gamax1/bulk_corpus.py
!sed -n '846,930p' gamax1/bulk_corpus.py
!sed -n '300,335p' gamax1/finetune.py

/content/gamax1_project


def _file_index_path(cache_dir: Path) -> Path:
    return cache_dir / "file_index.json"


def _progress_path(cache_dir: Path) -> Path:
    return cache_dir / "encode_progress.json"

def _encoding_timing_path(cache_dir: Path) -> Path:
    return cache_dir / "encoding_checkpoint_timing.jsonl"


def _load_file_index(cache_dir: Path, tokenizer_expected: dict) -> Optional[dict]:
    """Load the per-file token index, or None if absent/unusable.

    Only usable if the tokenizer identity (vocab size + BPE merges) matches
    exactly -- reusing token IDs encoded under different merges would
    silently corrupt the stream, so any tokenizer change forces a clean
    rebuild rather than a partial reuse.
    """
    path = _file_index_path(cache_dir)
    if not path.exists():
        return None
    try:
        index = json.loads(path.read_text(encoding="utf-8"))
    except (json.JSONDecodeError, OSError):
        return None
    if index.get("tokenizer") != tokenizer_e

In [ ]:
%cd /content/gamax1_project

import json
from pathlib import Path
from gamax1.tokenizer import BPETokenizer

checkpoint = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt"
)
cache_dir = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache"
)

# Inspect checkpoint keys without changing anything.
import torch
ckpt = torch.load(checkpoint, map_location="cpu", weights_only=False)
print("Checkpoint top-level keys:", list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt))

# Inspect cache tokenizer identity.
cache_tok = json.loads((cache_dir / "tokenizer.json").read_text())
print("\nCache tokenizer keys:", list(cache_tok.keys()))
print("Cache vocab size:", cache_tok.get("vocab_size"))
print("Cache merges:", len(cache_tok.get("merges", [])))

/content/gamax1_project
Checkpoint top-level keys: ['model_state', 'optimizer_state', 'sparsity_controller_state', 'ptm_states', 'step', 'vocab', 'merges', 'config', 'scaler_state', 'rng_state', 'cuda_rng_state_all']

Cache tokenizer keys: ['merges']
Cache vocab size: None
Cache merges: 15744


In [ ]:
import json
import torch
from pathlib import Path

checkpoint_path = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt"
)
cache_tokenizer_path = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache/tokenizer.json"
)

ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
cache_tok = json.loads(cache_tokenizer_path.read_text())

checkpoint_merges = ckpt.get("merges")
cache_merges = cache_tok.get("merges")

vocab = ckpt.get("vocab")

print("Checkpoint vocab type:", type(vocab).__name__)
print("Checkpoint vocab size:", len(vocab) if vocab is not None else "None")
print("Checkpoint merges:", len(checkpoint_merges) if checkpoint_merges is not None else "None")
print("Cache merges:", len(cache_merges) if cache_merges is not None else "None")

if checkpoint_merges is not None and cache_merges is not None:
    print("Exact merges match:", checkpoint_merges == cache_merges)
else:
    print("Cannot compare yet: merges missing from one side.")

Checkpoint vocab type: NoneType
Checkpoint vocab size: None
Checkpoint merges: 15744
Cache merges: 15744
Exact merges match: False


In [ ]:
import json
import torch
from pathlib import Path

checkpoint_path = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt"
)
cache_path = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache/tokenizer.json"
)

ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
cache = json.loads(cache_path.read_text())

a = ckpt["merges"]
b = cache["merges"]

print("Checkpoint merges type:", type(a).__name__)
print("Cache merges type:", type(b).__name__)
print("Checkpoint first 5:", a[:5])
print("Cache first 5:", b[:5])

for i, (x, y) in enumerate(zip(a, b)):
    if x != y:
        print("First mismatch index:", i)
        print("Checkpoint value:", repr(x))
        print("Cache value:", repr(y))
        break
else:
    print("No mismatch in shared range.")
    print("Different lengths:", len(a), len(b))

print("\nCheckpoint config:", ckpt.get("config"))

Checkpoint merges type: list
Cache merges type: list
Checkpoint first 5: [(32, 116), (32, 97), (105, 110), (104, 101), (114, 101)]
Cache first 5: [[32, 116], [32, 97], [105, 110], [104, 101], [114, 101]]
First mismatch index: 0
Checkpoint value: (32, 116)
Cache value: [32, 116]

Checkpoint config: {'data': '/content/gamax1_project/gamax1/../data/sample_corpus.txt', 'data_dir': '/content/drive/MyDrive/Aetherion_GamaX1/data', 'bulk_cache_dir': '/content/drive/MyDrive/Aetherion_GamaX1/cache/gamax1_bulk', 'rebuild_bulk_cache': False, 'out_dir': '/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full', 'd_model': 768, 'n_heads': 12, 'n_layers': 12, 'n_features': 3072, 'auto_size_model': False, 'auto_size_target_tokens_per_param': 40.0, 'block_size': 512, 'batch_size': 16, 'lr': 0.0003, 'weight_decay': 0.01, 'dropout': 0.1, 'max_steps': 100000, 'warmup_steps': 500, 'eval_interval': 200, 'eval_batches': 10, 'eval_batch_size': 16, 'validation_split': 'tail', 'validation_split_seed': 42, 'ove

In [ ]:
import json
import torch
from pathlib import Path

checkpoint_path = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt"
)
cache_path = Path(
    "/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache/tokenizer.json"
)

ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
cache = json.loads(cache_path.read_text())

checkpoint_merges = [tuple(pair) for pair in ckpt["merges"]]
cache_merges = [tuple(pair) for pair in cache["merges"]]

print("Checkpoint merges:", len(checkpoint_merges))
print("Cache merges:", len(cache_merges))
print("Normalized merges match:", checkpoint_merges == cache_merges)

if checkpoint_merges != cache_merges:
    for i, (a, b) in enumerate(zip(checkpoint_merges, cache_merges)):
        if a != b:
            print("First real mismatch:", i, a, b)
            break

Checkpoint merges: 15744
Cache merges: 15744
Normalized merges match: True


In [ ]:
%cd /content/gamax1_project

!python -u -m gamax1.finetune \
  --init_from "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt" \
  --data "/content/gamax1_data_subset_converted" \
  --out_dir "/content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/conversation-test-v1" \
  --max_steps 100 \
  --epochs 3 \
  --batch_size 2 \
  --lr 0.00005 \
  --warmup_steps 10 \
  --val_fraction 0.05 \
  --eval_interval 20 \
  --checkpoint_interval 20 \
  --replay_data_dir "/content/gamax1_data_subset/books_cleaned_v1" \
  --replay_cache_dir "/content/drive/MyDrive/Aetherion_GamaX1/cache/finetune_replay_cache" \
  --replay_ratio 0.20

/content/gamax1_project
Loaded checkpoint from /content/drive/MyDrive/Aetherion_GamaX1/runs/gamax1/full/gamax1.pt (step ?)
Sparsity k: 384 (frozen: True)
^C
